In [ ]:
#!/usr/bin/env python3
"""
Enhanced Train Travel Time Calculator v9.0
Advanced physics-based calculations with flexible station configuration
Supports multiple train types with accurate station-to-station timing

v9.0 FIXES:
- Fixed dwell times: 300s local, 600s express, 900s limited/terminus
- Tilt speed threshold: tilt only active below max_tilt_speed
- Tilt rate limiting: accounts for mechanical tilt speed (~4°/s)
- S-curve detection: extra speed reduction for rapid direction changes
- Multi-baseline curve detection for robust curve analysis
- Fixed gradient speed formula (per-percent reduction)
- Power-limited acceleration at high speeds
"""

import xml.etree.ElementTree as ET
import math
import zipfile
import os
import tempfile
from dataclasses import dataclass, field
from typing import List, Tuple, Optional, Dict, Union, Set
import json
from enum import Enum
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.collections import LineCollection
from matplotlib.patches import Circle
import numpy as np
from matplotlib.gridspec import GridSpec
from matplotlib.colorbar import Colorbar
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

plt.rcParams['figure.max_open_warning'] = 0

# ==================== PRIMARY CONFIGURATION ====================

# Path to your KMZ file (track geometry)
KMZ_PATH = r"C:\Users\peckb\Documents\APEx\Cincinnati Metro\Line 6_stitched.kmz"

# STATION CONFIGURATION OPTIONS
#STATION_SOURCE = "STATIONS_FILE"
STATION_SOURCE = "TRACK"


# Options: "TRACK", "STATIONS_FILE", "BOTH", "NONE"
# - "TRACK": Use stations from the track KMZ file only
# - "STATIONS_FILE": Use stations from the separate stations file only  
# - "BOTH": Merge stations from both files (removes duplicates)
# - "NONE": Calculate track timing without any stations

# Path to separate stations KMZ file (only used if STATION_SOURCE includes it)
STATIONS_KMZ_PATH = r"C:\Users\peckb\Documents\APEx\TC_Limited.kmz"

# Service pattern configuration
SERVICE_PATTERN = "ALL"  # Options: "ALL", "EXPRESS", "CUSTOM"
# - "ALL": Stop at all stations
# - "EXPRESS": Stop only at major/terminus stations
# - "CUSTOM": Use CUSTOM_STOPS list below

# Custom stops list (station names to stop at when SERVICE_PATTERN = "CUSTOM")
CUSTOM_STOPS = ["Dayton Station", "Cincinnati Union Terminal"]

# Train type selection
TRAIN_TYPE = "MBTA_ORANGE"  
# Diesel: "MP36PH", "F40PH", "DE30AC", "HSP46"
# High-Performance Diesel: "SIEMENS_CHARGER", "STADLER_FLIRT_DMU", "BUDD_RDC"
# Metro: "MBTA_ORANGE", "MBTA_RED", "BART", "NYC_R211", "WMATA"
# Regional: "STADLER_KISS", "M9", "M8", "HIGHLINER", "CRRC_EMU", "METROLINER", "ETR460", "BR_CLASS_442", "DB610E", "PENDOLINO_CVG", "CAF_CIVITY_TILT", "ADTRANZ_CAF"
# HSR: "AVELIA_LIBERTY", "ETR_1000", "VELARO_MS", "AGV_USA", "AGV_USA_TILT", "ORBIT_610"

# Unit system
UNIT_SYSTEM = "METRIC"  # "METRIC" or "IMPERIAL"

# Service type - determines station classification and dwell time behavior
SERVICE_TYPE = "APEX"  # Options: "METRO", "REGIONAL", "APEX"
# - "METRO": Urban transit (e.g., TRIMET, MBTA) - short dwell times (30s/60s/120s)
# - "REGIONAL": State/regional rail (e.g., ORBIT, Amtrak regional) - medium dwell times (60s/120s/300s)
# - "APEX": National HSR network - long dwell times (300s/600s/900s) with APEx station classifications

# Analysis options
COMPARE_SERVICE_PATTERNS = False  # If True, analyze multiple service patterns
CALCULATE_REVERSE = True  # Calculate reverse direction
CREATE_VISUALIZATIONS = True  # Generate plots
SAVE_RESULTS = True  # Save to JSON file

# Technical parameters
INTERPOLATION_SPACING = 10.0  # meters
PRECALCULATE_CURVES = True  # If True, calculate curves from ORIGINAL points before interpolation
CURVE_DETECTION_METHOD = "heading"  # "heading" (recommended) or "3point"
# "heading" - Uses rate of heading change, accurate for all curve sizes
# "3point"  - Uses 3-point circumcircle, requires wide spans for gentle curves

# v9.1: S-CURVE ACCELERATION MODEL (Gen 6 APEx)
# The S-curve profile gradually ramps jerk for passenger comfort.
# This results in lower effective acceleration than the peak rate.
# Reference: APEx_Journey_Time_Parameters.md
USE_SCURVE_ACCEL = True  # Enable 3-phase S-curve acceleration model

# Gen 6 S-curve parameters
SCURVE_JERK = 0.25  # m/s³ - rate of change of acceleration
SCURVE_PEAK_ACCEL = 2.5  # m/s² - maximum acceleration during hold phase
SCURVE_PEAK_DECEL = 2.0  # m/s² - maximum deceleration during hold phase
SCURVE_RAMP_TIME = 10.0  # seconds - time to ramp from 0 to peak accel (= peak/jerk)

# Empirical time-to-speed data (Gen 6, dry conditions)
# Format: {target_speed_ms: time_seconds}
SCURVE_TIME_TO_SPEED = {
    44.7: 36,   # 0-100 mph
    67.1: 55,   # 0-150 mph  
    89.4: 80,   # 0-200 mph
    111.76: 110,  # 0-250 mph
}

# Deceleration times (Gen 6, service braking)
# Slightly shorter than acceleration due to lower comfort requirements during braking
SCURVE_TIME_TO_STOP = {
    # Braking is FASTER than acceleration because:
    # - Regenerative braking + friction brakes = more stopping power
    # - Friction brakes are NOT power-limited at high speed (unlike motors)
    # - ~25% faster than acceleration, especially at high speeds
    44.7: 30,   # 100-0 mph: 30s (avg 1.49 m/s²)
    67.1: 45,   # 150-0 mph: 45s (avg 1.49 m/s²)
    89.4: 62,   # 200-0 mph: 62s (avg 1.44 m/s²)
    111.76: 82, # 250-0 mph: 82s (avg 1.36 m/s²) - ~25% faster than accel
}

# =============================================================================
# HSR CURVE SPEED REFERENCE TABLE
# Based on: 180mm cant, 90mm deficiency, 7° tilt (disabled >265 km/h), 250 mph max
# Format: radius_m: (speed_no_tilt_kmh, speed_with_tilt_kmh, recommended_limit_kmh)
# =============================================================================
HSR_CURVE_SPEED_TABLE = {
    # Legacy/yard curves (< 800m) - tilt essential
    300:  (83, 111, 111),    # 69 mph - yard/terminal only
    400:  (96, 128, 128),    # 79 mph
    500:  (107, 143, 143),   # 89 mph
    600:  (117, 157, 157),   # 97 mph
    700:  (127, 169, 169),   # 105 mph
    # Station approach curves (800-1500m) - tilt very beneficial
    800:  (136, 181, 181),   # 112 mph - minimum recommended HSR
    900:  (144, 192, 192),   # 119 mph
    1000: (152, 202, 202),   # 126 mph
    1250: (169, 226, 226),   # 140 mph
    1500: (186, 248, 248),   # 154 mph
    # Tilt-beneficial curves (1500-3000m)
    1750: (201, 265, 265),   # 165 mph - tilt maxed out
    2000: (214, 265, 265),   # 165 mph
    2250: (227, 265, 265),   # 165 mph
    2500: (240, 265, 265),   # 165 mph
    2750: (251, 265, 265),   # 165 mph
    3000: (263, 265, 265),   # 165 mph - tilt barely helps
    # Reduced speed curves (3000-7000m) - no tilt benefit
    3500: (284, 265, 284),   # 176 mph
    4000: (303, 265, 303),   # 188 mph
    4500: (322, 265, 322),   # 200 mph
    5000: (339, 265, 339),   # 211 mph
    5500: (355, 265, 355),   # 221 mph
    6000: (371, 265, 371),   # 231 mph
    6500: (386, 265, 386),   # 240 mph
    # Full HSR speed curves (>= 7000m)
    7000: (401, 265, 401),   # 249 mph
    7500: (402, 265, 402),   # 250 mph - full speed
    8000: (402, 265, 402),   # 250 mph
    10000: (402, 265, 402),  # 250 mph
}

def get_curve_speed_from_table(radius_m: float, use_tilt: bool = True) -> float:
    """
    Get curve speed limit from reference table with interpolation.
    
    Args:
        radius_m: Curve radius in meters
        use_tilt: Whether tilting train (affects speeds below ~3000m)
    
    Returns:
        Speed limit in km/h
    """
    if radius_m <= 0 or radius_m > 100000:
        return 402.336  # Max speed for straight track
    
    # Get sorted radii
    radii = sorted(HSR_CURVE_SPEED_TABLE.keys())
    
    # Below minimum - extrapolate using sqrt relationship
    if radius_m < radii[0]:
        r1 = radii[0]
        v1 = HSR_CURVE_SPEED_TABLE[r1][2] if use_tilt else HSR_CURVE_SPEED_TABLE[r1][0]
        return v1 * (radius_m / r1) ** 0.5
    
    # Above maximum - return max speed
    if radius_m >= radii[-1]:
        return 402.336
    
    # Find bracketing entries and interpolate
    for i in range(len(radii) - 1):
        if radii[i] <= radius_m < radii[i + 1]:
            r1, r2 = radii[i], radii[i + 1]
            v1 = HSR_CURVE_SPEED_TABLE[r1][2] if use_tilt else HSR_CURVE_SPEED_TABLE[r1][0]
            v2 = HSR_CURVE_SPEED_TABLE[r2][2] if use_tilt else HSR_CURVE_SPEED_TABLE[r2][0]
            # Linear interpolation
            ratio = (radius_m - r1) / (r2 - r1)
            return v1 + ratio * (v2 - v1)
    
    return 402.336  # Fallback
STATION_MATCH_TOLERANCE = 500.0  # meters - increased for long-distance routes
STATION_DEDUP_RADIUS = 10.0  # meters for duplicate detection
MIN_STATION_SPACING = 500.0  # minimum spacing between stations (meters)

# v9.0: FIXED DWELL TIMES (seconds) - varies by SERVICE_TYPE
# METRO: Urban transit - passengers board/alight quickly
DWELL_TIMES_METRO = {
    'LOCAL': 30,      # 30 seconds for minor stops
    'EXPRESS': 60,    # 1 minute for major stops  
    'LIMITED': 120,   # 2 minutes for terminus/transfer stations
}

# REGIONAL: State/regional services - moderate dwell for intercity
DWELL_TIMES_REGIONAL = {
    'LOCAL': 60,      # 1 minute for minor stations
    'EXPRESS': 120,   # 2 minutes for major stations
    'LIMITED': 300,   # 5 minutes for terminus/hub stations
}

# APEX: National HSR network - Gen 6 dwell times with PSD + Info System
# Based on APEx Station Categories:
#   Cat 1-2 (LIMITED): Major terminals/hubs - 3-8 min
#   Cat 3 (EXPRESS): Medium stations - 75 sec
#   Cat 4-5 (LOCAL): Minor stations - 45-60 sec
DWELL_TIMES_APEX = {
    'LOCAL': 45,      # 45 seconds (Cat 5: suburban/edge city)
    'EXPRESS': 75,    # 75 seconds (Cat 3: medium stations)
    'LIMITED': 180,   # 3 minutes (Cat 2B: major hubs, avg of Cat 1-2)
}

# Select active dwell times based on SERVICE_TYPE
def get_dwell_times():
    if SERVICE_TYPE == "METRO":
        return DWELL_TIMES_METRO
    elif SERVICE_TYPE == "REGIONAL":
        return DWELL_TIMES_REGIONAL
    else:  # APEX
        return DWELL_TIMES_APEX

# Legacy constants for backward compatibility
DWELL_TIME_LOCAL = get_dwell_times()['LOCAL']
DWELL_TIME_EXPRESS = get_dwell_times()['EXPRESS']
DWELL_TIME_LIMITED = get_dwell_times()['LIMITED']


# ==================== S-CURVE ACCELERATION PHYSICS ====================

def calculate_scurve_accel_time(v_initial_ms: float, v_final_ms: float, 
                                 accel_peak: float = None, jerk: float = None) -> float:
    """
    Calculate time for S-curve acceleration between two speeds.
    
    IMPORTANT: Real trains have power limiting at high speeds:
    - Below ~50 mph: torque-limited (good acceleration)
    - Above ~50 mph: power-limited (acceleration decreases with speed)
    
    This function uses empirical data interpolation to account for these effects.
    The empirical data represents cumulative time from 0 to each speed, so
    time for v1→v2 = time(0→v2) - time(0→v1).
    
    Returns: time in seconds
    """
    if not USE_SCURVE_ACCEL:
        if accel_peak is None:
            accel_peak = SCURVE_PEAK_ACCEL
        delta_v = abs(v_final_ms - v_initial_ms)
        return delta_v / accel_peak if accel_peak > 0 else 0
    
    if accel_peak is None:
        accel_peak = SCURVE_PEAK_ACCEL
    if jerk is None:
        jerk = SCURVE_JERK
    
    delta_v = abs(v_final_ms - v_initial_ms)
    
    if delta_v < 0.1:
        return 0
    
    def get_time_from_zero(target_v):
        """Get time to accelerate from 0 to target_v using empirical data"""
        if not SCURVE_TIME_TO_SPEED:
            # Fallback to physics
            return target_v / accel_peak
        
        speeds_sorted = sorted(SCURVE_TIME_TO_SPEED.keys())
        
        if target_v <= 0.1:
            return 0
        elif target_v <= speeds_sorted[0]:
            # Below minimum data - assume linear scaling
            ratio = target_v / speeds_sorted[0]
            return SCURVE_TIME_TO_SPEED[speeds_sorted[0]] * ratio
        elif target_v >= speeds_sorted[-1]:
            # Above maximum data - extrapolate
            if len(speeds_sorted) >= 2:
                v1, v2 = speeds_sorted[-2], speeds_sorted[-1]
                t1, t2 = SCURVE_TIME_TO_SPEED[v1], SCURVE_TIME_TO_SPEED[v2]
                slope = (t2 - t1) / (v2 - v1)
                return t2 + slope * (target_v - v2)
            return SCURVE_TIME_TO_SPEED[speeds_sorted[-1]] * (target_v / speeds_sorted[-1])
        else:
            # Interpolate
            for i in range(len(speeds_sorted) - 1):
                if speeds_sorted[i] <= target_v <= speeds_sorted[i+1]:
                    v1, v2 = speeds_sorted[i], speeds_sorted[i+1]
                    t1, t2 = SCURVE_TIME_TO_SPEED[v1], SCURVE_TIME_TO_SPEED[v2]
                    ratio = (target_v - v1) / (v2 - v1)
                    return t1 + ratio * (t2 - t1)
        return target_v / accel_peak  # Fallback
    
    # Time for v_initial → v_final = time(0→v_final) - time(0→v_initial)
    # This accounts for the fact that acceleration is slower at higher speeds
    time_to_final = get_time_from_zero(v_final_ms)
    time_to_initial = get_time_from_zero(v_initial_ms)
    
    return max(0, time_to_final - time_to_initial)


def calculate_scurve_decel_time(v_initial_ms: float, v_final_ms: float,
                                 decel_peak: float = None, jerk: float = None) -> float:
    """
    Calculate time for S-curve deceleration between two speeds.
    
    Uses empirical data interpolation similar to acceleration.
    Time for v_initial → v_final = time(v_initial→0) - time(v_final→0)
    
    Returns: time in seconds
    """
    if not USE_SCURVE_ACCEL:
        if decel_peak is None:
            decel_peak = SCURVE_PEAK_DECEL
        delta_v = abs(v_initial_ms - v_final_ms)
        return delta_v / decel_peak if decel_peak > 0 else 0
    
    if decel_peak is None:
        decel_peak = SCURVE_PEAK_DECEL
    if jerk is None:
        jerk = SCURVE_JERK
    
    delta_v = abs(v_initial_ms - v_final_ms)
    
    if delta_v < 0.1:
        return 0
    
    def get_time_to_stop(from_v):
        """Get time to decelerate from from_v to 0 using empirical data"""
        if not SCURVE_TIME_TO_STOP:
            return from_v / decel_peak
        
        speeds_sorted = sorted(SCURVE_TIME_TO_STOP.keys())
        
        if from_v <= 0.1:
            return 0
        elif from_v <= speeds_sorted[0]:
            ratio = from_v / speeds_sorted[0]
            return SCURVE_TIME_TO_STOP[speeds_sorted[0]] * ratio
        elif from_v >= speeds_sorted[-1]:
            if len(speeds_sorted) >= 2:
                v1, v2 = speeds_sorted[-2], speeds_sorted[-1]
                t1, t2 = SCURVE_TIME_TO_STOP[v1], SCURVE_TIME_TO_STOP[v2]
                slope = (t2 - t1) / (v2 - v1)
                return t2 + slope * (from_v - v2)
            return SCURVE_TIME_TO_STOP[speeds_sorted[-1]] * (from_v / speeds_sorted[-1])
        else:
            for i in range(len(speeds_sorted) - 1):
                if speeds_sorted[i] <= from_v <= speeds_sorted[i+1]:
                    v1, v2 = speeds_sorted[i], speeds_sorted[i+1]
                    t1, t2 = SCURVE_TIME_TO_STOP[v1], SCURVE_TIME_TO_STOP[v2]
                    ratio = (from_v - v1) / (v2 - v1)
                    return t1 + ratio * (t2 - t1)
        return from_v / decel_peak
    
    # Time for v_initial → v_final = time(v_initial→0) - time(v_final→0)
    time_from_initial = get_time_to_stop(v_initial_ms)
    time_from_final = get_time_to_stop(v_final_ms)
    
    return max(0, time_from_initial - time_from_final)


def calculate_scurve_distance(v_initial_ms: float, v_final_ms: float,
                               accel_peak: float = None, jerk: float = None) -> float:
    """
    Calculate distance covered during S-curve acceleration.
    
    Returns: distance in meters
    """
    if not USE_SCURVE_ACCEL:
        if accel_peak is None:
            accel_peak = SCURVE_PEAK_ACCEL
        delta_v = abs(v_final_ms - v_initial_ms)
        t = delta_v / accel_peak if accel_peak > 0 else 0
        return (v_initial_ms + v_final_ms) / 2 * t
    
    if accel_peak is None:
        accel_peak = SCURVE_PEAK_ACCEL
    if jerk is None:
        jerk = SCURVE_JERK
    
    delta_v = abs(v_final_ms - v_initial_ms)
    
    if delta_v < 0.1:
        return 0
    
    # Get time from empirical or physics-based calculation
    total_time = calculate_scurve_accel_time(v_initial_ms, v_final_ms, accel_peak, jerk)
    
    # Average speed approximation (slightly underestimates for S-curve but close enough)
    avg_speed = (v_initial_ms + v_final_ms) / 2
    
    return avg_speed * total_time


def calculate_scurve_effective_accel(v_initial_ms: float, v_final_ms: float) -> float:
    """
    Calculate effective acceleration for a speed change.
    This is lower than peak acceleration due to S-curve ramp phases.
    
    Returns: effective acceleration in m/s²
    """
    delta_v = abs(v_final_ms - v_initial_ms)
    if delta_v < 0.1:
        return SCURVE_PEAK_ACCEL
    
    time = calculate_scurve_accel_time(v_initial_ms, v_final_ms)
    if time > 0:
        return delta_v / time
    return SCURVE_PEAK_ACCEL


# ==================== APEx NETWORK STATION CLASSIFICATIONS ====================
# Extracted from APEx_Route_Stations2.xlsx - ALL ROUTES
# These sets determine dwell times based on service pattern
# =============================================================================
# STATION SERVICE LEVEL DETECTION
# Service levels are determined from KMZ folder structure
# =============================================================================

def get_station_service_level(station_name: str, station: 'Station' = None) -> str:
    """
    Determine service level for a station.
    
    For APEX service type:
    - Uses KMZ folder-based service level (folders named Limited/Express/Local)
    - Falls back to station properties (is_terminus, is_limited, is_major, is_express_stop)
    
    For METRO/REGIONAL service types:
    - Uses station properties
    - Falls back to name pattern matching
    
    Returns: 'LIMITED', 'EXPRESS', or 'LOCAL'
    """
    # First priority: KMZ folder-based service level
    if station is not None and hasattr(station, 'folder_service_level') and station.folder_service_level:
        return station.folder_service_level
    
    # Second priority: station object properties
    if station is not None:
        if station.is_terminus or getattr(station, 'is_limited', False):
            return 'LIMITED'
        elif station.is_major or getattr(station, 'is_express_stop', False):
            return 'EXPRESS'
    
    # Fallback for METRO/REGIONAL: check name patterns
    if SERVICE_TYPE in ("METRO", "REGIONAL"):
        name_lower = station_name.lower().strip()
        terminus_patterns = ['union', 'central', 'main', 'terminal', 'hub', 'junction']
        major_patterns = ['downtown', 'airport', 'university', 'transit center']
        
        for pattern in terminus_patterns:
            if pattern in name_lower:
                return 'LIMITED'
        for pattern in major_patterns:
            if pattern in name_lower:
                return 'EXPRESS'
    
    # Default to LOCAL
    return 'LOCAL'


def get_dwell_time_by_service(station_name: str, is_terminus: bool = False, station: 'Station' = None) -> int:
    """
    Get dwell time based on station's service classification and SERVICE_TYPE.
    
    Dwell times vary by SERVICE_TYPE:
    - METRO: 30s/60s/120s (local/express/limited)
    - REGIONAL: 60s/120s/300s
    - APEX: 300s/600s/900s
    """
    dwell_times = get_dwell_times()
    
    if is_terminus:
        return dwell_times['LIMITED']
    
    service_level = get_station_service_level(station_name, station)
    return dwell_times[service_level]

# ==================== TRAIN TYPE CONFIGURATIONS ====================

TRAIN_TYPES = {

    # EMU REGIONAL RAIL
        "STADLER_KISS": {
        "name": "Stadler KISS (Regional Rail)",
        "description": "Double-decker regional train",
        "category": "regional",
        "metric": {
            "max_speed": 177.0,  # km/h (110 mph)
            "acceleration": 1.0,  # m/s²
            "service_deceleration": 1.2,  # m/s²
            "emergency_deceleration": 2.5,  # m/s²
            "jerk_limit": 0.5,  # m/s³
            "min_curve_radius": 300,  # meters
            "max_cant": 150,  # mm - LIMITED for freight compatibility
            "gauge": 1435,  # mm
            "max_gradient_speed_reduction": 0.5,
            "passenger_comfort_factor": 0.85,
            "train_length": 100,  # meters
            "signal_headway": 120,  # seconds minimum between trains
            "station_approach_distance": 1000,  # meters to start braking
            "cant_deficiency_allowed": 100,  # mm - conservative for freight
        },
        "imperial": {
            "max_speed": 110.0,  # mph
            "acceleration": 3.28,  # ft/s²
            "service_deceleration": 3.94,  # ft/s²
            "emergency_deceleration": 8.2,  # ft/s²
            "jerk_limit": 1.64,  # ft/s³
            "min_curve_radius": 984,  # feet
            "max_cant": 6,  # inches - LIMITED for freight
            "gauge": 56.5,  # inches
            "max_gradient_speed_reduction": 0.5,
            "passenger_comfort_factor": 0.85,
            "train_length": 328,  # feet
            "signal_headway": 120,  # seconds
            "station_approach_distance": 3280,  # feet
            "cant_deficiency_allowed": 4,  # inches
        },
        "station_config": {
            "default_dwell_time": 300,
            "terminus_dwell_time": 600,
            "major_station_dwell_time": 300,
            "minor_station_dwell_time": 30,
            "express_penalty": 0,
        }
    },
    
    "M9": {
        "name": "Kawasaki M9 (LIRR/Metro-North)",
        "description": "Electric multiple unit for NYC commuter rail",
        "category": "regional",
        "metric": {
            "max_speed": 161.0,  # km/h (100 mph)
            "acceleration": 0.9,  # m/s²
            "service_deceleration": 1.1,  # m/s²
            "emergency_deceleration": 2.7,  # m/s²
            "jerk_limit": 0.45,  # m/s³
            "min_curve_radius": 300,  # meters - corrected
            "max_cant": 150,  # mm - LIMITED for freight
            "gauge": 1435,  # mm
            "max_gradient_speed_reduction": 0.5,
            "passenger_comfort_factor": 0.82,
            "train_length": 85,  # meters
            "signal_headway": 120,  # seconds
            "station_approach_distance": 900,  # meters
            "cant_deficiency_allowed": 100,  # mm
        },
        "imperial": {
            "max_speed": 100.0,  # mph
            "acceleration": 2.95,  # ft/s²
            "service_deceleration": 3.61,  # ft/s²
            "emergency_deceleration": 8.86,  # ft/s²
            "jerk_limit": 1.48,  # ft/s³
            "min_curve_radius": 984,  # feet - corrected
            "max_cant": 6,  # inches - LIMITED for freight
            "gauge": 56.5,  # inches
            "max_gradient_speed_reduction": 0.5,
            "passenger_comfort_factor": 0.82,
            "train_length": 279,  # feet
            "signal_headway": 120,  # seconds
            "station_approach_distance": 2950,  # feet
            "cant_deficiency_allowed": 4,  # inches
        },
        "station_config": {
            "default_dwell_time": 300,
            "terminus_dwell_time": 600,
            "major_station_dwell_time": 300,
            "minor_station_dwell_time": 30,
            "express_penalty": 0,
        }
    },

    

    # METRO ROLLING STOCK
    
    "RATP_MP14": {
        "name": "RATP MP 14 (Alstom Metropolis, rubber-tyred)",
        "description": "Automatic rubber-tyred heavy metro for Paris",
        "category": "metro",
        "metric": {
            "max_speed": 100,  # km/h service (design max 100) - published
            "acceleration": 1.4,  # m/s² - published (4.86 km/h/s)
            "service_deceleration": 1.2,  # m/s² - ESTIMATE
            "emergency_deceleration": 1.39,  # m/s² - published (5 km/h/s emergency brake)
            "jerk_limit": 0.75,  # m/s³ - ESTIMATE
            "min_curve_radius": 80,  # meters - ESTIMATE (modern metro)
            "max_cant": 120,  # mm - ESTIMATE
            "gauge": 1435,  # mm - published (rubber pads outside steel rails)
            "max_gradient_speed_reduction": 0.3,  # ESTIMATE (rubber = strong grade climbing)
            "passenger_comfort_factor": 0.75,  # ESTIMATE
            "train_length": 120,  # meters (8-car Line 14; 6-car=90, 5-car=75)
            "signal_headway": 85,  # seconds - automatic operation (Line 14 real-world)
            "station_approach_distance": 300,  # meters - ESTIMATE
            "cant_deficiency_allowed": 150,  # mm - ESTIMATE (metro)
        },
        "imperial": {
            "max_speed": 62,  # mph (design 62)
            "acceleration": 4.6,  # ft/s²
            "service_deceleration": 3.94,  # ft/s² - ESTIMATE
            "emergency_deceleration": 4.56,  # ft/s²
            "jerk_limit": 2.46,  # ft/s³ - ESTIMATE
            "min_curve_radius": 262,  # feet - ESTIMATE
            "max_cant": 4.7,  # inches - ESTIMATE
            "gauge": 56.5,  # inches
            "max_gradient_speed_reduction": 0.3,  # ESTIMATE
            "passenger_comfort_factor": 0.75,  # ESTIMATE
            "train_length": 394,  # feet
            "signal_headway": 85,  # seconds
            "station_approach_distance": 984,  # feet - ESTIMATE
            "cant_deficiency_allowed": 5.9,  # inches - ESTIMATE
        },
        "station_config": {
            "default_dwell_time": 25,
            "terminus_dwell_time": 240,
            "major_station_dwell_time": 40,
            "minor_station_dwell_time": 20,
            "express_penalty": 0,
        }
    },
    
    "MBTA_ORANGE": {
        "name": "MBTA Orange Line (CRRC)",
        "description": "Heavy rail metro for Boston",
        "category": "metro",
        "metric": {
            "max_speed": 96.6,  # km/h (60 mph)
            "acceleration": 1.34,  # m/s²
            "service_deceleration": 1.34,  # m/s²
            "emergency_deceleration": 3.0,  # m/s²
            "jerk_limit": 0.75,  # m/s³
            "min_curve_radius": 85,  # meters - corrected for tight curves
            "max_cant": 175,  # mm - MAXIMIZED for dedicated passenger
            "gauge": 1435,  # mm
            "max_gradient_speed_reduction": 0.4,
            "passenger_comfort_factor": 0.75,
            "train_length": 65,  # meters
            "signal_headway": 90,  # seconds
            "station_approach_distance": 400,  # meters
            "cant_deficiency_allowed": 125,  # mm - higher for metro
        },
        "imperial": {
            "max_speed": 60.0,  # mph
            "acceleration": 4.4,  # ft/s²
            "service_deceleration": 4.4,  # ft/s²
            "emergency_deceleration": 9.8,  # ft/s²
            "jerk_limit": 2.46,  # ft/s³
            "min_curve_radius": 279,  # feet - corrected
            "max_cant": 7,  # inches - MAXIMIZED
            "gauge": 56.5,  # inches
            "max_gradient_speed_reduction": 0.4,
            "passenger_comfort_factor": 0.75,
            "train_length": 213,  # feet
            "signal_headway": 90,  # seconds
            "station_approach_distance": 1312,  # feet
            "cant_deficiency_allowed": 5,  # inches
        },
        "station_config": {
            "default_dwell_time": 30,
            "terminus_dwell_time": 300,
            "major_station_dwell_time": 45,
            "minor_station_dwell_time": 30,
            "express_penalty": 0,
        }
    },
    
    "MBTA_RED": {
        "name": "MBTA Red Line (CRRC)",
        "description": "Heavy rail metro for Boston Red Line",
        "category": "metro",
        "metric": {
            "max_speed": 105.0,  # km/h (65 mph)
            "acceleration": 1.34,  # m/s²
            "service_deceleration": 1.34,  # m/s²
            "emergency_deceleration": 3.0,  # m/s²
            "jerk_limit": 0.75,  # m/s³
            "min_curve_radius": 120,  # meters
            "max_cant": 175,  # mm - MAXIMIZED for dedicated passenger
            "gauge": 1435,  # mm
            "max_gradient_speed_reduction": 0.4,
            "passenger_comfort_factor": 0.75,
            "train_length": 70,  # meters
            "signal_headway": 90,  # seconds
            "station_approach_distance": 450,  # meters
            "cant_deficiency_allowed": 125,  # mm
        },
        "imperial": {
            "max_speed": 65.0,  # mph
            "acceleration": 4.4,  # ft/s²
            "service_deceleration": 4.4,  # ft/s²
            "emergency_deceleration": 9.8,  # ft/s²
            "jerk_limit": 2.46,  # ft/s³
            "min_curve_radius": 394,  # feet
            "max_cant": 7,  # inches - MAXIMIZED
            "gauge": 56.5,  # inches
            "max_gradient_speed_reduction": 0.4,
            "passenger_comfort_factor": 0.75,
            "train_length": 230,  # feet
            "signal_headway": 90,  # seconds
            "station_approach_distance": 1476,  # feet
            "cant_deficiency_allowed": 5,  # inches
        },
        "station_config": {
            "default_dwell_time": 30,
            "terminus_dwell_time": 300,
            "major_station_dwell_time": 45,
            "minor_station_dwell_time": 30,
            "express_penalty": 0,
        }
    },
    
    "BART": {
        "name": "BART (Fleet of the Future)",
        "description": "Bay Area Rapid Transit",
        "category": "metro",
        "metric": {
            "max_speed": 129.0,  # km/h (80 mph)
            "acceleration": 1.34,  # m/s²
            "service_deceleration": 1.34,  # m/s²
            "emergency_deceleration": 3.2,  # m/s²
            "jerk_limit": 0.6,  # m/s³
            "min_curve_radius": 200,  # meters - corrected for 80mph
            "max_cant": 175,  # mm - MAXIMIZED for dedicated passenger
            "gauge": 1676,  # mm (broad gauge)
            "max_gradient_speed_reduction": 0.45,
            "passenger_comfort_factor": 0.80,
            "train_length": 105,  # meters
            "signal_headway": 90,  # seconds
            "station_approach_distance": 600,  # meters
            "cant_deficiency_allowed": 150,  # mm - higher due to broad gauge
        },
        "imperial": {
            "max_speed": 80.0,  # mph
            "acceleration": 4.4,  # ft/s²
            "service_deceleration": 4.4,  # ft/s²
            "emergency_deceleration": 10.5,  # ft/s²
            "jerk_limit": 1.97,  # ft/s³
            "min_curve_radius": 656,  # feet - corrected
            "max_cant": 7,  # inches - MAXIMIZED
            "gauge": 66,  # inches (broad gauge)
            "max_gradient_speed_reduction": 0.45,
            "passenger_comfort_factor": 0.80,
            "train_length": 344,  # feet
            "signal_headway": 90,  # seconds
            "station_approach_distance": 1968,  # feet
            "cant_deficiency_allowed": 6,  # inches
        },
        "station_config": {
            "default_dwell_time": 30,
            "terminus_dwell_time": 300,
            "major_station_dwell_time": 45,
            "minor_station_dwell_time": 30,
            "express_penalty": 0,
        }
    },
    
    "NYC_R211": {
        "name": "NYC Subway R211",
        "description": "Latest NYC subway cars",
        "category": "metro",
        "metric": {
            "max_speed": 89.0,  # km/h (55 mph)
            "acceleration": 1.2,  # m/s²
            "service_deceleration": 1.34,  # m/s²
            "emergency_deceleration": 3.0,  # m/s²
            "jerk_limit": 0.75,  # m/s³
            "min_curve_radius": 75,  # meters - very tight curves
            "max_cant": 175,  # mm - MAXIMIZED for dedicated passenger
            "gauge": 1435,  # mm
            "max_gradient_speed_reduction": 0.35,
            "passenger_comfort_factor": 0.72,
            "train_length": 75,  # meters
            "signal_headway": 90,  # seconds
            "station_approach_distance": 350,  # meters
            "cant_deficiency_allowed": 125,  # mm
        },
        "imperial": {
            "max_speed": 55.0,  # mph
            "acceleration": 3.94,  # ft/s²
            "service_deceleration": 4.4,  # ft/s²
            "emergency_deceleration": 9.8,  # ft/s²
            "jerk_limit": 2.46,  # ft/s³
            "min_curve_radius": 246,  # feet - corrected
            "max_cant": 7,  # inches - MAXIMIZED
            "gauge": 56.5,  # inches
            "max_gradient_speed_reduction": 0.35,
            "passenger_comfort_factor": 0.72,
            "train_length": 246,  # feet
            "signal_headway": 90,  # seconds
            "station_approach_distance": 1148,  # feet
            "cant_deficiency_allowed": 5,  # inches
        },
        "station_config": {
            "default_dwell_time": 30,
            "terminus_dwell_time": 300,
            "major_station_dwell_time": 45,
            "minor_station_dwell_time": 30,
            "express_penalty": 0,
        }
    },
    
    "WMATA": {
        "name": "WMATA 7000-Series",
        "description": "Washington DC Metro",
        "category": "metro",
        "metric": {
            "max_speed": 121.0,  # km/h (75 mph)
            "acceleration": 1.34,  # m/s²
            "service_deceleration": 1.34,  # m/s²
            "emergency_deceleration": 3.0,  # m/s²
            "jerk_limit": 0.6,  # m/s³
            "min_curve_radius": 150,  # meters - corrected for 75mph
            "max_cant": 175,  # mm - MAXIMIZED for dedicated passenger
            "gauge": 1435,  # mm
            "max_gradient_speed_reduction": 0.42,
            "passenger_comfort_factor": 0.78,
            "train_length": 95,  # meters
            "signal_headway": 90,  # seconds
            "station_approach_distance": 500,  # meters
            "cant_deficiency_allowed": 125,  # mm
        },
        "imperial": {
            "max_speed": 75.0,  # mph
            "acceleration": 4.4,  # ft/s²
            "service_deceleration": 4.4,  # ft/s²
            "emergency_deceleration": 9.8,  # ft/s²
            "jerk_limit": 1.97,  # ft/s³
            "min_curve_radius": 492,  # feet - corrected
            "max_cant": 7,  # inches - MAXIMIZED
            "gauge": 56.5,  # inches
            "max_gradient_speed_reduction": 0.42,
            "passenger_comfort_factor": 0.78,
            "train_length": 312,  # feet
            "signal_headway": 90,  # seconds
            "station_approach_distance": 1640,  # feet
            "cant_deficiency_allowed": 5,  # inches
        },
        "station_config": {
            "default_dwell_time": 30,
            "terminus_dwell_time": 300,
            "major_station_dwell_time": 45,
            "minor_station_dwell_time": 30,
            "express_penalty": 0,
        }
    },

    # HSR ROLLING STOCK
    
    "METROLINER": {
        "name": "Budd Metroliner",
        "description": "High-speed EMU for Northeast Corridor (1960s-2000s)",
        "category": "regional",
        "metric": {
            "max_speed": 241.0,  # km/h (150 mph) - design speed, often limited to 110
            "acceleration": 0.8,  # m/s² - 1960s high-speed EMU
            "service_deceleration": 1.2,  # m/s²
            "emergency_deceleration": 2.5,  # m/s²
            "jerk_limit": 0.4,  # m/s³
            "min_curve_radius": 300,  # meters - NEC curves
            "max_cant": 150,  # mm
            "gauge": 1435,  # mm
            "max_gradient_speed_reduction": 0.48,
            "passenger_comfort_factor": 0.77,  # 1960s ride quality at high speed
            "train_length": 85,  # meters (single car, typically 4-6 car consists)
            "signal_headway": 140,  # seconds
            "station_approach_distance": 1000,  # meters
            "cant_deficiency_allowed": 100,  # mm
        },
        "imperial": {
            "max_speed": 150.0,  # mph (often limited to 110 mph in service)
            "acceleration": 2.62,  # ft/s²
            "service_deceleration": 3.8,  # ft/s²
            "emergency_deceleration": 8.20,  # ft/s²
            "jerk_limit": 1.31,  # ft/s³
            "min_curve_radius": 985,  # feet
            "max_cant": 6,  # inches
            "gauge": 56.5,  # inches
            "max_gradient_speed_reduction": 0.48,
            "passenger_comfort_factor": 0.85,
            "train_length": 279,  # feet
            "signal_headway": 140,  # seconds
            "station_approach_distance": 3280,  # feet
            "cant_deficiency_allowed": 6,  # inches
        },
        "station_config": {
            "default_dwell_time": 300,  # intercity service
            "terminus_dwell_time": 600,
            "major_station_dwell_time": 300,
            "minor_station_dwell_time": 300,
            "express_penalty": 0,
        }
    },
    
    "AVELIA_LIBERTY": {
        "name": "Alstom Avelia Liberty (Acela II)",
        "description": "High-speed train with active tilt for Northeast Corridor",
        "category": "hsr",
        "metric": {
            "max_speed": 300.0,  # km/h (186 mph) - design speed, operational 257 km/h
            "acceleration": 0.7,  # m/s² - typical HSR acceleration
            "service_deceleration": 0.9,  # m/s²
            "emergency_deceleration": 1.5,  # m/s²
            "jerk_limit": 0.35,  # m/s³ - smooth for HSR
            "min_curve_radius": 500,  # meters - can handle tighter with tilt
            "max_cant": 180,  # mm - maximum for dedicated HSR ROW
            "gauge": 1435,  # mm
            "max_gradient_speed_reduction": 0.3,  # better hill climbing
            "passenger_comfort_factor": 0.95,  # high comfort
            "train_length": 200,  # meters
            "signal_headway": 180,  # seconds - ETCS Level 2
            "station_approach_distance": 2000,  # meters - longer for HSR
            "cant_deficiency_allowed": 178,  # mm - with active tilt (7 inches)
            "tilt_angle": 6.3,  # degrees - active tilt capability (mechanism: 6.6°)
            "max_tilt_speed": 240.0,  # km/h - v9 tilt rate limiting
            "max_tilt_rate": 4.0,  # degrees/second
        },
        "imperial": {
            "max_speed": 186.0,  # mph
            "acceleration": 2.3,  # ft/s²
            "service_deceleration": 2.95,  # ft/s²
            "emergency_deceleration": 4.92,  # ft/s²
            "jerk_limit": 1.15,  # ft/s³
            "min_curve_radius": 1640,  # feet
            "max_cant": 7,  # inches - maximum for dedicated HSR
            "gauge": 56.5,  # inches
            "max_gradient_speed_reduction": 0.3,
            "passenger_comfort_factor": 0.95,
            "train_length": 656,  # feet
            "signal_headway": 180,  # seconds
            "station_approach_distance": 6560,  # feet
            "cant_deficiency_allowed": 7,  # inches - with active tilt
            "tilt_angle": 6.3,  # degrees
            "max_tilt_speed": 150.0,  # mph
            "max_tilt_rate": 4.0,  # degrees/second
        },
        "station_config": {
            "default_dwell_time": 300,  # seconds - standard HSR dwell
            "terminus_dwell_time": 600,  # seconds - longer at terminals
            "major_station_dwell_time": 300,
            "minor_station_dwell_time": 60,
            "express_penalty": 00,
        }
    },
    "ETR_1000": {
        "name": "Alstom-Bombardier ETR 1000 (Frecciarossa)",
        "description": "Italian high-speed train with distributed traction",
        "category": "hsr",
        "metric": {
            "max_speed": 360.0,  # km/h (224 mph) - max commercial speed (certified 300, design 400)
            "acceleration": 0.6,  # m/s²
            "service_deceleration": 0.85,  # m/s²
            "emergency_deceleration": 1.4,  # m/s²
            "jerk_limit": 0.3,  # m/s³
            "min_curve_radius": 600,  # meters
            "max_cant": 180,  # mm - dedicated HSR ROW
            "gauge": 1435,  # mm
            "max_gradient_speed_reduction": 0.35,
            "passenger_comfort_factor": 0.93,
            "train_length": 202,  # meters - 8-car configuration
            "signal_headway": 180,  # seconds - ERTMS
            "station_approach_distance": 2500,  # meters
            "cant_deficiency_allowed": 250,  # mm - passive tilt design
            "tilt_angle": 0,  # degrees - no active tilt (has active lateral suspension)
        },
        "imperial": {
            "max_speed": 224.0,  # mph
            "acceleration": 1.97,  # ft/s²
            "service_deceleration": 2.79,  # ft/s²
            "emergency_deceleration": 4.59,  # ft/s²
            "jerk_limit": 0.98,  # ft/s³
            "min_curve_radius": 1969,  # feet
            "max_cant": 7,  # inches
            "gauge": 56.5,  # inches
            "max_gradient_speed_reduction": 0.35,
            "passenger_comfort_factor": 0.93,
            "train_length": 663,  # feet
            "signal_headway": 180,  # seconds
            "station_approach_distance": 8200,  # feet
            "cant_deficiency_allowed": 10,  # inches
            "tilt_angle": 0,  # degrees
        },
        "station_config": {
            "default_dwell_time": 300,  # seconds - standard HSR dwell
            "terminus_dwell_time": 600,  # seconds - longer at terminals
            "major_station_dwell_time": 300,
            "minor_station_dwell_time": 60,
            "express_penalty": 00,
        }
    },

    "AGV_USA": {
        "name": "Alstom AGV Tilting (US Mixed Infrastructure HSR)",
        "description": "Articulated very-high-speed EMU with selective tilting for mixed legacy/HSR infrastructure",
        "category": "very_high_speed_tilting",
        "metric": {
            "max_speed": 402.336,  # km/h (250 mph) - standard cruising speed
            "max_speed_recovery": 450.6163,  # km/h (280 mph) - schedule recovery mode
            #"max_speed_with_tilt": 320.0,  # km/h (200 mph) - tilting mode on legacy track
            "acceleration": 2.5,  # m/s² - Gen 6 peak acceleration (S-curve profile)
            "service_deceleration": 2.0,  # m/s² - Gen 6 service braking
            "emergency_deceleration": 2.8,  # m/s² - HSR safety standards
            "jerk_limit": 0.25,  # m/s³ - Gen 6 S-curve jerk limit for comfort
            "min_curve_radius": 800,  # meters - with tilting engaged
            "min_curve_radius_hsr": 4000,  # meters - at max speed (no tilt)
            "max_cant": 180,  # mm - European/US compatible
            "gauge": 1435,  # mm - standard gauge
            "max_gradient_speed_reduction": 0.25,  # excellent distributed traction
            "passenger_comfort_factor": 0.96,  # articulated design + active suspension
            "train_length": 200,  # meters (11-car articulated configuration)
            "signal_headway": 180,  # seconds - HSR spacing
            "station_approach_distance": 2500,  # meters - high-speed braking
            "cant_deficiency_allowed": 90,  # mm - comfortable HSR (0.6 m/s² lateral, matches China standards)
            "tilt_angle": 7.0,  # degrees - active tilting (conservative for HSR)
            "max_tilt_speed": 265.0,  # km/h - v9 tilt rate limiting
            "max_tilt_rate": 4.0,  # degrees/second
            "tilt_compensation": 0.70,  # 70% lateral force compensation
            "tilt_engagement_speed": 265.0,  # km/h - disables above this for safety
            "tilt_disengagement_speed": 320.0,  # km/h - fully disabled at max speed
            "seating_capacity": 450,  # 11-car articulated
            "standing_capacity": 150,  # limited on HSR
            "door_cycle_time": 15,  # seconds - HSR station stops
            "doors_per_side": 11,  # 1 per articulated segment
            "power_to_weight": 26.0,  # kW/tonne - high for acceleration
            "total_power": 11000,  # kW - distributed across multiple bogies
            "motorized_bogies_percent": 100,  # all bogies powered
        },
        "imperial": {
            "max_speed": 250.0,  # mph - standard cruising speed
            "max_speed_recovery": 280.0,  # mph - schedule recovery mode
            #"max_speed_with_tilt": 199.0,  # mph - legacy track sections
            "acceleration": 8.20,  # ft/s² - Gen 6 peak (2.5 m/s²)
            "service_deceleration": 6.56,  # ft/s² - Gen 6 braking (2.0 m/s²)
            "emergency_deceleration": 9.19,  # ft/s²
            "jerk_limit": 0.82,  # ft/s³ - Gen 6 (0.25 m/s³)
            "min_curve_radius": 2625,  # feet - with tilting
            "min_curve_radius_hsr": 13123,  # feet - straight HSR sections
            "max_cant": 7.1,  # inches
            "gauge": 56.5,  # inches
            "max_gradient_speed_reduction": 0.25,
            "passenger_comfort_factor": 0.96,
            "train_length": 656,  # feet
            "signal_headway": 180,  # seconds
            "station_approach_distance": 8202,  # feet
            "cant_deficiency_allowed": 3.5,  # inches (90mm - comfortable HSR)
            "tilt_angle": 7.0,  # degrees
            "max_tilt_speed": 165.0,  # mph
            "max_tilt_rate": 4.0,  # degrees/second
            "tilt_compensation": 0.70,
            "tilt_engagement_speed": 165.0,  # mph
            "tilt_disengagement_speed": 199.0,  # mph
            "seating_capacity": 450,
            "standing_capacity": 150,
            "door_cycle_time": 15,  # seconds
            "doors_per_side": 11,
            "power_to_weight": 26.0,  # kW/tonne
            "total_power": 14751,  # hp
            "motorized_bogies_percent": 100,
        },
        "station_config": {
            "default_dwell_time": 300,  # HSR major stations
            "terminus_dwell_time": 600,  # longer for HSR turnaround
            "major_station_dwell_time": 600,
            "minor_station_dwell_time": 300,  # limited stops
            "express_penalty": 0,  # higher for HSR operations
        }
    },
    
    "AGV_USA_TILT": {
        "name": "Alstom AGV with Active Tilt",
        "description": "250mph train with tilt for legacy curves",
        "category": "hsr",
        "metric": {
            "max_speed": 402.0,
            "acceleration": 0.5,
            "service_deceleration": 0.8,
            "emergency_deceleration": 1.3,
            "jerk_limit": 0.3,
            "min_curve_radius": 500,
            "max_cant": 180,
            "gauge": 1435,
            "max_gradient_speed_reduction": 0.06,
            "passenger_comfort_factor": 0.95,
            "train_length": 200,
            "signal_headway": 180,
            "station_approach_distance": 3000,
            "cant_deficiency_allowed": 200,
            "tilt_angle": 8.0,  # Active tilt
            "max_tilt_speed": 320.0,  # km/h - v9 tilt rate limiting
            "max_tilt_rate": 4.0,  # degrees/second
        },
        "imperial": {
            "max_speed": 250.0,
            "acceleration": 1.64,
            "service_deceleration": 2.62,
            "emergency_deceleration": 4.27,
            "jerk_limit": 0.98,
            "min_curve_radius": 1640,
            "max_cant": 7,
            "gauge": 56.5,
            "max_gradient_speed_reduction": 0.06,
            "passenger_comfort_factor": 0.95,
            "train_length": 656,
            "signal_headway": 180,
            "station_approach_distance": 9843,
            "cant_deficiency_allowed": 8,
            "tilt_angle": 8.0,
            "max_tilt_speed": 200.0,  # mph
            "max_tilt_rate": 4.0,  # degrees/second
        },
        "station_config": {
            "default_dwell_time": 90,
            "terminus_dwell_time": 180,
            "major_station_dwell_time": 120,
            "minor_station_dwell_time": 60,
            "express_penalty": 10,
        }
    },
    
    "ORBIT_610": {
        "name": "Alstom New Pendolino 610 (Ohio Regional)",
        "description": "7-car modular tilting EMU optimized for 155 mph regional intercity with Tiltronix",
        "category": "high_speed_regional_tilting",
        "metric": {
            "max_speed": 300.0,  # km/h (155 mph) - regional HSR
            "acceleration": 1.9,  # m/s² - excellent distributed traction
            "service_deceleration": 1.6,  # m/s² - regenerative braking optimized
            "emergency_deceleration": 3.1,  # m/s² - modern safety standards
            "jerk_limit": 0.70,  # m/s³ - passenger comfort optimized
            "min_curve_radius": 180,  # meters - 8° tilt enables tight curves
            "max_cant": 160,  # mm - North American standard
            "gauge": 1435,  # mm - standard gauge
            "max_gradient_speed_reduction": 0.28,  # excellent multi-voltage traction
            "passenger_comfort_factor": 0.95,  # Tiltronix + premium regional suspension
            "train_length": 187,  # meters (7-car standard configuration)
            "signal_headway": 120,  # seconds - regional service
            "station_approach_distance": 900,  # meters - 155 mph deceleration
            "cant_deficiency_allowed": 340,  # mm - 8° active Tiltronix
            "tilt_angle": 8.0,  # degrees - highest in class
            "max_tilt_speed": 250.0,  # km/h - v9 tilt rate limiting
            "max_tilt_rate": 5.0,  # degrees/second
            "tilt_compensation": 0.75,  # 75% lateral force compensation
            "tilt_response_time": 0.7,  # seconds - predictive system
            "seating_capacity": 430,  # 7-car mixed 1st/2nd class
            "standing_capacity": 120,  # regional intercity allows standing
            "door_cycle_time": 12,  # seconds - regional stops
            "doors_per_side": 14,  # 2 per car
            "door_width": 1300,  # mm - wide for passenger flow
            "power_to_weight": 23.0,  # kW/tonne
            "total_power": 5500,  # kW - distributed traction
            "motorized_bogies_percent": 50,  # every other bogie
            "voltage_systems": ["3000V DC", "15kV AC", "25kV AC"],  # multi-voltage
        },
        "imperial": {
            "max_speed": 186.0,  # mph - ORBIT requirement
            "acceleration": 6.23,  # ft/s² - excellent for regional
            "service_deceleration": 5.25,  # ft/s²
            "emergency_deceleration": 10.17,  # ft/s²
            "jerk_limit": 2.30,  # ft/s³
            "min_curve_radius": 591,  # feet - tight curve capability
            "max_cant": 6.3,  # inches
            "gauge": 56.5,  # inches
            "max_gradient_speed_reduction": 0.28,
            "passenger_comfort_factor": 0.95,
            "train_length": 614,  # feet
            "signal_headway": 120,  # seconds
            "station_approach_distance": 2953,  # feet
            "cant_deficiency_allowed": 13.4,  # inches
            "tilt_angle": 8.0,  # degrees
            "max_tilt_speed": 155.0,  # mph
            "max_tilt_rate": 5.0,  # degrees/second
            "tilt_compensation": 0.75,
            "tilt_response_time": 0.7,  # seconds
            "seating_capacity": 430,
            "standing_capacity": 120,
            "door_cycle_time": 12,  # seconds
            "doors_per_side": 14,
            "door_width": 51.2,  # inches
            "power_to_weight": 23.0,  # kW/tonne
            "total_power": 7375,  # hp
            "motorized_bogies_percent": 50,
        },
        "station_config": {
            "default_dwell_time": 300,  # regional intercity
            "terminus_dwell_time": 600,  # turnaround and servicing
            "major_station_dwell_time": 120,  # cities >30k population
            "minor_station_dwell_time": 90,  # smaller stops
            "express_penalty": 0,  # low due to excellent acceleration
        },
    },

    #DIESEL ROLLING STOCK
    
        "MP36PH": {
        "name": "EMD MP36PH-3S (Metra/MBTA)",
        "description": "Diesel-electric locomotive used by Metra and MBTA",
        "category": "regional",
        "metric": {
            "max_speed": 145.0,  # km/h (90 mph)
            "acceleration": 0.45,  # m/s² - diesel lower than electric
            "service_deceleration": 0.9,  # m/s²
            "emergency_deceleration": 2.1,  # m/s²
            "jerk_limit": 0.35,  # m/s³
            "min_curve_radius": 350,  # meters
            "max_cant": 150,  # mm - freight compatible
            "gauge": 1435,  # mm
            "max_gradient_speed_reduction": 0.6,  # diesel more affected by grades
            "passenger_comfort_factor": 0.78,  # lower due to diesel vibration
            "train_length": 145,  # meters (typical 5-car consist)
            "signal_headway": 150,  # seconds
            "station_approach_distance": 1200,  # meters - longer for diesel
            "cant_deficiency_allowed": 100,  # mm
        },
        "imperial": {
            "max_speed": 90.0,  # mph
            "acceleration": 1.48,  # ft/s²
            "service_deceleration": 2.95,  # ft/s²
            "emergency_deceleration": 6.89,  # ft/s²
            "jerk_limit": 1.15,  # ft/s³
            "min_curve_radius": 1148,  # feet
            "max_cant": 6,  # inches
            "gauge": 56.5,  # inches
            "max_gradient_speed_reduction": 0.6,
            "passenger_comfort_factor": 0.78,
            "train_length": 476,  # feet
            "signal_headway": 150,  # seconds
            "station_approach_distance": 3937,  # feet
            "cant_deficiency_allowed": 4,  # inches
        },
        "station_config": {
            "default_dwell_time": 300,
            "terminus_dwell_time": 600,
            "major_station_dwell_time": 300,
            "minor_station_dwell_time": 300,
            "express_penalty": 0,
        }
    },
    
    "F40PH": {
        "name": "EMD F40PH/F40PHM-2 (Metra/MBTA)",
        "description": "Classic diesel locomotive widely used in commuter service",
        "category": "regional",
        "metric": {
            "max_speed": 127.0,  # km/h (110 mph) - though often limited to 79 mph
            "acceleration": 0.2,  # m/s² - older design, lower acceleration
            "service_deceleration": 0.4,  # m/s²
            "emergency_deceleration": 2.0,  # m/s²
            "jerk_limit": 0.3,  # m/s³
            "min_curve_radius": 350,  # meters
            "max_cant": 150,  # mm
            "gauge": 1435,  # mm
            "max_gradient_speed_reduction": 0.65,  # older diesel more affected
            "passenger_comfort_factor": 0.75,  # older design
            "train_length": 140,  # meters
            "signal_headway": 150,  # seconds
            "station_approach_distance": 1300,  # meters
            "cant_deficiency_allowed": 100,  # mm
        },
        "imperial": {
            "max_speed": 79.0,  # mph (often limited to 79 mph in service)
            "acceleration": 0.656,  # ft/s²
            "service_deceleration": 1.31,  # ft/s²
            "emergency_deceleration": 6.56,  # ft/s²
            "jerk_limit": 0.98,  # ft/s³
            "min_curve_radius": 1148,  # feet
            "max_cant": 6,  # inches
            "gauge": 56.5,  # inches
            "max_gradient_speed_reduction": 0.65,
            "passenger_comfort_factor": 0.75,
            "train_length": 459,  # feet
            "signal_headway": 150,  # seconds
            "station_approach_distance": 4265,  # feet
            "cant_deficiency_allowed": 4,  # inches
        },
        "station_config": {
            "default_dwell_time": 300,
            "terminus_dwell_time": 600,
            "major_station_dwell_time": 300,
            "minor_station_dwell_time": 300,
            "express_penalty": 0,
        }
    },
    
    "DE30AC": {
        "name": "Bombardier DE30AC/DM30AC (LIRR)",
        "description": "Dual-mode locomotive for LIRR diesel/electric service",
        "category": "regional",
        "metric": {
            "max_speed": 161.0,  # km/h (100 mph)
            "acceleration": 0.5,  # m/s² - better than pure diesel
            "service_deceleration": 0.95,  # m/s²
            "emergency_deceleration": 2.2,  # m/s²
            "jerk_limit": 0.4,  # m/s³
            "min_curve_radius": 320,  # meters
            "max_cant": 150,  # mm
            "gauge": 1435,  # mm
            "max_gradient_speed_reduction": 0.55,  # dual mode helps on grades
            "passenger_comfort_factor": 0.80,  # newer design
            "train_length": 155,  # meters (typical consist)
            "signal_headway": 140,  # seconds
            "station_approach_distance": 1100,  # meters
            "cant_deficiency_allowed": 100,  # mm
        },
        "imperial": {
            "max_speed": 100.0,  # mph
            "acceleration": 1.64,  # ft/s²
            "service_deceleration": 3.12,  # ft/s²
            "emergency_deceleration": 7.22,  # ft/s²
            "jerk_limit": 1.31,  # ft/s³
            "min_curve_radius": 1050,  # feet
            "max_cant": 6,  # inches
            "gauge": 56.5,  # inches
            "max_gradient_speed_reduction": 0.55,
            "passenger_comfort_factor": 0.80,
            "train_length": 509,  # feet
            "signal_headway": 140,  # seconds
            "station_approach_distance": 3609,  # feet
            "cant_deficiency_allowed": 4,  # inches
        },
        "station_config": {
            "default_dwell_time": 300,
            "terminus_dwell_time": 600,
            "major_station_dwell_time": 300,
            "minor_station_dwell_time": 300,
            "express_penalty": 0,
        }
    },
    
    "HSP46": {
        "name": "MPI HSP46 (MBTA)",
        "description": "Modern high-speed diesel locomotive",
        "category": "regional",
        "metric": {
            "max_speed": 201.0,  # km/h (125 mph) capable
            "acceleration": 0.55,  # m/s² - modern high-power diesel
            "service_deceleration": 1.0,  # m/s²
            "emergency_deceleration": 2.3,  # m/s²
            "jerk_limit": 0.45,  # m/s³
            "min_curve_radius": 320,  # meters
            "max_cant": 150,  # mm
            "gauge": 1435,  # mm
            "max_gradient_speed_reduction": 0.5,  # powerful diesel
            "passenger_comfort_factor": 0.82,  # modern design
            "train_length": 150,  # meters
            "signal_headway": 140,  # seconds
            "station_approach_distance": 1100,  # meters
            "cant_deficiency_allowed": 100,  # mm
        },
        "imperial": {
            "max_speed": 125.0,  # mph
            "acceleration": 1.80,  # ft/s²
            "service_deceleration": 3.28,  # ft/s²
            "emergency_deceleration": 7.55,  # ft/s²
            "jerk_limit": 1.48,  # ft/s³
            "min_curve_radius": 1050,  # feet
            "max_cant": 6,  # inches
            "gauge": 56.5,  # inches
            "max_gradient_speed_reduction": 0.5,
            "passenger_comfort_factor": 0.82,
            "train_length": 492,  # feet
            "signal_headway": 140,  # seconds
            "station_approach_distance": 3609,  # feet
            "cant_deficiency_allowed": 4,  # inches
        },
        "station_config": {
            "default_dwell_time": 300,
            "terminus_dwell_time": 600,
            "major_station_dwell_time": 300,
            "minor_station_dwell_time": 300,
            "express_penalty": 0,
        }
    },
    
    #  HIGH-PERFORMANCE INTERNATIONAL DIESEL TRAINS 
    
    "SIEMENS_CHARGER": {
        "name": "Siemens Charger SC-44",
        "description": "Modern diesel-electric locomotive (Amtrak/Brightline)",
        "category": "regional",
        "metric": {
            "max_speed": 201.0,  # km/h (125 mph)
            "acceleration": 0.6,  # m/s² - excellent for diesel
            "service_deceleration": 1.05,  # m/s²
            "emergency_deceleration": 2.4,  # m/s²
            "jerk_limit": 0.5,  # m/s³
            "min_curve_radius": 320,  # meters
            "max_cant": 150,  # mm
            "gauge": 1435,  # mm
            "max_gradient_speed_reduction": 0.45,  # efficient modern diesel
            "passenger_comfort_factor": 0.85,  # modern design with good isolation
            "train_length": 160,  # meters
            "signal_headway": 130,  # seconds
            "station_approach_distance": 1050,  # meters
            "cant_deficiency_allowed": 110,  # mm
        },
        "imperial": {
            "max_speed": 125.0,  # mph
            "acceleration": 1.97,  # ft/s²
            "service_deceleration": 3.44,  # ft/s²
            "emergency_deceleration": 7.87,  # ft/s²
            "jerk_limit": 1.64,  # ft/s³
            "min_curve_radius": 1050,  # feet
            "max_cant": 6,  # inches
            "gauge": 56.5,  # inches
            "max_gradient_speed_reduction": 0.45,
            "passenger_comfort_factor": 0.85,
            "train_length": 525,  # feet
            "signal_headway": 130,  # seconds
            "station_approach_distance": 3445,  # feet
            "cant_deficiency_allowed": 4.3,  # inches
        },
        "station_config": {
            "default_dwell_time": 300,
            "terminus_dwell_time": 600,
            "major_station_dwell_time": 300,
            "minor_station_dwell_time": 30,
            "express_penalty": 0,
        }
    },
    
    "STADLER_FLIRT_DMU": {
        "name": "Stadler FLIRT DMU",
        "description": "High-performance diesel multiple unit (European/UK)",
        "category": "regional",
        "metric": {
            "max_speed": 160.0,  # km/h (100 mph)
            "acceleration": 0.7,  # m/s² - excellent for DMU
            "service_deceleration": 1.1,  # m/s²
            "emergency_deceleration": 2.5,  # m/s²
            "jerk_limit": 0.55,  # m/s³
            "min_curve_radius": 250,  # meters - good for branch lines
            "max_cant": 160,  # mm
            "gauge": 1435,  # mm
            "max_gradient_speed_reduction": 0.4,  # lightweight DMU handles grades well
            "passenger_comfort_factor": 0.88,  # modern DMU with good ride quality
            "train_length": 75,  # meters (3-car unit)
            "signal_headway": 120,  # seconds
            "station_approach_distance": 900,  # meters
            "cant_deficiency_allowed": 120,  # mm
        },
        "imperial": {
            "max_speed": 100.0,  # mph
            "acceleration": 2.30,  # ft/s²
            "service_deceleration": 3.61,  # ft/s²
            "emergency_deceleration": 8.20,  # ft/s²
            "jerk_limit": 1.80,  # ft/s³
            "min_curve_radius": 820,  # feet
            "max_cant": 6.3,  # inches
            "gauge": 56.5,  # inches
            "max_gradient_speed_reduction": 0.4,
            "passenger_comfort_factor": 0.88,
            "train_length": 246,  # feet
            "signal_headway": 120,  # seconds
            "station_approach_distance": 2953,  # feet
            "cant_deficiency_allowed": 4.7,  # inches
        },
        "station_config": {
            "default_dwell_time": 300,
            "terminus_dwell_time": 600,
            "major_station_dwell_time": 300,
            "minor_station_dwell_time": 25,
            "express_penalty": 0,
        }
    },
    
    "VELARO_MS": {
        "name": "Siemens Velaro MS",
        "description": "Multi-system high-speed train platform",
        "category": "hsr",
        "metric": {
            "max_speed": 320.0,  # km/h (200 mph) - AC mode service speed (DC: 200 km/h)
            "acceleration": 0.5,  # m/s² - documented acceleration rate
            "service_deceleration": 0.9,  # m/s²
            "emergency_deceleration": 1.45,  # m/s²
            "jerk_limit": 0.32,  # m/s³
            "min_curve_radius": 550,  # meters
            "max_cant": 180,  # mm - dedicated HSR ROW
            "gauge": 1435,  # mm
            "max_gradient_speed_reduction": 0.32,
            "passenger_comfort_factor": 0.94,
            "train_length": 200,  # meters - 8-car
            "signal_headway": 180,  # seconds
            "station_approach_distance": 2200,  # meters
            "cant_deficiency_allowed": 200,  # mm - moderate cant deficiency
            "tilt_angle": 0,  # degrees - no active tilt
        },
        "imperial": {
            "max_speed": 200.0,  # mph
            "acceleration": 1.64,  # ft/s² - corrected from 2.13
            "service_deceleration": 2.95,  # ft/s²
            "emergency_deceleration": 4.76,  # ft/s²
            "jerk_limit": 1.05,  # ft/s³
            "min_curve_radius": 1804,  # feet
            "max_cant": 7,  # inches
            "gauge": 56.5,  # inches
            "max_gradient_speed_reduction": 0.32,
            "passenger_comfort_factor": 0.94,
            "train_length": 656,  # feet
            "signal_headway": 180,  # seconds
            "station_approach_distance": 7220,  # feet
            "cant_deficiency_allowed": 8,  # inches
            "tilt_angle": 0,  # degrees
        },
        "station_config": {
            "default_dwell_time": 300,  # seconds - standard HSR dwell
            "terminus_dwell_time": 600,  # seconds - longer at terminals
            "major_station_dwell_time": 300,
            "minor_station_dwell_time": 60,
            "express_penalty": 00,
        }
    },

        "BUDD_RDC": {
        "name": "Budd RDC1/RDC2",
        "description": "Classic self-propelled diesel railcar (1950s-1980s)",
        "category": "regional",
        "metric": {
            "max_speed": 137.0,  # km/h (85 mph)
            "acceleration": 0.35,  # m/s² - older diesel technology
            "service_deceleration": 0.75,  # m/s²
            "emergency_deceleration": 1.8,  # m/s²
            "jerk_limit": 0.25,  # m/s³ - older technology
            "min_curve_radius": 400,  # meters
            "max_cant": 150,  # mm
            "gauge": 1435,  # mm
            "max_gradient_speed_reduction": 0.7,  # older diesel very affected by grades
            "passenger_comfort_factor": 0.70,  # 1950s ride quality
            "train_length": 26,  # meters (single car)
            "signal_headway": 160,  # seconds
            "station_approach_distance": 1400,  # meters - older braking
            "cant_deficiency_allowed": 90,  # mm
        },
        "imperial": {
            "max_speed": 85.0,  # mph
            "acceleration": 1.15,  # ft/s²
            "service_deceleration": 2.46,  # ft/s²
            "emergency_deceleration": 5.91,  # ft/s²
            "jerk_limit": 0.82,  # ft/s³
            "min_curve_radius": 1312,  # feet
            "max_cant": 6,  # inches
            "gauge": 56.5,  # inches
            "max_gradient_speed_reduction": 0.7,
            "passenger_comfort_factor": 0.70,
            "train_length": 85,  # feet
            "signal_headway": 160,  # seconds
            "station_approach_distance": 4593,  # feet
            "cant_deficiency_allowed": 3.5,  # inches
        },
        "station_config": {
            "default_dwell_time": 300,
            "terminus_dwell_time": 600,
            "major_station_dwell_time": 300,
            "minor_station_dwell_time": 30,
            "express_penalty": 0,
        }
    },
    
    # ==================== REGIONAL ELECTRIC TRAINS ====================
    
    "M8": {
        "name": "Kawasaki M8 (Metro-North/LIRR)",
        "description": "Modern electric multiple unit for NYC area commuter rail",
        "category": "regional",
        "metric": {
            "max_speed": 161.0,  # km/h (100 mph)
            "acceleration": 0.95,  # m/s² - modern EMU
            "service_deceleration": 1.15,  # m/s²
            "emergency_deceleration": 2.8,  # m/s²
            "jerk_limit": 0.5,  # m/s³
            "min_curve_radius": 300,  # meters
            "max_cant": 150,  # mm - freight compatible
            "gauge": 1435,  # mm
            "max_gradient_speed_reduction": 0.45,
            "passenger_comfort_factor": 0.85,
            "train_length": 85,  # meters (single car)
            "signal_headway": 120,  # seconds
            "station_approach_distance": 850,  # meters
            "cant_deficiency_allowed": 100,  # mm
        },
        "imperial": {
            "max_speed": 100.0,  # mph
            "acceleration": 3.12,  # ft/s²
            "service_deceleration": 3.77,  # ft/s²
            "emergency_deceleration": 9.19,  # ft/s²
            "jerk_limit": 1.64,  # ft/s³
            "min_curve_radius": 984,  # feet
            "max_cant": 6,  # inches
            "gauge": 56.5,  # inches
            "max_gradient_speed_reduction": 0.45,
            "passenger_comfort_factor": 0.85,
            "train_length": 279,  # feet
            "signal_headway": 120,  # seconds
            "station_approach_distance": 2789,  # feet
            "cant_deficiency_allowed": 4,  # inches
        },
        "station_config": {
            "default_dwell_time": 300,
            "terminus_dwell_time": 600,
            "major_station_dwell_time": 300,
            "minor_station_dwell_time": 30,
            "express_penalty": 0,
        }
    },

    "PENDOLINO_CVG": {
        "name": "Alstom Avelia Pendolino (Cincinnati CVG Express)",
        "description": "3-car airport express EMU based on ETR 600 platform with enhanced acceleration and active tilting",
        "category": "airport_express",
        "metric": {
            "max_speed": 201.168,  # km/h (100 mph) - optimized for regional/airport service
            "acceleration": 2.1,  # m/s² - 91% better than DB610E due to modern distributed traction
            "service_deceleration": 1.6,  # m/s² - 28% better than DB610E
            "emergency_deceleration": 3.2,  # m/s² - 60% better with modern composite brakes
            "jerk_limit": 0.75,  # m/s³ - improved traction control (vs 0.70)
            "min_curve_radius": 160,  # meters - shorter than DB610E due to modern bogies + tilt
            "max_cant": 160,  # mm - North American standard
            "gauge": 1435,  # mm - standard gauge
            "max_gradient_speed_reduction": 0.25,  # improved from 0.28 - excellent distributed traction
            "passenger_comfort_factor": 0.95,  # superior to DB610E - Tiltronix + premium suspension
            "train_length": 78,  # meters (3-car configuration)
            "signal_headway": 80,  # seconds - much faster than DB610E (110s)
            "station_approach_distance": 620,  # meters - 22% shorter than DB610E
            "cant_deficiency_allowed": 340,  # mm - 75% compensation achieves higher CD
            "tilt_angle": 8.0,  # degrees - active hydraulic Tiltronix
            "max_tilt_speed": 201.0,  # km/h - v9 tilt rate limiting
            "max_tilt_rate": 4.5,  # degrees/second
            "tilt_compensation": 0.75,  # 75% proven by research (not 68%)
            "tilt_response_time": 0.8,  # seconds - fast predictive system
            "door_cycle_time": 10,  # seconds - extra-wide pneumatic doors for airport
            "power_to_weight": 28.0,  # kW/tonne - lightweight 3-car beats 22.6 standard
            "doors_per_side": 3,  # per car - optimized for rapid boarding
            "door_width": 1400,  # mm - extra-wide for airport passengers with luggage
        },
        "imperial": {
            "max_speed": 125.0,  # mph
            "acceleration": 6.89,  # ft/s² - vs DB610E's 3.61 ft/s²
            "service_deceleration": 5.25,  # ft/s² - vs DB610E's 4.10 ft/s²
            "emergency_deceleration": 10.50,  # ft/s² - vs DB610E's 6.56 ft/s²
            "jerk_limit": 2.46,  # ft/s³ (improved)
            "min_curve_radius": 525,  # feet - vs DB610E's 656 feet
            "max_cant": 6.3,  # inches
            "gauge": 56.5,  # inches
            "max_gradient_speed_reduction": 0.25,  # improved
            "passenger_comfort_factor": 0.95,
            "train_length": 256,  # feet
            "signal_headway": 80,  # seconds
            "station_approach_distance": 2034,  # feet
            "cant_deficiency_allowed": 13.4,  # inches - improved from 12.6
            "tilt_angle": 8.0,  # degrees
            "max_tilt_speed": 125.0,  # mph
            "max_tilt_rate": 4.5,  # degrees/second
            "tilt_compensation": 0.75,  # corrected to research data
            "tilt_response_time": 0.8,  # seconds
            "door_cycle_time": 10,  # seconds - improved from 12
            "power_to_weight": 28.0,  # kW/tonne - improved from 24.5
            "doors_per_side": 3,  # per car
            "door_width": 55.1,  # inches
        },
        "station_config": {
            "default_dwell_time": 25,  # reduced from 30 - faster doors enable shorter dwells
            "terminus_dwell_time": 40,  # reduced from 45 - rapid turnaround
            "major_station_dwell_time": 25,  # reduced from 30
            "minor_station_dwell_time": 20,  # reduced from 25 - minimal stops
            "express_penalty": 0,  # lower penalty due to superior acceleration
        }
    },

    "CAF_CIVITY_TILT": {
        "name": "CAF Civity ISIT",
        "description": "6-car high-speed tilting EMU based on Civity platform with ISIT predictive tilting",
        "category": "regional_commuter_express",
        "metric": {
            "max_speed": 200.0,  # km/h (125 mph) - upgraded from standard 160 km/h
            "acceleration": 1.5,  # m/s² - proven Class 331 performance at this length
            "service_deceleration": 1.6,  # m/s² - enhanced regenerative braking
            "emergency_deceleration": 3.1,  # m/s² - modern composite disc brakes
            "jerk_limit": 0.72,  # m/s³ - optimized traction control
            "min_curve_radius": 180,  # meters - ISIT allows tighter curves
            "max_cant": 160,  # mm - North American standard
            "gauge": 1435,  # mm - standard gauge
            "max_gradient_speed_reduction": 0.30,  # excellent distributed traction
            "passenger_comfort_factor": 0.94,  # ISIT GPS-predictive tilt + modern suspension
            "train_length": 150,  # meters (6-car: ~25m per car)
            "signal_headway": 95,  # seconds - fast acceleration enables tight spacing
            "station_approach_distance": 680,  # meters - superior acceleration/deceleration
            "cant_deficiency_allowed": 340,  # mm - 8° ISIT tilting system
            "tilt_angle": 8.0,  # degrees - ISIT active hydraulic tilt
            "max_tilt_speed": 200.0,  # km/h - v9 tilt rate limiting
            "max_tilt_rate": 5.0,  # degrees/second
            "tilt_compensation": 0.75,  # 75% lateral force compensation
            "tilt_response_time": 0.6,  # seconds - GPS predictive faster than reactive
            "seating_capacity": 460,  # 6-car: ~75-80 seats per car
            "door_cycle_time": 11,  # seconds - wide pneumatic doors
            "doors_per_side": 12,  # 2 per car
            "door_width": 1400,  # mm - extra-wide for commuter flow
            "power_to_weight": 24.0,  # kW/tonne - high performance
            "total_power": 4800,  # kW distributed traction
        },
        "imperial": {
            "max_speed": 125.0,  # mph - HSR-compatible
            "acceleration": 4.92,  # ft/s² - excellent for commuter rail
            "service_deceleration": 5.25,  # ft/s²
            "emergency_deceleration": 10.17,  # ft/s²
            "jerk_limit": 2.36,  # ft/s³
            "min_curve_radius": 591,  # feet - tight curve capability
            "max_cant": 6.3,  # inches
            "gauge": 56.5,  # inches
            "max_gradient_speed_reduction": 0.30,
            "passenger_comfort_factor": 0.94,
            "train_length": 492,  # feet
            "signal_headway": 95,  # seconds
            "station_approach_distance": 2231,  # feet
            "cant_deficiency_allowed": 13.4,  # inches - high with ISIT
            "tilt_angle": 8.0,  # degrees
            "max_tilt_speed": 125.0,  # mph
            "max_tilt_rate": 5.0,  # degrees/second
            "tilt_compensation": 0.75,
            "tilt_response_time": 0.6,  # seconds
            "seating_capacity": 460,
            "door_cycle_time": 11,  # seconds
            "doors_per_side": 12,
            "door_width": 55.1,  # inches
            "power_to_weight": 24.0,  # kW/tonne
            "total_power": 6435,  # hp
        },
        "station_config": {
            "default_dwell_time": 300,  # standard commuter service
            "terminus_dwell_time": 50,  # longer turnaround for longer train
            "major_station_dwell_time": 300,  # major stops
            "minor_station_dwell_time": 25,  # express skip-stop patterns
            "express_penalty": 0,  # low penalty due to excellent acceleration
        }
    },
    "HIGHLINER": {
        "name": "St. Louis Car Highliner (Metra Electric)",
        "description": "Classic electric multiple unit from 1970s",
        "category": "regional",
        "metric": {
            "max_speed": 137.0,  # km/h (85 mph)
            "acceleration": 0.85,  # m/s² - 1970s EMU
            "service_deceleration": 1.0,  # m/s²
            "emergency_deceleration": 2.4,  # m/s²
            "jerk_limit": 0.4,  # m/s³
            "min_curve_radius": 320,  # meters
            "max_cant": 150,  # mm
            "gauge": 1435,  # mm
            "max_gradient_speed_reduction": 0.5,
            "passenger_comfort_factor": 0.75,  # older design
            "train_length": 26,  # meters (single car)
            "signal_headway": 130,  # seconds
            "station_approach_distance": 950,  # meters
            "cant_deficiency_allowed": 100,  # mm
        },
        "imperial": {
            "max_speed": 85.0,  # mph
            "acceleration": 2.78,  # ft/s²
            "service_deceleration": 3.28,  # ft/s²
            "emergency_deceleration": 7.87,  # ft/s²
            "jerk_limit": 1.31,  # ft/s³
            "min_curve_radius": 1050,  # feet
            "max_cant": 6,  # inches
            "gauge": 56.5,  # inches
            "max_gradient_speed_reduction": 0.5,
            "passenger_comfort_factor": 0.75,
            "train_length": 157,  # feet
            "signal_headway": 130,  # seconds
            "station_approach_distance": 3117,  # feet
            "cant_deficiency_allowed": 4,  # inches
        },
        "station_config": {
            "default_dwell_time": 300,
            "terminus_dwell_time": 600,
            "major_station_dwell_time": 300,
            "minor_station_dwell_time": 300,
            "express_penalty": 0,
        }
    },
    
    "CRRC_EMU": {
        "name": "CRRC Changchun EMU",
        "description": "Chinese electric multiple unit for commuter service",
        "category": "regional",
        "metric": {
            "max_speed": 160.0,  # km/h (100 mph)
            "acceleration": 1.0,  # m/s² - modern EMU
            "service_deceleration": 1.2,  # m/s²
            "emergency_deceleration": 2.8,  # m/s²
            "jerk_limit": 0.5,  # m/s³
            "min_curve_radius": 280,  # meters
            "max_cant": 160,  # mm
            "gauge": 1435,  # mm
            "max_gradient_speed_reduction": 0.45,
            "passenger_comfort_factor": 0.83,  # modern Chinese EMU
            "train_length": 100,  # meters (4-car unit)
            "signal_headway": 120,  # seconds
            "station_approach_distance": 900,  # meters
            "cant_deficiency_allowed": 110,  # mm
        },
        "imperial": {
            "max_speed": 100.0,  # mph
            "acceleration": 3.28,  # ft/s²
            "service_deceleration": 3.94,  # ft/s²
            "emergency_deceleration": 9.19,  # ft/s²
            "jerk_limit": 1.64,  # ft/s³
            "min_curve_radius": 919,  # feet
            "max_cant": 6.3,  # inches
            "gauge": 56.5,  # inches
            "max_gradient_speed_reduction": 0.45,
            "passenger_comfort_factor": 0.83,
            "train_length": 328,  # feet
            "signal_headway": 120,  # seconds
            "station_approach_distance": 2953,  # feet
            "cant_deficiency_allowed": 4.3,  # inches
        },
        "station_config": {
            "default_dwell_time": 300,
            "terminus_dwell_time": 600,
            "major_station_dwell_time": 300,
            "minor_station_dwell_time": 25,
            "express_penalty": 0,
        }
    },

        "ADTRANZ_CAF": {
        "name": "Adtranz-CAF EMU",
        "description": "Joint venture electric multiple unit for regional/metro service",
        "category": "regional",
        "metric": {
            "max_speed": 145.0,  # km/h (90 mph)
            "acceleration": 0.9,  # m/s² - modern EMU performance
            "service_deceleration": 1.1,  # m/s²
            "emergency_deceleration": 2.7,  # m/s²
            "jerk_limit": 0.48,  # m/s³
            "min_curve_radius": 290,  # meters
            "max_cant": 155,  # mm
            "gauge": 1435,  # mm
            "max_gradient_speed_reduction": 0.47,
            "passenger_comfort_factor": 0.81,  # late 1990s/2000s design
            "train_length": 90,  # meters (3-car unit)
            "signal_headway": 125,  # seconds
            "station_approach_distance": 920,  # meters
            "cant_deficiency_allowed": 105,  # mm
        },
        "imperial": {
            "max_speed": 90.0,  # mph
            "acceleration": 2.95,  # ft/s²
            "service_deceleration": 3.61,  # ft/s²
            "emergency_deceleration": 8.86,  # ft/s²
            "jerk_limit": 1.57,  # ft/s³
            "min_curve_radius": 951,  # feet
            "max_cant": 6.1,  # inches
            "gauge": 56.5,  # inches
            "max_gradient_speed_reduction": 0.47,
            "passenger_comfort_factor": 0.81,
            "train_length": 295,  # feet
            "signal_headway": 125,  # seconds
            "station_approach_distance": 3018,  # feet
            "cant_deficiency_allowed": 4.1,  # inches
        },
        "station_config": {
            "default_dwell_time": 300,
            "terminus_dwell_time": 600,
            "major_station_dwell_time": 300,
            "minor_station_dwell_time": 30,
            "express_penalty": 0,
        }
    },

    "DB610E": {
        "name": "DB Class 610E (Theoretical Electric)",
        "description": "Theoretical electrified version of German tilting DMU",
        "category": "regional",
        "metric": {
            "max_speed": 160.0,  # km/h (100 mph) - same as diesel version
            "acceleration": 1.1,  # m/s² - improved from diesel's ~0.6
            "service_deceleration": 1.25,  # m/s²
            "emergency_deceleration": 2.0,  # m/s²
            "jerk_limit": 0.55,  # m/s³
            "min_curve_radius": 200,  # meters - excellent with tilting
            "max_cant": 180,  # mm - German standards with tilt
            "gauge": 1435,  # mm
            "max_gradient_speed_reduction": 0.35,  # electric much better than diesel
            "passenger_comfort_factor": 0.90,  # excellent with tilt + no diesel vibration
            "train_length": 52,  # meters (2-car unit like original)
            "signal_headway": 110,  # seconds
            "station_approach_distance": 800,  # meters
            "cant_deficiency_allowed": 300,  # mm - 8° tilt capability adds significant cant deficiency
            "tilt_angle": 8,  # degrees - Fiat tilting technology
            "max_tilt_speed": 160.0,  # km/h - v9 tilt rate limiting
            "max_tilt_rate": 4.0,  # degrees/second
        },
        "imperial": {
            "max_speed": 100.0,  # mph
            "acceleration": 3.61,  # ft/s²
            "service_deceleration": 4.10,  # ft/s²
            "emergency_deceleration": 6.56,  # ft/s²
            "jerk_limit": 1.80,  # ft/s³
            "min_curve_radius": 656,  # feet - excellent with tilting
            "max_cant": 7,  # inches
            "gauge": 56.5,  # inches
            "max_gradient_speed_reduction": 0.35,
            "passenger_comfort_factor": 0.90,
            "train_length": 171,  # feet
            "signal_headway": 110,  # seconds
            "station_approach_distance": 2625,  # feet
            "cant_deficiency_allowed": 11.8,  # inches - high due to tilting
            "tilt_angle": 8,  # degrees
            "max_tilt_speed": 100.0,  # mph
            "max_tilt_rate": 4.0,  # degrees/second
        },
        "station_config": {
            "default_dwell_time": 300,
            "terminus_dwell_time": 600,
            "major_station_dwell_time": 300,
            "minor_station_dwell_time": 25,
            "express_penalty": 0,
        }
    },

        "BR_CLASS_442": {
        "name": "British Rail Class 442 'Wessex Electric'",
        "description": "British EMU for South West Main Line (1988-2007)",
        "category": "regional",
        "metric": {
            "max_speed": 161.0,  # km/h (100 mph)
            "acceleration": 0.85,  # m/s² - 1980s British EMU
            "service_deceleration": 1.1,  # m/s²
            "emergency_deceleration": 2.6,  # m/s²
            "jerk_limit": 0.45,  # m/s³
            "min_curve_radius": 280,  # meters - UK mainline standards
            "max_cant": 160,  # mm - UK standards
            "gauge": 1435,  # mm
            "max_gradient_speed_reduction": 0.47,
            "passenger_comfort_factor": 0.80,  # 1980s British EMU ride quality
            "train_length": 101,  # meters (5-car unit)
            "signal_headway": 125,  # seconds
            "station_approach_distance": 920,  # meters
            "cant_deficiency_allowed": 110,  # mm
        },
        "imperial": {
            "max_speed": 100.0,  # mph
            "acceleration": 2.79,  # ft/s²
            "service_deceleration": 3.61,  # ft/s²
            "emergency_deceleration": 8.53,  # ft/s²
            "jerk_limit": 1.48,  # ft/s³
            "min_curve_radius": 919,  # feet
            "max_cant": 6.3,  # inches
            "gauge": 56.5,  # inches
            "max_gradient_speed_reduction": 0.47,
            "passenger_comfort_factor": 0.80,
            "train_length": 331,  # feet
            "signal_headway": 125,  # seconds
            "station_approach_distance": 3018,  # feet
            "cant_deficiency_allowed": 4.3,  # inches
        },
        "station_config": {
            "default_dwell_time": 300,
            "terminus_dwell_time": 600,
            "major_station_dwell_time": 300,
            "minor_station_dwell_time": 30,
            "express_penalty": 0,
        }
    },

        "ETR460": {
        "name": "Alstom ETR 460 Pendolino (4-car)",
        "description": "Tilting EMU configured for regional service",
        "category": "regional",
        "metric": {
            "max_speed": 200.0,  # km/h (125 mph) - limited for regional service
            "acceleration": 0.9,  # m/s² - distributed traction
            "service_deceleration": 1.1,  # m/s²
            "emergency_deceleration": 2.6,  # m/s²
            "jerk_limit": 0.5,  # m/s³
            "min_curve_radius": 250,  # meters - tilting helps on tight curves
            "max_cant": 180,  # mm - with tilting
            "gauge": 1435,  # mm
            "max_gradient_speed_reduction": 0.42,  # good power-to-weight ratio
            "passenger_comfort_factor": 0.87,  # tilting provides comfort at speed
            "train_length": 105,  # meters (4-car unit)
            "signal_headway": 120,  # seconds
            "station_approach_distance": 950,  # meters
            "cant_deficiency_allowed": 275,  # mm - high due to 8° tilt capability
            "tilt_angle": 8,  # degrees - active tilting
            "max_tilt_speed": 200.0,  # km/h - v9 tilt rate limiting
            "max_tilt_rate": 4.0,  # degrees/second
        },
        "imperial": {
            "max_speed": 125.0,  # mph
            "acceleration": 2.95,  # ft/s²
            "service_deceleration": 3.61,  # ft/s²
            "emergency_deceleration": 8.53,  # ft/s²
            "jerk_limit": 1.64,  # ft/s³
            "min_curve_radius": 820,  # feet
            "max_cant": 7,  # inches
            "gauge": 56.5,  # inches
            "max_gradient_speed_reduction": 0.42,
            "passenger_comfort_factor": 0.87,
            "train_length": 344,  # feet
            "signal_headway": 120,  # seconds
            "station_approach_distance": 3117,  # feet
            "cant_deficiency_allowed": 10.8,  # inches - high with tilting
            "tilt_angle": 8,  # degrees
            "max_tilt_speed": 125.0,  # mph
            "max_tilt_rate": 4.0,  # degrees/second
        },
        "station_config": {
            "default_dwell_time": 300,
            "terminus_dwell_time": 600,
            "major_station_dwell_time": 300,
            "minor_station_dwell_time": 30,
            "express_penalty": 0,
        }
    }
}

# Get active configuration
ACTIVE_TRAIN = TRAIN_TYPES[TRAIN_TYPE]
TRAIN_CONFIG = ACTIVE_TRAIN["metric" if UNIT_SYSTEM == "METRIC" else "imperial"].copy()
STATION_CONFIG = ACTIVE_TRAIN["station_config"].copy()

# Operational parameters
train_category = ACTIVE_TRAIN.get("category", "regional")
if UNIT_SYSTEM == "METRIC":
    OPERATIONAL_CONFIG = {
        "signal_spacing": 3000 if train_category == "hsr" else (2000 if train_category == "regional" else 1000),
        "speed_restriction_buffer": 0.98 if train_category == "hsr" else 0.95,
        "weather_factor": 1.0,
        "congestion_factor": 1.0,
        "curve_smoothing_window": 5,  # points for curve smoothing
        "gradient_smoothing_window": 5,  # points for gradient smoothing
        "express_speed_bonus": 1.05,  # Express trains can go 5% faster between stops
    }
else:
    OPERATIONAL_CONFIG = {
        "signal_spacing": 9843 if train_category == "hsr" else (6562 if train_category == "regional" else 3280),
        "speed_restriction_buffer": 0.98 if train_category == "hsr" else 0.95,
        "weather_factor": 1.0,
        "congestion_factor": 1.0,
        "curve_smoothing_window": 5,
        "gradient_smoothing_window": 5,
        "express_speed_bonus": 1.05,
    }

# REGIONAL/METRO major stations (only used when SERVICE_TYPE != "APEX")
# For APEX mode, station classification comes from KMZ folder structure
REGIONAL_MAJOR_STATIONS = [
    # Ohio ORBIT / Cincinnati region
    'Hamilton Station', 'Springdale Station', 'Cincinnati Union Terminal', 'Norwood Station', 
    'Northside / Spring Grove', 'Mitchell Ave / WINTON PLACE', 'Dayton Station', 
    'Covington Union Station', 'Mason', 'CVG Airport', 'Lunken', 'Newport',
    'Erlanger', 'Riverfront Transit Center', 'Middletown Station', 'Miamisburg',
    # Add other regional major stations as needed
]

# ==================== UNIT CONVERSION ====================

class UnitSystem(Enum):
    METRIC = "metric"
    IMPERIAL = "imperial"

class UnitConverter:
    """Centralized unit conversion to prevent errors"""
    KM_TO_MILE = 0.621371
    METER_TO_FOOT = 3.28084
    MM_TO_INCH = 0.0393701
    KMH_TO_MPH = 0.621371
    MS2_TO_FTS2 = 3.28084
    MS_TO_KMH = 3.6
    
    def __init__(self, system: str):
        self.system = UnitSystem.METRIC if system.upper() == "METRIC" else UnitSystem.IMPERIAL
        self._conversion_cache = {}
    
    def speed_ms_to_display(self, speed_ms: float) -> float:
        """Convert m/s to display units (km/h or mph)"""
        kmh = speed_ms * self.MS_TO_KMH
        return kmh if self.system == UnitSystem.METRIC else kmh * self.KMH_TO_MPH
    
    def speed_display_to_ms(self, speed_display: float) -> float:
        """Convert display units to m/s"""
        if self.system == UnitSystem.IMPERIAL:
            speed_display = speed_display / self.KMH_TO_MPH
        return speed_display / self.MS_TO_KMH
    
    def speed(self, value_kmh: float, from_metric: bool = True) -> float:
        if self.system == UnitSystem.METRIC:
            return value_kmh if from_metric else value_kmh / self.KMH_TO_MPH
        else:
            return value_kmh * self.KMH_TO_MPH if from_metric else value_kmh
    
    def distance(self, value_m: float, from_metric: bool = True) -> float:
        if self.system == UnitSystem.METRIC:
            return value_m if from_metric else value_m / self.METER_TO_FOOT
        else:
            return value_m * self.METER_TO_FOOT if from_metric else value_m
    
    def long_distance(self, value_km: float, from_metric: bool = True) -> float:
        if self.system == UnitSystem.METRIC:
            return value_km if from_metric else value_km / self.KM_TO_MILE
        else:
            return value_km * self.KM_TO_MILE if from_metric else value_km
    
    def acceleration(self, value_ms2: float, from_metric: bool = True) -> float:
        if self.system == UnitSystem.METRIC:
            return value_ms2 if from_metric else value_ms2 / self.MS2_TO_FTS2
        else:
            return value_ms2 * self.MS2_TO_FTS2 if from_metric else value_ms2
    
    def small_distance(self, value_mm: float, from_metric: bool = True) -> float:
        if self.system == UnitSystem.METRIC:
            return value_mm if from_metric else value_mm / self.MM_TO_INCH
        else:
            return value_mm * self.MM_TO_INCH if from_metric else value_mm
    
    @property
    def speed_unit(self) -> str:
        return "km/h" if self.system == UnitSystem.METRIC else "mph"
    
    @property
    def distance_unit(self) -> str:
        return "m" if self.system == UnitSystem.METRIC else "ft"
    
    @property
    def long_distance_unit(self) -> str:
        return "km" if self.system == UnitSystem.METRIC else "mi"
    
    @property
    def acceleration_unit(self) -> str:
        return "m/sÂ²" if self.system == UnitSystem.METRIC else "ft/sÂ²"
    
    @property
    def small_distance_unit(self) -> str:
        return "mm" if self.system == UnitSystem.METRIC else "in"

units = UnitConverter(UNIT_SYSTEM)

# ==================== DATA CLASSES ====================

@dataclass
class TrainSpecs:
    """Train specifications with all values in metric (SI) units internally"""
    max_speed: float = 160.0  # km/h (stored internally)
    max_speed_ms: float = field(init=False)  # m/s (calculated)
    acceleration: float = 1.0  # m/sÂ²
    deceleration: float = 1.2  # m/sÂ²
    max_gradient_speed_reduction: float = 0.5
    min_curve_radius: float = 300  # meters
    max_cant: float = 150  # mm
    gauge: float = 1435  # mm
    jerk_limit: float = 0.5  # m/sÂ³
    emergency_brake_deceleration: float = 2.5  # m/sÂ²
    passenger_comfort_factor: float = 0.85
    train_length: float = 100  # meters
    signal_headway: float = 120  # seconds
    station_approach_distance: float = 1000  # meters
    train_type: str = "STADLER_KISS"
    cant_deficiency_allowed: float = 100  # mm - NEW FIELD
    tilt_angle: float = 0  # v9: degrees of active tilt
    max_tilt_speed: float = 0  # v9: km/h - tilt only active below this speed
    max_tilt_rate: float = 4.0  # v9: degrees/second - tilt mechanism speed
    
    def __post_init__(self):
        # Always store max speed in m/s internally for calculations
        self.max_speed_ms = self.max_speed / 3.6
    
    @classmethod
    def from_config(cls, config: Dict, unit_system: str, train_type: str = "STADLER_KISS"):
        """Create TrainSpecs from config, converting all to metric internally"""
        specs = {}
        converter = UnitConverter(unit_system)
        
        if unit_system.upper() == "IMPERIAL":
            # Convert from imperial to metric for internal storage
            specs['max_speed'] = config['max_speed'] / converter.KMH_TO_MPH
            specs['acceleration'] = config['acceleration'] / converter.MS2_TO_FTS2
            specs['deceleration'] = config.get('service_deceleration', 1.2) / converter.MS2_TO_FTS2
            specs['emergency_brake_deceleration'] = config.get('emergency_deceleration', 2.5) / converter.MS2_TO_FTS2
            specs['min_curve_radius'] = config['min_curve_radius'] / converter.METER_TO_FOOT
            specs['max_cant'] = config['max_cant'] / converter.MM_TO_INCH
            specs['gauge'] = config['gauge'] / converter.MM_TO_INCH
            specs['jerk_limit'] = config['jerk_limit'] / converter.MS2_TO_FTS2
            specs['train_length'] = config.get('train_length', 328) / converter.METER_TO_FOOT
            specs['signal_headway'] = config.get('signal_headway', 120)
            specs['station_approach_distance'] = config.get('station_approach_distance', 3280) / converter.METER_TO_FOOT
            specs['max_gradient_speed_reduction'] = config['max_gradient_speed_reduction']
            specs['passenger_comfort_factor'] = config.get('passenger_comfort_factor', 0.85)
            specs['cant_deficiency_allowed'] = config.get('cant_deficiency_allowed', 4) / converter.MM_TO_INCH
            # v9: tilt parameters
            specs['tilt_angle'] = config.get('tilt_angle', 0)
            specs['max_tilt_speed'] = config.get('max_tilt_speed', 0) / converter.KMH_TO_MPH  # convert mph to km/h
            specs['max_tilt_rate'] = config.get('max_tilt_rate', 4.0)  # degrees/second (same in both systems)
        else:
            # Already in metric - extract only the fields we need
            specs['max_speed'] = config.get('max_speed', 160.0)
            specs['acceleration'] = config.get('acceleration', 1.0)
            specs['deceleration'] = config.get('service_deceleration', 1.2)
            specs['emergency_brake_deceleration'] = config.get('emergency_deceleration', 2.5)
            specs['min_curve_radius'] = config.get('min_curve_radius', 300)
            specs['max_cant'] = config.get('max_cant', 150)
            specs['gauge'] = config.get('gauge', 1435)
            specs['jerk_limit'] = config.get('jerk_limit', 0.5)
            specs['max_gradient_speed_reduction'] = config.get('max_gradient_speed_reduction', 0.5)
            specs['passenger_comfort_factor'] = config.get('passenger_comfort_factor', 0.85)
            specs['train_length'] = config.get('train_length', 100)
            specs['signal_headway'] = config.get('signal_headway', 120)
            specs['station_approach_distance'] = config.get('station_approach_distance', 1000)
            specs['cant_deficiency_allowed'] = config.get('cant_deficiency_allowed', 100)
            # v9: tilt parameters
            specs['tilt_angle'] = config.get('tilt_angle', 0)
            specs['max_tilt_speed'] = config.get('max_tilt_speed', 0)
            specs['max_tilt_rate'] = config.get('max_tilt_rate', 4.0)
        
        specs['train_type'] = train_type
        return cls(**specs)

@dataclass
class Station:
    name: str
    lat: float
    lon: float
    elevation: float
    dwell_time: float = 45
    is_major: bool = False
    is_terminus: bool = False
    is_limited: bool = False  # v9: for limited-stop stations
    platform_length: float = 200
    track_position: float = 0  # Distance along track in meters
    folder_service_level: Optional[str] = None  # v9: from KMZ folder structure ('LIMITED', 'EXPRESS', 'LOCAL')
    is_express_stop: bool = field(init=False)
    
    def __post_init__(self):
        # Express stops are terminus, major, or limited stations by default
        self.is_express_stop = self.is_terminus or self.is_major or self.is_limited
    
    def get_dwell_time(self, is_express: bool = False, train_length: float = 100) -> float:
        """
        v9.0: Fixed dwell times based on station service classification and SERVICE_TYPE.
        
        SERVICE_TYPE determines dwell time scale:
        - METRO: 30s/60s/120s (quick urban transit)
        - REGIONAL: 60s/120s/300s (state/regional rail)
        - APEX: 300s/600s/900s (national HSR with APEx classifications)
        """
        return get_dwell_time_by_service(self.name, self.is_terminus, self)

@dataclass
class TrackPoint:
    lat: float
    lon: float
    elevation: float
    distance_from_start: float = 0
    speed: float = 0  # km/h for display
    speed_ms: float = 0  # m/s for calculations
    acceleration: float = 0  # m/sÂ²
    jerk: float = 0  # m/sÂ³
    time_from_start: float = 0
    at_station: Optional[Station] = None
    segment_index: int = 0
    gradient: float = 0  # percent
    curve_radius: float = float('inf')  # meters
    speed_limit: float = 0  # km/h

@dataclass
class StationSegmentTime:
    from_station: Station
    to_station: Station
    segment_distance: float
    running_time: float
    max_speed_achieved: float  # km/h
    avg_speed: float  # km/h
    start_time: float
    arrival_time: float
    energy_consumed: float = 0  # kWh estimate
    stations_skipped: int = 0  # For express services

@dataclass
class ServiceComparison:
    """Results from comparing different service patterns"""
    service_name: str
    stations_served: int
    stations_skipped: int
    journey_time_min: float
    running_time_min: float
    commercial_speed_kmh: float
    time_savings_min: float = 0  # Compared to all-stops
    speed_improvement_percent: float = 0

# ==================== HELPER FUNCTIONS ====================

def haversine_distance(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """Calculate distance between two points on Earth's surface"""
    R = 6371000  # Earth radius in meters
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return R * c

def smooth_array(arr: np.ndarray, window_size: int = 5) -> np.ndarray:
    """Apply smoothing to an array using a moving average"""
    if len(arr) <= window_size:
        return arr
    
    # Use convolution for efficient moving average
    kernel = np.ones(window_size) / window_size
    # Pad the array to handle edges properly
    padded = np.pad(arr, (window_size//2, window_size//2), mode='edge')
    smoothed = np.convolve(padded, kernel, mode='valid')
    
    return smoothed

def interpolate_track_points(track_points: List[Tuple[float, float, float]], 
                           spacing_m: float = 10.0) -> List[Tuple[float, float, float]]:
    """Interpolate track points with consistent spacing"""
    if len(track_points) < 2:
        return track_points

    interpolated_points = []
    total_original_distance = 0
    
    for i in range(len(track_points) - 1):
        start = track_points[i]
        end = track_points[i + 1]

        horizontal_dist = haversine_distance(start[0], start[1], end[0], end[1])
        elev_diff = end[2] - start[2]
        total_dist_3d = math.sqrt(horizontal_dist ** 2 + elev_diff ** 2)
        total_original_distance += total_dist_3d

        if total_dist_3d < spacing_m * 0.5:  # Skip very short segments
            interpolated_points.append(start)
        else:
            num_segments = max(1, round(total_dist_3d / spacing_m))
            interpolated_points.append(start)
            
            for j in range(1, num_segments):
                fraction = j / num_segments
                lat = start[0] + fraction * (end[0] - start[0])
                lon = start[1] + fraction * (end[1] - start[1])
                elev = start[2] + fraction * elev_diff
                interpolated_points.append((lat, lon, elev))

    interpolated_points.append(track_points[-1])
    
    # Remove duplicates while preserving order
    seen = set()
    unique_points = []
    for point in interpolated_points:
        point_tuple = (round(point[0], 6), round(point[1], 6), round(point[2], 2))
        if point_tuple not in seen:
            seen.add(point_tuple)
            unique_points.append(point)
    
    if len(unique_points) > 1:
        avg_spacing = total_original_distance / (len(unique_points) - 1)
        print(f"    Interpolation: {len(track_points)} â†’ {len(unique_points)} points")
        print(f"    Average spacing: {avg_spacing:.1f}m")
    
    return unique_points


def precalculate_curve_radii(track_points: List[Tuple[float, float, float]], 
                             min_radius: float = 100.0) -> List[Tuple[float, float]]:
    """
    Pre-calculate curve radii from ORIGINAL track points before interpolation.
    Returns list of (cumulative_distance, curve_radius) pairs.
    
    SPACING-AWARE: Uses heading change method for widely-spaced points, which can
    detect tighter curves than the 3-point circumcircle method when points are sparse.
    
    Uses multiple measurement spans to detect:
    - Simple curves (constant radius)
    - Transition spirals (tightening/expanding radius)
    - Compound curves (multiple radii joined)
    
    This preserves the true track geometry since original KMZ points represent 
    actual surveyed/planned positions, not interpolated ones.
    """
    if len(track_points) < 3:
        return [(0.0, float('inf'))]
    
    # Calculate cumulative distances
    distances = [0.0]
    for i in range(1, len(track_points)):
        dist = haversine_distance(
            track_points[i-1][0], track_points[i-1][1],
            track_points[i][0], track_points[i][1]
        )
        distances.append(distances[-1] + dist)
    
    def calc_radius_3pt(p1, p2, p3):
        """Calculate radius using circumscribed circle of 3 points"""
        lat_avg = (p1[0] + p2[0] + p3[0]) / 3
        m_per_deg_lat = 111319.9
        m_per_deg_lon = 111319.9 * math.cos(math.radians(lat_avg))
        
        x1 = (p1[1] - p2[1]) * m_per_deg_lon
        y1 = (p1[0] - p2[0]) * m_per_deg_lat
        x2 = 0.0
        y2 = 0.0
        x3 = (p3[1] - p2[1]) * m_per_deg_lon
        y3 = (p3[0] - p2[0]) * m_per_deg_lat
        
        a = math.sqrt((x2 - x1)**2 + (y2 - y1)**2)
        b = math.sqrt((x3 - x2)**2 + (y3 - y2)**2)
        c = math.sqrt((x1 - x3)**2 + (y1 - y3)**2)
        
        s = (a + b + c) / 2
        area_sq = s * (s - a) * (s - b) * (s - c)
        
        if area_sq > 0 and a * b * c > 0:
            area = math.sqrt(area_sq)
            return (a * b * c) / (4 * area)
        return float('inf')
    
    def calc_heading(p1, p2):
        """Calculate heading between two points in degrees"""
        lat1, lon1 = math.radians(p1[0]), math.radians(p1[1])
        lat2, lon2 = math.radians(p2[0]), math.radians(p2[1])
        x = math.cos(lat2) * math.sin(lon2 - lon1)
        y = math.cos(lat1) * math.sin(lat2) - math.sin(lat1) * math.cos(lat2) * math.cos(lon2 - lon1)
        return (math.degrees(math.atan2(x, y)) + 360) % 360
    
    def radius_from_heading_change(h1, h2, spacing_before, spacing_after):
        """
        Calculate radius from heading change - works well even for widely-spaced points.
        
        IMPORTANT: Chord headings represent tangent directions at segment midpoints,
        so the effective arc length is the distance between midpoints (half the total).
        R = effective_arc / delta_heading (in radians)
        """
        delta = h2 - h1
        while delta > 180: delta -= 360
        while delta < -180: delta += 360
        delta_rad = math.radians(abs(delta))
        if delta_rad < 0.001:  # Nearly straight
            return float('inf')
        # Effective arc = distance from midpoint of seg1 to midpoint of seg2
        effective_arc = (spacing_before + spacing_after) / 2
        return effective_arc / delta_rad
    
    # Analyze point spacing
    spacings = [distances[i] - distances[i-1] for i in range(1, len(distances))]
    avg_spacing = sum(spacings) / len(spacings) if spacings else 0
    max_spacing = max(spacings) if spacings else 0
    min_spacing = min(spacings) if spacings else 0
    
    # Warn if spacing is large (curves tighter than spacing may be missed)
    if max_spacing > 200:
        print(f"    [CurvePrecalc] WARNING: Large point spacing detected")
        print(f"    [CurvePrecalc]   Average: {avg_spacing:.0f}m, Max: {max_spacing:.0f}m, Min: {min_spacing:.0f}m")
        print(f"    [CurvePrecalc]   Curves tighter than ~{max_spacing/2:.0f}m may not be fully detected")
        print(f"    [CurvePrecalc]   Using heading-change method for better detection")
    
    # For large radius curves (>5km), we need wider spans for reliable detection
    # Sagitta (curve deviation) = span² / (8*R), so for R=10km, span=1000m gives sagitta=12.5m
    # This is the minimum span needed to detect 10km curves reliably with typical GPS noise
    total_length = distances[-1] if distances else 0
    
    curve_map = []
    
    for i in range(len(track_points)):
        if i == 0 or i == len(track_points) - 1:
            curve_map.append((distances[i], float('inf')))
            continue
        
        # Get local point spacing
        spacing_before = distances[i] - distances[i-1]
        spacing_after = distances[i+1] - distances[i] if i < len(track_points)-1 else spacing_before
        local_spacing = (spacing_before + spacing_after) / 2
        
        # Separate lists for narrow and wide span detections
        narrow_span_radii = []  # From spans < 100m - good for tight curves
        wide_span_radii = []    # From spans >= 500m - good for gentle curves
        
        # METHOD 1: Narrow span - 3-point circumcircle with adjacent points
        r1 = calc_radius_3pt(track_points[i-1], track_points[i], track_points[i+1])
        if r1 != float('inf') and r1 > 0:
            narrow_span_radii.append(r1)
        
        # METHOD 2: Narrow span - Heading change with adjacent points
        h_before = calc_heading(track_points[i-1], track_points[i])
        h_after = calc_heading(track_points[i], track_points[i+1])
        r_heading = radius_from_heading_change(h_before, h_after, spacing_before, spacing_after)
        if r_heading != float('inf') and r_heading > 0:
            narrow_span_radii.append(r_heading)
        
        # METHOD 3: Medium span - Using i-2, i, i+2
        if i >= 2 and i < len(track_points) - 2:
            r2 = calc_radius_3pt(track_points[i-2], track_points[i], track_points[i+2])
            if r2 != float('inf') and r2 > 0:
                narrow_span_radii.append(r2)
            
            h_back = calc_heading(track_points[i-2], track_points[i])
            h_fwd = calc_heading(track_points[i], track_points[i+2])
            spacing_back = distances[i] - distances[i-2]
            spacing_fwd = distances[i+2] - distances[i]
            r_heading_long = radius_from_heading_change(h_back, h_fwd, spacing_back, spacing_fwd)
            if r_heading_long != float('inf') and r_heading_long > 0:
                narrow_span_radii.append(r_heading_long)
        
        # METHOD 4: WIDE SPANS for large radius curves
        # These are reliable for detecting 5km+ curves
        for target_span in [500, 1000, 2000]:
            back_idx = None
            fwd_idx = None
            half_span = target_span / 2
            
            for j in range(i-1, -1, -1):
                if distances[i] - distances[j] >= half_span:
                    back_idx = j
                    break
            
            for j in range(i+1, len(track_points)):
                if distances[j] - distances[i] >= half_span:
                    fwd_idx = j
                    break
            
            if back_idx is not None and fwd_idx is not None:
                r_wide = calc_radius_3pt(track_points[back_idx], track_points[i], track_points[fwd_idx])
                if r_wide != float('inf') and r_wide > 0:
                    wide_span_radii.append(r_wide)
                
                h_back_w = calc_heading(track_points[back_idx], track_points[i])
                h_fwd_w = calc_heading(track_points[i], track_points[fwd_idx])
                span_back = distances[i] - distances[back_idx]
                span_fwd = distances[fwd_idx] - distances[i]
                r_heading_wide = radius_from_heading_change(h_back_w, h_fwd_w, span_back, span_fwd)
                if r_heading_wide != float('inf') and r_heading_wide > 0:
                    wide_span_radii.append(r_heading_wide)
        
        # Combine narrow and wide results for the selection logic
        radii_detected = narrow_span_radii + wide_span_radii
        
        if radii_detected:
            # Radius selection strategy:
            #
            # PHYSICS: For narrow spans (20-40m), the sagitta (curve deviation) is:
            #   sagitta = span²/(8R)
            #   - 2km curve with 40m span: sagitta = 1600/16000 = 10cm (marginal vs noise)
            #   - 5km curve with 40m span: sagitta = 1600/40000 = 4cm (below noise)
            #   - 10km curve with 40m span: sagitta = 1600/80000 = 2cm (way below noise)
            #
            # CONCLUSION: Narrow spans can only reliably detect curves < ~2km radius.
            # For gentle curves, ALWAYS trust wide span results.
            
            wide_median = None
            if wide_span_radii:
                wide_span_radii_sorted = sorted(wide_span_radii)
                wide_median = wide_span_radii_sorted[len(wide_span_radii_sorted)//2]
            
            narrow_min = min(narrow_span_radii) if narrow_span_radii else float('inf')
            
            if wide_median is not None:
                # Wide spans are available
                if wide_median > 2000:
                    # Wide spans see a gentle curve (>2km)
                    # TRUST THE WIDE SPANS - narrow spans are just noise for gentle curves
                    radius = wide_median
                else:
                    # Wide spans see a tight curve (<2km) - use tightest detection
                    radius = min(narrow_min, wide_median)
            else:
                # No wide span data (short track segment)
                # Use narrow spans but cap at 2km (can't reliably detect gentler)
                if narrow_min < 2000:
                    radius = narrow_min
                else:
                    # Narrow detecting > 2km is unreliable - treat as gentle curve
                    radius = float('inf')
            
            radius = max(radius, min_radius)
        else:
            radius = float('inf')
        
        curve_map.append((distances[i], radius))
    
    # Detect and report curve statistics
    finite_radii = [r for d, r in curve_map if r != float('inf') and r < 1e8]
    if finite_radii:
        min_r = min(finite_radii)
        max_r = max(finite_radii)
        if max_r / min_r > 2.0:
            print(f"    [CurvePrecalc] Detected variable curvature: {min_r:.0f}m to {max_r:.0f}m")
        else:
            print(f"    [CurvePrecalc] Tightest curve: {min_r:.0f}m")
    
    return curve_map




def analyze_curves_heading_based(track_points: List[Tuple[float, float, float]], 
                                  min_radius: float = 100.0) -> List[Tuple[float, float]]:
    """
    Heading-based curve detection with circle fitting.
    
    Process:
    1. Calculate heading at each point (direction to next point)
    2. Calculate heading change rate (curvature indicator)
    3. Smooth heavily to filter noise
    4. Identify curve sections by sustained heading change
    5. Fit circles to each curve section
    6. Return per-point radius map
    
    Returns: List of (cumulative_distance, curve_radius) pairs
    """
    n = len(track_points)
    if n < 3:
        return [(0.0, float('inf'))]
    
    # Local coordinate conversion
    ref_lat = track_points[0][0]
    ref_lon = track_points[0][1]
    m_per_deg_lat = 111319.9
    m_per_deg_lon = 111319.9 * math.cos(math.radians(ref_lat))
    
    def to_xy(p):
        return ((p[1] - ref_lon) * m_per_deg_lon, (p[0] - ref_lat) * m_per_deg_lat)
    
    def calc_heading_local(p1, p2):
        lat1, lon1 = math.radians(p1[0]), math.radians(p1[1])
        lat2, lon2 = math.radians(p2[0]), math.radians(p2[1])
        x = math.cos(lat2) * math.sin(lon2 - lon1)
        y = math.cos(lat1)*math.sin(lat2) - math.sin(lat1)*math.cos(lat2)*math.cos(lon2-lon1)
        return math.degrees(math.atan2(x, y))
    
    def normalize_angle(angle):
        while angle > 180: angle -= 360
        while angle < -180: angle += 360
        return angle
    
    def fit_circle_local(points_xy):
        if len(points_xy) < 3:
            return float('inf')
        x = [p[0] for p in points_xy]
        y = [p[1] for p in points_xy]
        
        # Kåsa method for circle fitting
        try:
            import numpy as np
            x = np.array(x)
            y = np.array(y)
            A = np.column_stack([x, y, np.ones(len(x))])
            b = x**2 + y**2
            c, _, _, _ = np.linalg.lstsq(A, b, rcond=None)
            cx, cy = c[0] / 2, c[1] / 2
            r = np.sqrt(c[2] + cx**2 + cy**2)
            return r if r > 0 else float('inf')
        except:
            return float('inf')
    
    # Step 1: Calculate cumulative distances
    distances = [0.0]
    for i in range(1, n):
        d = haversine_distance(track_points[i-1][0], track_points[i-1][1],
                               track_points[i][0], track_points[i][1])
        distances.append(distances[-1] + d)
    
    total_length = distances[-1]
    avg_spacing = total_length / (n - 1) if n > 1 else 10
    
    # Step 2: Calculate headings
    headings = []
    for i in range(n - 1):
        h = calc_heading_local(track_points[i], track_points[i+1])
        headings.append(h)
    headings.append(headings[-1] if headings else 0)
    
    # Step 3: Calculate heading change rate using sliding window
    # Use window proportional to track length but at least 100m
    window_m = max(100, min(500, total_length / 20))
    window_pts = max(5, int(window_m / avg_spacing))
    
    curvature = []  # deg/m (positive = right turn)
    
    for i in range(n):
        # Get points in window
        half_window = window_pts // 2
        i_start = max(0, i - half_window)
        i_end = min(n - 1, i + half_window)
        
        if i_end > i_start:
            # Total heading change
            total_heading_change = 0
            for j in range(i_start, i_end):
                if j < len(headings) - 1:
                    delta = normalize_angle(headings[j+1] - headings[j])
                    total_heading_change += delta
            
            # Distance covered
            dist_covered = distances[i_end] - distances[i_start]
            
            if dist_covered > 0:
                rate = total_heading_change / dist_covered
                curvature.append(rate)
            else:
                curvature.append(0)
        else:
            curvature.append(0)
    
    # Step 4: Identify curve sections
    # Threshold: anything with radius > 50km is "straight"
    STRAIGHT_THRESHOLD = 57.2958 / 50000  # ~0.00115 deg/m
    
    sections = []  # List of {'start': idx, 'end': idx, 'type': str, 'avg_curv': float}
    current_section = {'start': 0, 'avg_curv': curvature[0]}
    
    for i in range(1, n):
        # Check if curvature changed significantly
        if abs(curvature[i]) < STRAIGHT_THRESHOLD:
            current_type = 'straight'
        else:
            current_type = 'right' if curvature[i] > 0 else 'left'
        
        if abs(curvature[current_section['start']]) < STRAIGHT_THRESHOLD:
            section_type = 'straight'
        else:
            section_type = 'right' if curvature[current_section['start']] > 0 else 'left'
        
        # Different type = new section
        if current_type != section_type:
            current_section['end'] = i - 1
            current_section['type'] = section_type
            sections.append(current_section)
            current_section = {'start': i, 'avg_curv': curvature[i]}
    
    # Close final section
    current_section['end'] = n - 1
    if abs(curvature[current_section['start']]) < STRAIGHT_THRESHOLD:
        current_section['type'] = 'straight'
    else:
        current_section['type'] = 'right' if curvature[current_section['start']] > 0 else 'left'
    sections.append(current_section)
    
    # Merge short sections (<100m) with neighbors
    MIN_SECTION_M = 100
    merged = []
    for sec in sections:
        sec_length = distances[sec['end']] - distances[sec['start']]
        if sec_length < MIN_SECTION_M and merged:
            merged[-1]['end'] = sec['end']
        else:
            merged.append(sec)
    sections = merged
    
    # Step 5: Fit circles to curve sections
    for sec in sections:
        if sec['type'] == 'straight':
            sec['radius'] = float('inf')
        else:
            sec_points = track_points[sec['start']:sec['end']+1]
            if len(sec_points) >= 5:
                points_xy = [to_xy(p) for p in sec_points]
                r = fit_circle_local(points_xy)
                sec['radius'] = r if min_radius < r < 1e8 else float('inf')
            else:
                # Not enough points for circle fit - use curvature estimate
                avg_curv = sum(curvature[sec['start']:sec['end']+1]) / (sec['end'] - sec['start'] + 1)
                if abs(avg_curv) > 0.0001:
                    sec['radius'] = abs(57.2958 / avg_curv)
                else:
                    sec['radius'] = float('inf')
    
    # Step 6: Build curve map
    curve_map = []
    for i in range(n):
        r = float('inf')
        for sec in sections:
            if sec['start'] <= i <= sec['end']:
                r = sec['radius']
                break
        r = max(r, min_radius)
        curve_map.append((distances[i], r))
    
    # Report
    curves_found = [s for s in sections if s['type'] != 'straight']
    if curves_found:
        radii_str = ", ".join(f"{s['radius']:.0f}m" for s in curves_found if s['radius'] < 1e8)
        print(f"    [HeadingCurve] Found {len(curves_found)} curves: {radii_str if radii_str else 'gentle curves'}")
    else:
        print(f"    [HeadingCurve] Straight track (no significant curves)")
    
    return curve_map


def interpolate_track_with_curves(track_points: List[Tuple[float, float, float]], 
                                  curve_map: List[Tuple[float, float]],
                                  spacing_m: float = 10.0) -> Tuple[List[Tuple[float, float, float]], List[float]]:
    """
    Interpolate track points AND map pre-calculated curve radii to interpolated points.
    Returns (interpolated_points, curve_radii_for_each_point)
    
    Handles transition spirals (clothoids) by interpolating CURVATURE (1/R) linearly,
    which is how real railroad spirals work - curvature changes linearly with distance.
    """
    if len(track_points) < 2:
        return track_points, [float('inf')]
    
    interpolated_points = []
    interpolated_distances = []
    cumulative_dist = 0.0
    
    for i in range(len(track_points) - 1):
        start = track_points[i]
        end = track_points[i + 1]
        
        horizontal_dist = haversine_distance(start[0], start[1], end[0], end[1])
        elev_diff = end[2] - start[2]
        total_dist_3d = math.sqrt(horizontal_dist ** 2 + elev_diff ** 2)
        
        if total_dist_3d < spacing_m * 0.5:
            interpolated_points.append(start)
            interpolated_distances.append(cumulative_dist)
        else:
            num_segments = max(1, round(total_dist_3d / spacing_m))
            interpolated_points.append(start)
            interpolated_distances.append(cumulative_dist)
            
            for j in range(1, num_segments):
                fraction = j / num_segments
                lat = start[0] + fraction * (end[0] - start[0])
                lon = start[1] + fraction * (end[1] - start[1])
                elev = start[2] + fraction * elev_diff
                interpolated_points.append((lat, lon, elev))
                interpolated_distances.append(cumulative_dist + fraction * horizontal_dist)
        
        cumulative_dist += horizontal_dist
    
    interpolated_points.append(track_points[-1])
    interpolated_distances.append(cumulative_dist)
    
    # Map curve radii to interpolated points by distance
    # Use curvature interpolation for proper spiral/transition curve handling
    curve_radii = []
    curve_idx = 0
    
    def radius_to_curvature(r):
        """Convert radius to curvature (handles infinity)"""
        if r == float('inf') or r > 1e8:
            return 0.0  # Straight = zero curvature
        return 1.0 / r
    
    def curvature_to_radius(k):
        """Convert curvature back to radius"""
        if abs(k) < 1e-9:
            return float('inf')  # Zero curvature = straight
        return 1.0 / k
    
    for dist in interpolated_distances:
        # Find the two curve_map entries that bracket this distance
        while curve_idx < len(curve_map) - 1 and curve_map[curve_idx + 1][0] < dist:
            curve_idx += 1
        
        if curve_idx >= len(curve_map) - 1:
            curve_radii.append(curve_map[-1][1])
        else:
            d1, r1 = curve_map[curve_idx]
            d2, r2 = curve_map[curve_idx + 1]
            
            if d2 - d1 > 0:
                t = (dist - d1) / (d2 - d1)
                
                # Convert to curvature, interpolate linearly, convert back
                # This properly models transition spirals (clothoids)
                k1 = radius_to_curvature(r1)
                k2 = radius_to_curvature(r2)
                
                # Linear interpolation of curvature
                k_interp = k1 + t * (k2 - k1)
                
                # Convert back to radius
                radius = curvature_to_radius(k_interp)
            else:
                radius = r1
            
            curve_radii.append(radius)
    
    # Remove duplicates while preserving order
    seen = set()
    unique_points = []
    unique_radii = []
    for point, radius in zip(interpolated_points, curve_radii):
        point_tuple = (round(point[0], 6), round(point[1], 6), round(point[2], 2))
        if point_tuple not in seen:
            seen.add(point_tuple)
            unique_points.append(point)
            unique_radii.append(radius)
    
    return unique_points, unique_radii


def filter_stations_for_service(stations: List[Station], service_pattern: str, 
                               custom_stops: List[str] = None) -> List[Station]:
    """Filter stations based on service pattern"""
    if service_pattern == "ALL":
        return stations
    elif service_pattern == "EXPRESS":
        return [s for s in stations if s.is_express_stop]
    elif service_pattern == "CUSTOM" and custom_stops:
        return [s for s in stations if s.name in custom_stops]
    else:
        return stations

# ==================== IMPROVED PHYSICS CALCULATOR ====================

class ImprovedPhysicsCalculator:
    """Enhanced physics calculator with proper jerk limiting and accurate timing"""
    
    def __init__(self, train_specs: TrainSpecs):
        self.train = train_specs
        self.operational_config = OPERATIONAL_CONFIG
        self.units = units
    
    def calculate_curve_radius_robust(self, p1, p2, p3, threshold: float = 1e-4) -> float:
        """Calculate curve radius with improved numerical stability"""
        def lat_lon_to_xy(lat, lon, ref_lat, ref_lon):
            R = 6371000
            x = R * math.radians(lon - ref_lon) * math.cos(math.radians(ref_lat))
            y = R * math.radians(lat - ref_lat)
            return x, y
        
        ref_lat, ref_lon = p2[0], p2[1]
        
        x1, y1 = lat_lon_to_xy(p1[0], p1[1], ref_lat, ref_lon)
        x2, y2 = lat_lon_to_xy(p2[0], p2[1], ref_lat, ref_lon)
        x3, y3 = lat_lon_to_xy(p3[0], p3[1], ref_lat, ref_lon)
        
        # Check for collinearity
        area = abs((x2 - x1) * (y3 - y1) - (x3 - x1) * (y2 - y1))
        if area < threshold:
            return float('inf')
        
        # Use more stable formula
        a = math.sqrt((x2 - x3)**2 + (y2 - y3)**2)
        b = math.sqrt((x1 - x3)**2 + (y1 - y3)**2)
        c = math.sqrt((x1 - x2)**2 + (y1 - y2)**2)
        
        # Check for degenerate triangle
        if a < 1e-6 or b < 1e-6 or c < 1e-6:
            return float('inf')
        
        # Heron's formula with numerical stability check
        s = (a + b + c) / 2
        area_sq = s * (s - a) * (s - b) * (s - c)
        
        if area_sq < 1e-10:
            return float('inf')
        
        area = math.sqrt(area_sq)
        
        if area < 1e-6:
            return float('inf')
        
        radius = (a * b * c) / (4 * area)
        
        # Apply minimum radius constraint
        return max(radius, self.train.min_curve_radius)
    
    def _calculate_heading(self, p1, p2):
        """Calculate heading between two points"""
        lat1, lon1 = math.radians(p1[0]), math.radians(p1[1])
        lat2, lon2 = math.radians(p2[0]), math.radians(p2[1])
        x = math.cos(lat2) * math.sin(lon2 - lon1)
        y = math.cos(lat1) * math.sin(lat2) - math.sin(lat1) * math.cos(lat2) * math.cos(lon2 - lon1)
        return (math.degrees(math.atan2(x, y)) + 360) % 360
    
    def _radius_from_heading_change(self, h1, h2, arc_length):
        """Calculate radius from heading change"""
        delta = h2 - h1
        while delta > 180: delta -= 360
        while delta < -180: delta += 360
        delta_rad = math.radians(abs(delta))
        return arc_length / delta_rad if delta_rad >= 0.001 else float('inf')
    
    def calculate_curve_radius_multi_baseline(self, track_points, idx, distances) -> float:
        """
        v9.0: Multi-baseline curve detection
        Uses multiple measurement baselines to detect curves of varying radii
        """
        n = len(track_points)
        center_d = distances[idx]
        
        # Choose baselines based on train speed capability
        max_speed_kmh = self.train.max_speed
        if max_speed_kmh > 300:
            baselines = [100, 200, 500, 1000, 2000]
        elif max_speed_kmh > 200:
            baselines = [100, 200, 500, 1000, 1500]
        elif max_speed_kmh > 160:
            baselines = [50, 100, 200, 500, 1000]
        else:
            baselines = [50, 100, 200, 500]
        
        detected_radii = []
        
        for baseline in baselines:
            half = baseline / 2
            
            # Find points at baseline/2 before and after
            back_idx = forward_idx = None
            for i in range(idx - 1, -1, -1):
                if center_d - distances[i] >= half:
                    back_idx = i
                    break
            for i in range(idx + 1, n):
                if distances[i] - center_d >= half:
                    forward_idx = i
                    break
            
            if back_idx is None or forward_idx is None:
                continue
            
            # Method 1: Three-point radius
            r_3pt = self.calculate_curve_radius_robust(
                track_points[back_idx], track_points[idx], track_points[forward_idx]
            )
            
            # Method 2: Heading change radius
            h_back = self._calculate_heading(track_points[back_idx], track_points[idx])
            h_fwd = self._calculate_heading(track_points[idx], track_points[forward_idx])
            arc_length = distances[forward_idx] - distances[back_idx]
            r_heading = self._radius_from_heading_change(h_back, h_fwd, arc_length)
            
            # Combine methods
            if r_3pt != float('inf') and r_heading != float('inf') and r_3pt > 0 and r_heading > 0:
                # If both agree reasonably, use geometric mean
                if 0.5 < r_3pt / r_heading < 2.0:
                    detected_radii.append(math.sqrt(r_3pt * r_heading))
                else:
                    detected_radii.append(max(r_3pt, r_heading))
            elif r_3pt != float('inf') and r_3pt > 0:
                detected_radii.append(r_3pt)
            elif r_heading != float('inf') and r_heading > 0:
                detected_radii.append(r_heading)
        
        if not detected_radii:
            return float('inf')
        
        # Use median of detected radii (robust to outliers)
        median_r = np.median(detected_radii)
        min_r = min(detected_radii)
        
        # If minimum is much smaller than median, there's likely a real curve
        if min_r < median_r * 0.3:
            return max(min_r, self.train.min_curve_radius)
        
        return max(median_r, self.train.min_curve_radius)
    
    def calculate_curve_speed_limit(self, radius: float, check_speed: float = None) -> float:
        """
        Calculate speed limit for a curve (returns km/h)
        
        Args:
            radius: curve radius in meters
            check_speed: if provided, only apply tilt if this speed is below tilt_engagement_speed
        """
        if radius == float('inf'):
            return self.train.max_speed
        
        g = 9.81  # m/s²
        cant_m = self.train.max_cant / 1000
        gauge_m = self.train.gauge / 1000
        
        # Use train-specific cant deficiency if available, otherwise default
        cant_deficiency_mm = self.train.cant_deficiency_allowed if hasattr(self.train, 'cant_deficiency_allowed') else 100
        cant_deficiency_m = cant_deficiency_mm / 1000
        
        # Check for active tilt capability
        tilt_angle = getattr(self.train, 'tilt_angle', 0)
        max_tilt_speed = getattr(self.train, 'max_tilt_speed', 0)
        
        # Only apply tilt benefit if:
        # 1. Train has tilt capability
        # 2. Max tilt speed is defined
        # 3. Resulting speed would be below tilt engagement threshold
        can_use_tilt = tilt_angle > 0 and max_tilt_speed > 0
        
        if can_use_tilt:
            # First calculate speed WITHOUT tilt
            v_no_tilt_ms = math.sqrt(g * radius * (cant_m + cant_deficiency_m) / gauge_m)
            v_no_tilt_kmh = v_no_tilt_ms * 3.6
            
            # If no-tilt speed already exceeds max_tilt_speed, tilt won't help
            if v_no_tilt_kmh >= max_tilt_speed:
                # At high speed, tilt is disabled - use base cant deficiency only
                v_kmh = v_no_tilt_kmh * self.operational_config["speed_restriction_buffer"]
                return min(v_kmh, self.train.max_speed)
            
            # Calculate speed WITH tilt
            tilt_compensation_m = (tilt_angle * 30) / 1000
            effective_cant_deficiency_m = cant_deficiency_m + tilt_compensation_m
            v_with_tilt_ms = math.sqrt(g * radius * (cant_m + effective_cant_deficiency_m) / gauge_m)
            v_with_tilt_kmh = v_with_tilt_ms * 3.6
            
            # Cap tilt-assisted speed at tilt engagement speed
            v_with_tilt_kmh = min(v_with_tilt_kmh, max_tilt_speed)
            
            # Return the better of no-tilt (uncapped) or tilt (capped)
            v_kmh = max(v_no_tilt_kmh, v_with_tilt_kmh) * self.operational_config["speed_restriction_buffer"]
        else:
            # Non-tilting train
            v_ms = math.sqrt(g * radius * (cant_m + cant_deficiency_m) / gauge_m)
            v_kmh = v_ms * 3.6 * self.operational_config["speed_restriction_buffer"]
        
        return min(v_kmh, self.train.max_speed)
    
    def calculate_required_tilt_angle(self, radius: float, speed_kmh: float) -> float:
        """
        Calculate the tilt angle required to maintain comfort at given speed/radius
        """
        if radius == float('inf') or radius > 100000 or speed_kmh <= 0:
            return 0.0
        
        g = 9.81
        speed_ms = speed_kmh / 3.6
        gauge_m = self.train.gauge / 1000
        cant_m = self.train.max_cant / 1000
        cant_deficiency_m = self.train.cant_deficiency_allowed / 1000
        
        # Centrifugal acceleration
        centrifugal = (speed_ms ** 2) / radius
        
        # What cant + deficiency can handle without tilt
        handled_by_cant = g * (cant_m + cant_deficiency_m) / gauge_m
        
        # Excess that needs tilt compensation
        excess = centrifugal - handled_by_cant
        
        if excess <= 0:
            return 0.0  # No tilt needed
        
        # Each degree of tilt compensates ~30mm equivalent cant deficiency
        tilt_compensation_per_degree = g * 0.030 / gauge_m
        
        required_tilt = excess / tilt_compensation_per_degree
        
        # Cap at max tilt angle
        return min(required_tilt, getattr(self.train, 'tilt_angle', 0))
    
    def calculate_curve_speed_with_tilt_rate_limit(self, radius: float, current_tilt: float, 
                                                    time_to_curve: float, curve_direction: int) -> tuple:
        """
        Calculate speed limit considering tilt rate constraints
        
        Args:
            radius: curve radius in meters
            current_tilt: current tilt angle (positive = tilted right)
            time_to_curve: time until reaching this curve point
            curve_direction: +1 for right curve, -1 for left curve, 0 for straight
        
        Returns:
            (speed_limit_kmh, achievable_tilt, new_tilt_direction)
        """
        if radius == float('inf') or radius > 100000:
            return self.train.max_speed, 0.0, 0
        
        tilt_angle = getattr(self.train, 'tilt_angle', 0)
        max_tilt_speed = getattr(self.train, 'max_tilt_speed', 0)
        max_tilt_rate = getattr(self.train, 'max_tilt_rate', 4.0)
        
        if tilt_angle <= 0 or max_tilt_speed <= 0:
            # Non-tilting train
            return self.calculate_curve_speed_limit(radius), 0.0, 0
        
        g = 9.81
        cant_m = self.train.max_cant / 1000
        gauge_m = self.train.gauge / 1000
        cant_deficiency_m = self.train.cant_deficiency_allowed / 1000
        comfort = self.train.passenger_comfort_factor
        
        # Base speed without tilt
        total_no_tilt = cant_m + cant_deficiency_m
        v_no_tilt_ms = math.sqrt(g * radius * total_no_tilt / gauge_m)
        v_no_tilt_kmh = v_no_tilt_ms * 3.6 * comfort
        
        # Target tilt for this curve
        target_tilt = tilt_angle * curve_direction
        
        # Calculate achievable tilt given current state and time available
        tilt_change_needed = target_tilt - current_tilt
        time_needed = abs(tilt_change_needed) / max_tilt_rate if max_tilt_rate > 0 else float('inf')
        
        # Modern systems anticipate ~2-3 seconds ahead
        anticipation_time = 2.5
        available_time = time_to_curve + anticipation_time
        
        if available_time >= time_needed:
            achievable_tilt = target_tilt
        else:
            max_change = max_tilt_rate * available_time
            if tilt_change_needed >= 0:
                achievable_tilt = current_tilt + min(max_change, tilt_change_needed)
            else:
                achievable_tilt = current_tilt - min(max_change, abs(tilt_change_needed))
        
        effective_tilt_magnitude = abs(achievable_tilt)
        
        if effective_tilt_magnitude <= 0.1:
            return min(v_no_tilt_kmh, self.train.max_speed), achievable_tilt, curve_direction
        
        # Calculate speed with effective tilt
        tilt_compensation_m = (effective_tilt_magnitude * 30) / 1000
        total_with_tilt = cant_m + cant_deficiency_m + tilt_compensation_m
        v_with_tilt_ms = math.sqrt(g * radius * total_with_tilt / gauge_m)
        v_with_tilt_kmh = v_with_tilt_ms * 3.6 * comfort
        
        # Cap tilt-assisted speed at max tilt speed
        v_with_tilt_kmh = min(v_with_tilt_kmh, max_tilt_speed)
        
        # CRITICAL: Return the HIGHER of no-tilt or tilt speed!
        # At high speeds (above tilt engagement), no-tilt speed can be higher
        # because tilt is capped but base physics allows higher speeds
        best_speed = max(v_no_tilt_kmh, v_with_tilt_kmh)
        
        # If no-tilt is better, we don't actually use tilt
        if v_no_tilt_kmh > v_with_tilt_kmh:
            achievable_tilt = 0.0
        
        return min(best_speed, self.train.max_speed), achievable_tilt, curve_direction
    
    def detect_curve_direction(self, track_points, idx, distances) -> int:
        """
        Detect curve direction: +1 = right turn, -1 = left turn, 0 = straight
        """
        if idx < 2 or idx >= len(track_points) - 2:
            return 0
        
        h1 = self._calculate_heading(track_points[idx-2], track_points[idx])
        h2 = self._calculate_heading(track_points[idx], track_points[idx+2])
        
        delta = h2 - h1
        while delta > 180: delta -= 360
        while delta < -180: delta += 360
        
        if abs(delta) < 0.5:
            return 0
        return 1 if delta > 0 else -1
    
    def calculate_gradient_speed_limit(self, gradient: float) -> float:
        """
        Calculate speed limit based on gradient (returns km/h)
        
        v9.0: Fixed formula - per-percent reduction
        - Uphill: reduce speed based on gradient steepness
        - Downhill: minor reduction for steep descents (braking concerns)
        - max_gradient_speed_reduction is now ~0.12 (12% per 1% grade)
        """
        if abs(gradient) < 0.1:
            # Negligible gradient
            return self.train.max_speed
        
        if gradient > 0:  # Uphill
            # Reduce speed based on gradient
            # With 0.12 reduction factor: 1% grade = 12% speed reduction
            reduction_factor = abs(gradient) * self.train.max_gradient_speed_reduction
            reduction_factor = min(reduction_factor, 0.7)  # Never reduce more than 70%
            return self.train.max_speed * max(1.0 - reduction_factor, 0.3)
        else:  # Downhill
            # Downhill: slight reduction for steep grades (braking capacity)
            if gradient > -2.0:
                return self.train.max_speed  # Mild downhill - no reduction
            elif gradient > -4.0:
                # Moderate downhill: 5% per percent over 2%
                excess = abs(gradient) - 2.0
                return self.train.max_speed * (1.0 - excess * 0.05)
            else:
                # Steep downhill: more conservative
                base_reduction = 0.10  # 10% for first 4%
                excess = abs(gradient) - 4.0
                additional = excess * 0.08  # 8% per additional percent
                return self.train.max_speed * max(1.0 - base_reduction - additional, 0.5)
    
    def apply_jerk_limit(self, current_accel: float, target_accel: float, dt: float) -> float:
        """Apply jerk limiting to acceleration changes"""
        if dt <= 0:
            return current_accel
        
        accel_change = target_accel - current_accel
        max_accel_change = self.train.jerk_limit * dt
        
        if abs(accel_change) > max_accel_change:
            accel_change = max_accel_change * np.sign(accel_change)
        
        return current_accel + accel_change
    
    def calculate_segment_with_jerk(self, v_initial_ms: float, v_target_ms: float, 
                                   distance_m: float, current_accel: float = 0) -> Tuple[float, float, float]:
        """
        Calculate time and acceleration for a segment using S-curve model.
        
        For Gen 6 APEx trains, uses 3-phase S-curve acceleration:
        - Phase 1: Ramp up jerk (0 → a_max over 10 sec)
        - Phase 2: Hold at peak acceleration
        - Phase 3: Ramp down jerk (a_max → 0 over 10 sec)
        
        Returns: (time_s, final_accel_ms2, avg_speed_ms)
        """
        if distance_m <= 0:
            return 0, current_accel, 0
        
        # Check if we need to accelerate or decelerate
        if abs(v_target_ms - v_initial_ms) < 0.01:
            # Constant speed
            time_s = distance_m / max(v_initial_ms, 0.1)
            return time_s, 0, v_initial_ms
        
        # Use S-curve model for acceleration/deceleration
        if USE_SCURVE_ACCEL and v_target_ms > v_initial_ms:
            # Acceleration case - use S-curve effective acceleration
            effective_accel = calculate_scurve_effective_accel(v_initial_ms, v_target_ms)
            
            # Check if we can achieve target speed in this distance
            # v² = v₀² + 2*a*d → v_max = sqrt(v₀² + 2*a*d)
            max_achievable_v = math.sqrt(v_initial_ms**2 + 2 * effective_accel * distance_m)
            
            if max_achievable_v >= v_target_ms:
                # Can reach target - calculate time for S-curve accel
                accel_distance = calculate_scurve_distance(v_initial_ms, v_target_ms)
                accel_time = calculate_scurve_accel_time(v_initial_ms, v_target_ms)
                
                if accel_distance < distance_m:
                    # Accelerate then cruise
                    cruise_distance = distance_m - accel_distance
                    cruise_time = cruise_distance / v_target_ms if v_target_ms > 0 else 0
                    time_s = accel_time + cruise_time
                else:
                    # Still accelerating at end of segment
                    time_s = accel_time * (distance_m / accel_distance) if accel_distance > 0 else 0
            else:
                # Can't quite reach target - accelerate whole segment
                time_s = calculate_scurve_accel_time(v_initial_ms, max_achievable_v)
                
        elif USE_SCURVE_ACCEL and v_target_ms < v_initial_ms:
            # Deceleration case - use empirical interpolation for all cases
            # This properly handles both full stops AND partial deceleration (e.g., 250→150 mph for curve)
            decel_time = calculate_scurve_decel_time(v_initial_ms, v_target_ms)
            
            # Calculate distance during deceleration
            avg_speed = (v_initial_ms + v_target_ms) / 2
            decel_distance = avg_speed * decel_time
            
            if decel_distance < distance_m:
                # Cruise then decelerate
                cruise_distance = distance_m - decel_distance
                cruise_time = cruise_distance / v_initial_ms if v_initial_ms > 0 else 0
                time_s = cruise_time + decel_time
            else:
                # Decelerating whole segment
                time_s = distance_m / avg_speed if avg_speed > 0 else 0
        else:
            # Legacy calculation (USE_SCURVE_ACCEL = False)
            # Calculate required acceleration
            required_accel = (v_target_ms**2 - v_initial_ms**2) / (2 * distance_m)
            
            # Apply physical limits
            if required_accel > 0:
                max_accel = self.train.acceleration * self.train.passenger_comfort_factor
                required_accel = min(required_accel, max_accel)
            else:
                max_decel = self.train.deceleration * self.train.passenger_comfort_factor
                required_accel = max(required_accel, -max_decel)
            
            # Consider jerk limit for transition time
            jerk_time = abs(required_accel - current_accel) / self.train.jerk_limit
            
            if jerk_time > 0:
                v_avg = (v_initial_ms + v_target_ms) / 2
                time_s = distance_m / max(v_avg, 0.1)
            else:
                time_s = 2 * distance_m / (v_initial_ms + v_target_ms)
        
        # Calculate final values
        avg_speed = distance_m / max(time_s, 0.1)
        final_accel = (v_target_ms - v_initial_ms) / max(time_s, 0.1)
        
        return time_s, final_accel, avg_speed
    
    def calculate_track_geometry(self, track_points: List[Tuple[float, float, float]]) -> Tuple[List[float], List[float], List[float]]:
        """
        Calculate gradients, curve radii, and cumulative distances for track
        Returns: (distances, gradients, curve_radii) all as lists
        
        If track_points contain 4 elements (lat, lon, elev, curve_radius), 
        uses pre-calculated curve radii from original points for more accurate geometry.
        """
        distances = [0]
        gradients = []
        curve_radii = []
        
        # Check if we have pre-calculated curve radii (4-element tuples)
        has_precalc_curves = len(track_points) > 0 and len(track_points[0]) >= 4
        
        # Calculate cumulative distances and gradients
        for i in range(len(track_points) - 1):
            # Extract just lat/lon/elev for segment calculation
            p1 = track_points[i][:3]
            p2 = track_points[i + 1][:3]
            segment = RailwaySegment(p1, p2)
            distances.append(distances[-1] + segment.distance)
            gradients.append(segment.gradient)
        
        # Add final gradient (same as last)
        if gradients:
            gradients.append(gradients[-1])
        else:
            gradients.append(0)
        
        if has_precalc_curves:
            # Use pre-calculated curve radii from original points
            print("[TrackGeom] Using pre-calculated curves from original track points...")
            curve_radii = [pt[3] if len(pt) >= 4 else float('inf') for pt in track_points]
            
            # Light smoothing to remove noise while PRESERVING TRANSITIONS
            # Don't use min() which would spread tight curves and lose transition benefits
            if len(curve_radii) > 3:
                filtered_radii = []
                window = 3  # Smaller window to preserve transitions
                for i in range(len(curve_radii)):
                    start = max(0, i - window // 2)
                    end = min(len(curve_radii), i + window // 2 + 1)
                    window_radii = [r if r != float('inf') else 1e9 for r in curve_radii[start:end]]
                    # Use weighted average favoring center point (preserves transitions)
                    # Center point gets 2x weight, neighbors get 1x
                    if len(window_radii) >= 3:
                        center_idx = len(window_radii) // 2
                        weighted = (window_radii[center_idx] * 2 + sum(window_radii)) / (len(window_radii) + 2)
                    else:
                        weighted = sum(window_radii) / len(window_radii)
                    filtered_radii.append(float('inf') if weighted >= 1e8 else weighted)
                curve_radii = filtered_radii
        else:
            # v9.0: Calculate curve radii using multi-baseline detection
            print("[TrackGeom] Analyzing curves with multi-baseline detection...")
            for i in range(len(track_points)):
                if i < 5 or i >= len(track_points) - 5:
                    # Need sufficient points for multi-baseline
                    curve_radii.append(float('inf'))
                else:
                    # Extract just lat/lon/elev for curve calculation
                    pts = [p[:3] for p in track_points]
                    radius = self.calculate_curve_radius_multi_baseline(pts, i, distances)
                    curve_radii.append(radius)
            
            # Apply median filter to smooth curve radii
            if len(curve_radii) > 7:
                filtered_radii = []
                window = 7
                for i in range(len(curve_radii)):
                    start = max(0, i - window // 2)
                    end = min(len(curve_radii), i + window // 2 + 1)
                    window_radii = [r if r != float('inf') else 1e9 for r in curve_radii[start:end]]
                    median = np.median(window_radii)
                    filtered_radii.append(float('inf') if median >= 1e8 else median)
                curve_radii = filtered_radii
        
        # Report curve statistics
        finite_radii = [r for r in curve_radii if r != float('inf') and r < 1e8]
        if finite_radii:
            print(f"[TrackGeom] Tightest curve: {min(finite_radii):.0f}m ({min(finite_radii)/1609:.2f}mi)")
            
            # Detailed curve breakdown - including tight curves
            r_under_500m = sum(1 for r in finite_radii if r < 500)
            r_500_to_1km = sum(1 for r in finite_radii if 500 <= r < 1000)
            r_1_to_2km = sum(1 for r in finite_radii if 1000 <= r < 2000)
            r_2_to_4km = sum(1 for r in finite_radii if 2000 <= r < 4000)
            r_4_to_6km = sum(1 for r in finite_radii if 4000 <= r < 6000)
            r_6_to_8km = sum(1 for r in finite_radii if 6000 <= r < 8000)
            r_over_8km = sum(1 for r in finite_radii if r >= 8000)
            
            total_points = len(curve_radii)
            straight_points = total_points - len(finite_radii)
            
            print(f"[TrackGeom] Curve distribution ({total_points} points, 10m spacing = {total_points*10/1000:.0f}km):")
            print(f"  Straight (no curve): {straight_points} ({100*straight_points/total_points:.1f}%)")
            print(f"  R < 500m (67-87 mph max): {r_under_500m} ({100*r_under_500m/total_points:.2f}%) = {r_under_500m*10/1000:.1f}km")
            print(f"  R 500m-1km (87-122 mph): {r_500_to_1km} ({100*r_500_to_1km/total_points:.2f}%) = {r_500_to_1km*10/1000:.1f}km")
            print(f"  R 1-2km (122-165 mph): {r_1_to_2km} ({100*r_1_to_2km/total_points:.2f}%) = {r_1_to_2km*10/1000:.1f}km")
            print(f"  R 2-4km (165-185 mph): {r_2_to_4km} ({100*r_2_to_4km/total_points:.2f}%) = {r_2_to_4km*10/1000:.1f}km")
            print(f"  R 4-6km (185-226 mph): {r_4_to_6km} ({100*r_4_to_6km/total_points:.2f}%) = {r_4_to_6km*10/1000:.1f}km")
            print(f"  R 6-8km (226-250 mph): {r_6_to_8km} ({100*r_6_to_8km/total_points:.2f}%) = {r_6_to_8km*10/1000:.1f}km")
            print(f"  R >= 8km (250 mph OK): {r_over_8km} ({100*r_over_8km/total_points:.2f}%) = {r_over_8km*10/1000:.1f}km")
        
        # Smooth gradients
        if len(gradients) > self.operational_config["gradient_smoothing_window"]:
            gradients = smooth_array(np.array(gradients), 
                                   self.operational_config["gradient_smoothing_window"]).tolist()
        
        return distances, gradients, curve_radii
    
    def calculate_speed_profile(self, track_points: List[Tuple[float, float, float]], 
                               station_indices: Dict[int, Station],
                               is_express: bool = False) -> Tuple[List[float], List[float], List[float]]:
        """
        Calculate speed profile with proper physics and jerk limiting
        Returns: (speeds_kmh, times_s, accelerations_ms2)
        """
        n_points = len(track_points)
        
        # Get track geometry
        distances, gradients, curve_radii = self.calculate_track_geometry(track_points)
        
        # v9: Calculate speed limits with tilt rate limiting
        speed_limits_kmh = []
        tilt_angles = []
        curve_directions = []
        
        # First pass: detect curve directions
        for i in range(n_points):
            curve_directions.append(self.detect_curve_direction(track_points, i, distances))
        
        # Get tilt parameters first (needed for S-curve detection)
        tilt_angle = getattr(self.train, 'tilt_angle', 0)
        max_tilt_rate = getattr(self.train, 'max_tilt_rate', 4.0)
        has_tilt = tilt_angle > 0 and max_tilt_rate > 0
        tilt_reversal_time = (tilt_angle * 2) / max_tilt_rate if has_tilt and max_tilt_rate > 0 else 0
        
        # Detect S-curves (direction reversals) and analyze transition zones
        # CRITICAL: During tilt reversal (+tilt → 0 → -tilt), train passes through 0° tilt
        # At 0° tilt, there is NO tilt compensation, so speed must be limited accordingly
        s_curve_transitions = []  # List of (transition_start_idx, transition_end_idx, straight_distance)
        
        if has_tilt:
            # Find all direction changes
            i = 0
            while i < len(curve_directions) - 1:
                if curve_directions[i] != 0:
                    # We're in a curve, find where it ends or reverses
                    curve_start_dir = curve_directions[i]
                    curve_end_idx = i
                    
                    # Find end of this curve section
                    for j in range(i + 1, len(curve_directions)):
                        if curve_directions[j] == curve_start_dir:
                            curve_end_idx = j
                        elif curve_directions[j] == -curve_start_dir:
                            # S-curve! Direction reversed
                            # Find where the opposite curve starts
                            next_curve_start = j
                            
                            # Calculate straight distance between curves (if any)
                            straight_distance = distances[next_curve_start] - distances[curve_end_idx]
                            
                            # Get the radii of both curves
                            curve1_radius = curve_radii[curve_end_idx] if curve_end_idx < len(curve_radii) else float('inf')
                            curve2_radius = curve_radii[next_curve_start] if next_curve_start < len(curve_radii) else float('inf')
                            
                            s_curve_transitions.append({
                                'curve1_end': curve_end_idx,
                                'curve2_start': next_curve_start,
                                'straight_distance': straight_distance,
                                'curve1_radius': curve1_radius,
                                'curve2_radius': curve2_radius,
                            })
                            i = j - 1  # Continue from here
                            break
                        elif curve_directions[j] == 0:
                            # Entered straight section
                            curve_end_idx = j - 1
                        else:
                            break
                    else:
                        break
                i += 1
        
        # Calculate speed limits for S-curve transitions
        # Key insight: if straight_distance < reversal_distance_at_curve_speed,
        # then tilt=0° occurs inside curve 2, and we must use no-tilt limit there
        s_curve_speed_limits = {}  # idx -> max_speed_kmh
        
        for trans in s_curve_transitions:
            straight_dist = trans['straight_distance']
            curve2_radius = trans['curve2_radius']
            
            if curve2_radius >= 10000 or curve2_radius <= 0:
                continue  # Very gentle curve, no constraint needed
            
            # Calculate no-tilt and with-tilt speed limits for curve 2
            g = 9.81
            cant_m = self.train.max_cant / 1000
            gauge_m = self.train.gauge / 1000
            cant_deficiency_m = self.train.cant_deficiency_allowed / 1000
            tilt_comp_m = (tilt_angle * 30) / 1000
            
            v_no_tilt_ms = math.sqrt(g * curve2_radius * (cant_m + cant_deficiency_m) / gauge_m)
            v_no_tilt_kmh = v_no_tilt_ms * 3.6 * self.operational_config["speed_restriction_buffer"]
            
            v_tilt_ms = math.sqrt(g * curve2_radius * (cant_m + cant_deficiency_m + tilt_comp_m) / gauge_m)
            v_tilt_kmh = min(v_tilt_ms * 3.6 * self.operational_config["speed_restriction_buffer"], 
                           getattr(self.train, 'max_tilt_speed', 265))
            
            # Distance needed to complete reversal at the tilt-assisted speed
            reversal_distance_at_tilt_speed = (v_tilt_kmh / 3.6) * tilt_reversal_time
            
            # If there's enough straight to complete reversal, no penalty
            if straight_dist >= reversal_distance_at_tilt_speed:
                continue  # Can use full tilt speed in curve 2
            
            # Not enough straight - tilt=0° will occur somewhere in/after the transition
            # The constraint is: at the point where tilt=0°, speed must be ≤ no-tilt limit
            # Since we can't predict exactly where tilt=0° occurs, apply no-tilt limit
            # to the entire transition zone
            
            # Mark points in the transition zone
            zone_start = max(0, trans['curve1_end'] - 10)
            zone_end = min(n_points, trans['curve2_start'] + 30)
            
            for idx in range(zone_start, zone_end):
                if idx not in s_curve_speed_limits or v_no_tilt_kmh < s_curve_speed_limits[idx]:
                    s_curve_speed_limits[idx] = v_no_tilt_kmh
        
        if s_curve_transitions:
            tight_transitions = sum(1 for t in s_curve_transitions if t['straight_distance'] < 200)
            print(f"[TiltRate] Detected {len(s_curve_transitions)} S-curve transitions ({tight_transitions} with <200m straight)")
        
        # Second pass: calculate speed limits with tilt rate awareness
        current_tilt = 0.0
        
        for i in range(n_points):
            gradient = gradients[i] if i < len(gradients) else 0
            curve_radius = curve_radii[i] if i < len(curve_radii) else float('inf')
            
            gradient_limit = self.calculate_gradient_speed_limit(gradient)
            
            if has_tilt and curve_radius != float('inf') and curve_radius < 10000:
                # Estimate time to reach this point for tilt anticipation
                if i > 0:
                    seg_dist = distances[i] - distances[i-1]
                    est_speed_ms = self.train.max_speed_ms * 0.8  # Estimate at 80% max speed
                    time_to_point = seg_dist / est_speed_ms if est_speed_ms > 0 else 0
                else:
                    time_to_point = 0
                
                curve_dir = curve_directions[i]
                
                # Check for S-curve transition speed limits
                # These account for: transition distance, tilt reversal time, and no-tilt speed at tilt=0°
                if i in s_curve_speed_limits:
                    # Apply the pre-calculated S-curve speed limit
                    s_curve_limit = s_curve_speed_limits[i]
                    
                    # Also calculate normal curve limit
                    normal_limit, achievable_tilt, _ = self.calculate_curve_speed_with_tilt_rate_limit(
                        curve_radius, current_tilt, time_to_point, curve_dir
                    )
                    
                    # Use the more restrictive
                    curve_limit = min(s_curve_limit, normal_limit)
                    
                    # If S-curve limit is active, tilt may be reduced
                    if s_curve_limit < normal_limit:
                        achievable_tilt = 0.0  # Conservative: assume tilt=0 during transition
                else:
                    curve_limit, achievable_tilt, _ = self.calculate_curve_speed_with_tilt_rate_limit(
                        curve_radius, current_tilt, time_to_point, curve_dir
                    )
                
                # Update current tilt state for next iteration
                current_tilt = achievable_tilt
                tilt_angles.append(achievable_tilt)
            else:
                # No tilt needed - use standard curve speed limit
                curve_limit = self.calculate_curve_speed_limit(curve_radius)
                # Gradually return tilt to neutral on straight track
                if has_tilt and abs(current_tilt) > 0.1:
                    # Assume ~0.1s per point at typical speed
                    tilt_return_rate = max_tilt_rate * 0.1
                    if current_tilt > 0:
                        current_tilt = max(0, current_tilt - tilt_return_rate)
                    else:
                        current_tilt = min(0, current_tilt + tilt_return_rate)
                tilt_angles.append(current_tilt)
            
            # Apply express bonus if applicable
            if is_express and i not in station_indices:
                max_speed = min(gradient_limit, curve_limit, self.train.max_speed * self.operational_config["express_speed_bonus"])
            else:
                max_speed = min(gradient_limit, curve_limit, self.train.max_speed)
            
            speed_limits_kmh.append(max_speed)
        
        if has_tilt:
            active_tilt_points = sum(1 for t in tilt_angles if abs(t) > 0.5)
            limited_points = sum(1 for i, t in enumerate(tilt_angles) if abs(t) < tilt_angle * 0.9 and curve_radii[i] < 5000)
            print(f"[TiltRate] Active tilt at {active_tilt_points}/{n_points} points")
            if limited_points > 0:
                print(f"[TiltRate] Tilt-rate-limited at {limited_points} curve points")
        
        # Initialize arrays
        speeds_ms = np.zeros(n_points)
        times = np.zeros(n_points)
        accelerations = np.zeros(n_points)
        
        # Set zero speed at stations
        for idx in station_indices:
            speeds_ms[idx] = 0
        
        # Forward pass - acceleration phase
        current_accel = 0
        for i in range(1, n_points):
            if i in station_indices:
                speeds_ms[i] = 0
                current_accel = 0
                continue
            
            seg_dist = distances[i] - distances[i-1]
            if seg_dist <= 0:
                speeds_ms[i] = speeds_ms[i-1]
                continue
            
            max_speed_ms = speed_limits_kmh[i] / 3.6
            gradient = gradients[i-1] if i-1 < len(gradients) else 0
            
            # Account for gradient in effective acceleration
            g = 9.81
            gradient_effect = g * math.sin(math.atan(gradient / 100))
            
            # Use S-curve model for realistic acceleration
            if USE_SCURVE_ACCEL and speeds_ms[i-1] < max_speed_ms:
                # Calculate what speed we can reach in this segment using S-curve
                # Get effective acceleration for this speed range
                effective_accel = calculate_scurve_effective_accel(speeds_ms[i-1], max_speed_ms)
                
                # Adjust for gradient
                effective_accel = effective_accel - gradient_effect
                
                if effective_accel > 0:
                    # Calculate achievable speed: v² = v₀² + 2*a*d
                    v_squared = speeds_ms[i-1]**2 + 2 * effective_accel * seg_dist
                    achievable_speed = math.sqrt(max(0, v_squared))
                    speeds_ms[i] = min(achievable_speed, max_speed_ms)
                else:
                    # Gradient too steep, maintain or reduce
                    speeds_ms[i] = min(speeds_ms[i-1], max_speed_ms)
            else:
                # Legacy power-limited model
                speed_ratio = speeds_ms[i-1] / self.train.max_speed_ms if self.train.max_speed_ms > 0 else 0
                power_factor = 1.0 - 0.5 * (speed_ratio ** 2)
                effective_accel = self.train.acceleration * self.train.passenger_comfort_factor * power_factor - gradient_effect
                
                if effective_accel > 0:
                    v_squared = speeds_ms[i-1]**2 + 2 * effective_accel * seg_dist
                    speeds_ms[i] = min(math.sqrt(max(0, v_squared)), max_speed_ms)
                else:
                    speeds_ms[i] = min(speeds_ms[i-1], max_speed_ms)
        
        # Backward pass - braking phase using S-curve model
        for i in range(n_points - 2, -1, -1):
            if i in station_indices:
                speeds_ms[i] = 0
                continue
            
            seg_dist = distances[i+1] - distances[i]
            if seg_dist <= 0:
                continue
            
            next_speed = speeds_ms[i+1]
            current_limit = speed_limits_kmh[i] / 3.6  # Convert to m/s
            
            # If next point is slower, we need to brake
            if next_speed < speeds_ms[i]:
                # Calculate braking distance using S-curve model
                # This accounts for realistic jerk-limited braking
                if USE_SCURVE_ACCEL:
                    # Get time to brake from current speed to next speed
                    brake_time = calculate_scurve_decel_time(speeds_ms[i], next_speed)
                    # Distance = average speed * time
                    avg_speed = (speeds_ms[i] + next_speed) / 2
                    required_brake_dist = avg_speed * brake_time
                    
                    # If we don't have enough distance, we need to enter slower
                    if required_brake_dist > seg_dist and seg_dist > 0:
                        # Binary search for max entry speed that allows braking in distance
                        low, high = next_speed, speeds_ms[i]
                        for _ in range(20):  # 20 iterations gives good precision
                            mid = (low + high) / 2
                            t = calculate_scurve_decel_time(mid, next_speed)
                            d = (mid + next_speed) / 2 * t
                            if d > seg_dist:
                                high = mid
                            else:
                                low = mid
                        max_entry_speed = low
                        speeds_ms[i] = min(speeds_ms[i], max_entry_speed, current_limit)
                    else:
                        speeds_ms[i] = min(speeds_ms[i], current_limit)
                else:
                    # Legacy: simple kinematic calculation
                    gradient = gradients[i] if i < len(gradients) else 0
                    g = 9.81
                    gradient_effect = g * math.sin(math.atan(gradient / 100))
                    effective_decel = self.train.deceleration * self.train.passenger_comfort_factor + gradient_effect
                    
                    if effective_decel > 0:
                        v_squared = next_speed**2 + 2 * effective_decel * seg_dist
                        max_entry_speed = math.sqrt(max(0, v_squared))
                        speeds_ms[i] = min(speeds_ms[i], max_entry_speed, current_limit)
        
        # Light smoothing for realism - small window to preserve curve transition benefits
        # Larger windows would blur the speed changes at transition spirals
        speeds_ms = smooth_array(speeds_ms, 2)
        
        # Enforce station stops again after smoothing
        for idx in station_indices:
            speeds_ms[idx] = 0
        
        # Calculate times and accelerations
        current_accel = 0
        for i in range(1, n_points):
            seg_dist = distances[i] - distances[i-1]
            if seg_dist <= 0:
                times[i] = times[i-1]
                accelerations[i] = 0
                continue
            
            v_initial = speeds_ms[i-1]
            v_final = speeds_ms[i]
            
            # Calculate segment with jerk limiting
            seg_time, accel, avg_speed = self.calculate_segment_with_jerk(
                v_initial, v_final, seg_dist, current_accel
            )
            
            times[i] = times[i-1] + seg_time
            accelerations[i] = self.apply_jerk_limit(current_accel, accel, seg_time)
            current_accel = accelerations[i]
        
        # Convert speeds to km/h for display
        speeds_kmh = [s * 3.6 for s in speeds_ms]
        
        # Report speed profile statistics
        max_speed_kmh = self.train.max_speed
        at_max = sum(1 for s in speeds_kmh if s >= max_speed_kmh * 0.98)
        below_200 = sum(1 for s in speeds_kmh if s < 200)
        below_300 = sum(1 for s in speeds_kmh if 200 <= s < 300)
        below_350 = sum(1 for s in speeds_kmh if 300 <= s < 350)
        below_400 = sum(1 for s in speeds_kmh if 350 <= s < max_speed_kmh * 0.98)
        
        print(f"[SpeedProfile] Speed distribution ({len(speeds_kmh)} points):")
        print(f"  At max speed (>={max_speed_kmh*0.98:.0f} km/h): {at_max} ({100*at_max/len(speeds_kmh):.1f}%)")
        print(f"  350-{max_speed_kmh*0.98:.0f} km/h: {below_400} ({100*below_400/len(speeds_kmh):.1f}%)")
        print(f"  300-350 km/h: {below_350} ({100*below_350/len(speeds_kmh):.1f}%)")
        print(f"  200-300 km/h: {below_300} ({100*below_300/len(speeds_kmh):.1f}%)")
        print(f"  < 200 km/h: {below_200} ({100*below_200/len(speeds_kmh):.1f}%)")
        
        # Estimate time lost to curves
        time_at_max = sum(1 for i, s in enumerate(speeds_kmh) if s >= max_speed_kmh * 0.98 and i not in station_indices)
        time_below_max = len(speeds_kmh) - time_at_max - len(station_indices)
        print(f"[SpeedProfile] Points below max (excluding stations): {time_below_max}")
        
        return speeds_kmh, times.tolist(), accelerations.tolist()

# ==================== RAILWAY SEGMENT ====================

class RailwaySegment:
    """Represents a segment of railway track between two points"""
    
    def __init__(self, start_point: Tuple[float, float, float], 
                 end_point: Tuple[float, float, float]):
        self.start_lat, self.start_lon, self.start_elev = start_point
        self.end_lat, self.end_lon, self.end_elev = end_point
        self.distance = self._calculate_distance()
        self.gradient = self._calculate_gradient()
    
    def _calculate_distance(self) -> float:
        """Calculate 3D distance accounting for elevation"""
        horizontal_distance = haversine_distance(
            self.start_lat, self.start_lon, 
            self.end_lat, self.end_lon
        )
        elevation_change = self.end_elev - self.start_elev
        return math.sqrt(horizontal_distance**2 + elevation_change**2)
    
    def _calculate_gradient(self) -> float:
        """Calculate gradient as percentage"""
        if self.distance < 0.01:  # Less than 1cm
            return 0
        
        horizontal_distance = haversine_distance(
            self.start_lat, self.start_lon, 
            self.end_lat, self.end_lon
        )
        
        if horizontal_distance < 0.01:
            return 0
        
        elevation_change = self.end_elev - self.start_elev
        return (elevation_change / horizontal_distance) * 100

# ==================== KMZ/KML PARSER ====================

class KMZParser:
    @staticmethod
    def extract_kmz(kmz_path: str) -> str:
        """Extract KML from KMZ file"""
        with tempfile.TemporaryDirectory() as temp_dir:
            with zipfile.ZipFile(kmz_path, 'r') as kmz:
                kmz.extractall(temp_dir)
                
                for file in os.listdir(temp_dir):
                    if file.endswith('.kml'):
                        kml_content = open(os.path.join(temp_dir, file), 'r', encoding='utf-8').read()
                        temp_kml = tempfile.NamedTemporaryFile(mode='w', suffix='.kml', delete=False, encoding='utf-8')
                        temp_kml.write(kml_content)
                        temp_kml.close()
                        return temp_kml.name
                
                raise ValueError("No KML file found in KMZ archive")
    
    @staticmethod
    def parse_kml(filepath: str, interpolate: bool = True, spacing_m: float = 10.0, 
                  parse_stations: bool = True, parse_tracks: bool = True) -> Tuple[List[Station], List[List[Tuple[float, float, float]]]]:
        """
        Parse KML file and extract stations and/or track segments
        
        Supports folder-based station classification:
        - Folders named 'Limited*', 'Ltd*' → LIMITED service level
        - Folders named 'Express*', 'Exp*' → EXPRESS service level  
        - Folders named 'Local*', 'Lcl*' → LOCAL service level
        
        Args:
            filepath: Path to KML file
            interpolate: Whether to interpolate track points
            spacing_m: Interpolation spacing in meters
            parse_stations: Whether to parse station points
            parse_tracks: Whether to parse track linestrings
        
        Returns:
            Tuple of (stations, track_segments)
        """
        tree = ET.parse(filepath)
        root = tree.getroot()
        
        # Handle namespace
        ns = {'kml': 'http://www.opengis.net/kml/2.2'}
        if root.tag == 'kml' or root.tag.endswith('}kml'):
            if '}' in root.tag:
                ns = {'kml': root.tag.split('}')[0].strip('{')}
            else:
                ns = {}
        
        stations = []
        track_segments = []
        
        # Build a map of placemark name -> folder service level
        # This handles folder-based station classification in KMZ files
        # We use station name as key since XML element objects differ between findall() calls
        placemark_service_levels = {}  # name -> service level
        
        def get_folder_service_level(folder_name: str) -> Optional[str]:
            """Determine service level from folder name"""
            if not folder_name:
                return None
            name_lower = folder_name.lower()
            if 'limited' in name_lower or name_lower.startswith('ltd'):
                return 'LIMITED'
            elif 'express' in name_lower or name_lower.startswith('exp'):
                return 'EXPRESS'
            elif 'local' in name_lower or name_lower.startswith('lcl'):
                return 'LOCAL'
            return None
        
        def process_folder(folder_elem, inherited_level=None):
            """Recursively process folders and map placemarks to service levels"""
            # Get this folder's name
            name_elem = folder_elem.find('kml:name', ns) if ns else folder_elem.find('name')
            folder_name = name_elem.text if name_elem is not None else None
            
            # Determine service level (folder's own or inherited)
            folder_level = get_folder_service_level(folder_name)
            current_level = folder_level if folder_level else inherited_level
            
            # Process placemarks DIRECTLY in this folder (not nested)
            if ns:
                direct_placemarks = folder_elem.findall('kml:Placemark', ns)
            else:
                direct_placemarks = folder_elem.findall('Placemark')
            
            for placemark in direct_placemarks:
                if current_level:
                    # Get placemark name to use as key
                    pm_name_elem = placemark.find('kml:name', ns) if ns else placemark.find('name')
                    pm_name = pm_name_elem.text if pm_name_elem is not None else None
                    if pm_name:
                        placemark_service_levels[pm_name] = current_level
            
            # Recursively process DIRECT subfolders only
            if ns:
                direct_subfolders = folder_elem.findall('kml:Folder', ns)
            else:
                direct_subfolders = folder_elem.findall('Folder')
            
            for subfolder in direct_subfolders:
                process_folder(subfolder, current_level)
        
        # Process only TOP-LEVEL folders (direct children of Document)
        if ns:
            doc_elem = root.find('.//kml:Document', ns)
            document = doc_elem if doc_elem is not None else root
            top_folders = document.findall('kml:Folder', ns)
        else:
            doc_elem = root.find('.//Document')
            document = doc_elem if doc_elem is not None else root
            top_folders = document.findall('Folder')
        
        for folder in top_folders:
            process_folder(folder)
        
        if placemark_service_levels:
            level_counts = {}
            for level in placemark_service_levels.values():
                level_counts[level] = level_counts.get(level, 0) + 1
            print(f"  [KMZ Folders] Found service levels: {level_counts}")
        
        # Find all placemarks
        if ns:
            placemarks = root.findall('.//kml:Placemark', ns)
        else:
            placemarks = root.findall('.//Placemark')
        
        for placemark in placemarks:
            name_elem = placemark.find('.//kml:name', ns) if ns else placemark.find('.//name')
            name = name_elem.text if name_elem is not None else "Unknown"
            
            desc_elem = placemark.find('.//kml:description', ns) if ns else placemark.find('.//description')
            description = desc_elem.text if desc_elem is not None else ""
            
            point = placemark.find('.//kml:Point', ns) if ns else placemark.find('.//Point')
            linestring = placemark.find('.//kml:LineString', ns) if ns else placemark.find('.//LineString')
            
            if point is not None and parse_stations:
                # Parse station
                coords_elem = point.find('.//kml:coordinates', ns) if ns else point.find('.//coordinates')
                if coords_elem is not None:
                    coords = coords_elem.text.strip().split(',')
                    if len(coords) >= 2:
                        lon = float(coords[0])
                        lat = float(coords[1])
                        elev = float(coords[2]) if len(coords) > 2 else 0
                        
                        # For APEX, is_major is ignored (APEx lists determine classification)
                        # For METRO/REGIONAL, use REGIONAL_MAJOR_STATIONS list
                        is_major = False
                        if SERVICE_TYPE != "APEX":
                            is_major = name in REGIONAL_MAJOR_STATIONS or "major" in description.lower() or "central" in description.lower()
                        is_terminus = (
                            "terminus" in description.lower() or 
                            "terminal" in description.lower()
                        )
                        
                        station = Station(name, lat, lon, elev, is_major=is_major, is_terminus=is_terminus)
                        
                        # Check for express designation from description
                        if "express" in description.lower():
                            station.is_express_stop = True
                        
                        # Apply folder-based service level if available
                        folder_level = placemark_service_levels.get(name)
                        if folder_level:
                            station.folder_service_level = folder_level
                            # Also set flags based on folder level
                            if folder_level == 'LIMITED':
                                station.is_limited = True
                            elif folder_level == 'EXPRESS':
                                station.is_express_stop = True
                        
                        stations.append(station)
            
            elif linestring is not None and parse_tracks:
                # Parse track
                coords_elem = linestring.find('.//kml:coordinates', ns) if ns else linestring.find('.//coordinates')
                if coords_elem is not None:
                    coords_text = coords_elem.text.strip()
                    track_points = []
                    for coord in coords_text.split():
                        parts = coord.split(',')
                        if len(parts) >= 2:
                            lon = float(parts[0])
                            lat = float(parts[1])
                            elev = float(parts[2]) if len(parts) > 2 else 0
                            track_points.append((lat, lon, elev))
                    
                    if track_points:
                        print(f"  Processing: '{name}' with {len(track_points)} points")
                        
                        if interpolate and len(track_points) > 1:
                            if PRECALCULATE_CURVES:
                                # Calculate curves from ORIGINAL points first
                                if CURVE_DETECTION_METHOD == "heading":
                                    curve_map = analyze_curves_heading_based(track_points)
                                else:
                                    curve_map = precalculate_curve_radii(track_points)
                                interpolated_points, curve_radii = interpolate_track_with_curves(
                                    track_points, curve_map, spacing_m
                                )
                                # Store curve radii as metadata (attached to track segment)
                                # We'll store as 4-tuple: (lat, lon, elev, curve_radius)
                                points_with_curves = [
                                    (p[0], p[1], p[2], r) 
                                    for p, r in zip(interpolated_points, curve_radii)
                                ]
                                track_segments.append(points_with_curves)
                                print(f"    Pre-calculated {len(curve_map)} curve radii from original points")
                            else:
                                interpolated_points = interpolate_track_points(track_points, spacing_m)
                                track_segments.append(interpolated_points)
                        else:
                            track_segments.append(track_points)
        
        return stations, track_segments
    
    @staticmethod
    def parse_kml_stations_only(filepath: str) -> List[Station]:
        """Parse KML file and extract only stations (no tracks)"""
        stations, _ = KMZParser.parse_kml(filepath, interpolate=False, parse_stations=True, parse_tracks=False)
        return stations
    
    @staticmethod
    def parse_kml_tracks_only(filepath: str, interpolate: bool = True, spacing_m: float = 10.0) -> List[List[Tuple[float, float, float]]]:
        """Parse KML file and extract only tracks (no stations)"""
        _, tracks = KMZParser.parse_kml(filepath, interpolate=interpolate, spacing_m=spacing_m, 
                                        parse_stations=False, parse_tracks=True)
        return tracks
    
    @staticmethod
    def merge_stations(stations1: List[Station], stations2: List[Station], 
                      dedup_radius: float = 50.0) -> List[Station]:
        """
        Merge two lists of stations, removing duplicates within dedup_radius
        
        Args:
            stations1: First list of stations (takes priority)
            stations2: Second list of stations
            dedup_radius: Radius in meters to consider stations as duplicates
        
        Returns:
            Merged list of unique stations
        """
        merged = stations1.copy()
        
        for station2 in stations2:
            is_duplicate = False
            for station1 in stations1:
                dist = haversine_distance(station1.lat, station1.lon, 
                                        station2.lat, station2.lon)
                if dist < dedup_radius:
                    is_duplicate = True
                    break
            
            if not is_duplicate:
                merged.append(station2)
        
        return merged
    
    @staticmethod
    def preview_station_sources(track_kmz_path: str, stations_kmz_path: Optional[str] = None) -> None:
        """
        Preview what stations are available from different sources
        
        Args:
            track_kmz_path: Path to track KMZ file
            stations_kmz_path: Optional path to stations KMZ file
        """
        print("\n" + "="*60)
        print("STATION SOURCE PREVIEW")
        print("="*60)
        
        temp_track_kml = None
        temp_stations_kml = None
        
        try:
            # Check track file stations
            temp_track_kml = KMZParser.extract_kmz(track_kmz_path)
            track_stations = KMZParser.parse_kml_stations_only(temp_track_kml)
            
            print(f"\nTrack file stations ({len(track_stations)} found):")
            for i, station in enumerate(track_stations):
                # For APEX mode, show service level from APEx classifications
                # For METRO/REGIONAL, show traditional flags
                if SERVICE_TYPE == "APEX":
                    service_level = get_station_service_level(station.name, station)
                    terminus_flag = " [TERMINUS]" if station.is_terminus else ""
                    print(f"  {i+1:2}. {station.name} ({service_level}){terminus_flag}")
                else:
                    flags = []
                    if station.is_terminus:
                        flags.append("TERMINUS")
                    if station.is_major:
                        flags.append("MAJOR")
                    if station.is_express_stop:
                        flags.append("EXPRESS")
                    flag_str = f" ({', '.join(flags)})" if flags else ""
                    print(f"  {i+1:2}. {station.name}{flag_str}")
            
            # Check stations file if provided
            if stations_kmz_path:
                temp_stations_kml = KMZParser.extract_kmz(stations_kmz_path)
                file_stations = KMZParser.parse_kml_stations_only(temp_stations_kml)
                
                print(f"\nStations file stations ({len(file_stations)} found):")
                for i, station in enumerate(file_stations):
                    if SERVICE_TYPE == "APEX":
                        service_level = get_station_service_level(station.name, station)
                        terminus_flag = " [TERMINUS]" if station.is_terminus else ""
                        print(f"  {i+1:2}. {station.name} ({service_level}){terminus_flag}")
                    else:
                        flags = []
                        if station.is_terminus:
                            flags.append("TERMINUS")
                        if station.is_major:
                            flags.append("MAJOR")
                        if station.is_express_stop:
                            flags.append("EXPRESS")
                        flag_str = f" ({', '.join(flags)})" if flags else ""
                        print(f"  {i+1:2}. {station.name}{flag_str}")
                
                # Show what would be merged
                merged = KMZParser.merge_stations(file_stations, track_stations, STATION_DEDUP_RADIUS)
                print(f"\nMerged stations ({len(merged)} unique):")
                print(f"  {len(track_stations)} from track file")
                print(f"  {len(file_stations)} from stations file")
                print(f"  {len(track_stations) + len(file_stations) - len(merged)} duplicates removed")
        
        finally:
            if temp_track_kml and os.path.exists(temp_track_kml):
                os.unlink(temp_track_kml)
            if temp_stations_kml and os.path.exists(temp_stations_kml):
                os.unlink(temp_stations_kml)
        
        print("="*60)

# ==================== JOURNEY CALCULATOR ====================

class JourneyCalculator:
    """Main class for calculating journey times with improved physics"""
    
    def __init__(self, train_specs: TrainSpecs):
        self.train_specs = train_specs
        self.physics = ImprovedPhysicsCalculator(train_specs)
        self.units = units
    
    def match_stations_to_track(self, track_points: List[Tuple[float, float, float]], 
                               stations: List[Station]) -> Dict[int, Station]:
        """Match stations to track points with improved tolerance"""
        station_indices = {}
        unmatched_stations = []
        collisions = []  # Track when multiple stations match to same index
        
        for station in stations:
            min_dist = float('inf')
            closest_idx = -1
            
            for i, point in enumerate(track_points):
                dist = haversine_distance(station.lat, station.lon, point[0], point[1])
                if dist < min_dist:
                    min_dist = dist
                    closest_idx = i
            
            # Use configurable tolerance
            if min_dist < STATION_MATCH_TOLERANCE:
                station_copy = Station(
                    name=station.name,
                    lat=station.lat,
                    lon=station.lon,
                    elevation=station.elevation,
                    dwell_time=station.dwell_time,
                    is_major=station.is_major,
                    is_terminus=station.is_terminus,
                    is_limited=getattr(station, 'is_limited', False),
                    platform_length=station.platform_length
                )
                station_copy.is_express_stop = station.is_express_stop
                station_copy.track_position = min_dist
                # Copy folder-based service level (critical for correct dwell times!)
                station_copy.folder_service_level = getattr(station, 'folder_service_level', None)
                
                # Check for collision (another station already at this index)
                if closest_idx in station_indices:
                    existing = station_indices[closest_idx]
                    collisions.append((station.name, existing.name, closest_idx))
                    # Keep the one closer to the track
                    if min_dist < existing.track_position:
                        station_indices[closest_idx] = station_copy
                else:
                    station_indices[closest_idx] = station_copy
            else:
                unmatched_stations.append((station.name, min_dist))
        
        # Report matching results
        print(f"\n[StationMatch] Matched {len(station_indices)}/{len(stations)} stations to track")
        
        # Report unmatched stations
        if unmatched_stations:
            print(f"[StationMatch] WARNING: {len(unmatched_stations)} station(s) not matched:")
            for name, dist in unmatched_stations:
                print(f"   - {name}: nearest track point is {dist:.1f}m away (tolerance: {STATION_MATCH_TOLERANCE}m)")
        
        # Report collisions
        if collisions:
            print(f"[StationMatch] WARNING: {len(collisions)} collision(s) - stations matching same track point:")
            for new_name, existing_name, idx in collisions:
                print(f"   - '{new_name}' and '{existing_name}' both match index {idx}")
        
        return station_indices
    
    def calculate_journey(self, track_points: List[Tuple[float, float, float]], 
                         stations: List[Station] = None, 
                         service_pattern: str = "ALL",
                         custom_stops: List[str] = None,
                         is_reverse: bool = False) -> Dict:
        """
        Calculate journey with improved physics and accurate timing
        
        Args:
            track_points: List of (lat, lon, elevation) track points
            stations: List of Station objects (can be None for track-only)
            service_pattern: "ALL", "EXPRESS", or "CUSTOM"
            custom_stops: List of station names for CUSTOM pattern
            is_reverse: Whether this is reverse direction
        
        Returns:
            Dictionary with journey analysis results
        """
        if len(track_points) < 2:
            return {"total_time": 0, "track_data": [], "journey_stations": [], "segment_times": []}
        
        # Filter stations based on service pattern
        if stations:
            filtered_stations = filter_stations_for_service(stations, service_pattern, custom_stops)
        else:
            filtered_stations = []
        
        # Match stations to track
        station_indices = {}
        journey_stations = []
        stations_skipped = []
        
        if filtered_stations:
            station_indices = self.match_stations_to_track(track_points, filtered_stations)
            
            # Build journey stations list
            for idx in sorted(station_indices.keys()):
                journey_stations.append(station_indices[idx])
        
        # Calculate skipped stations for express services
        if stations and service_pattern in ["EXPRESS", "CUSTOM"]:
            for station in stations:
                if station not in filtered_stations:
                    stations_skipped.append(station.name)
        
        # Determine journey extent
        if station_indices and len(station_indices) >= 2:
            # Use the actual stopping stations for journey extent
            sorted_stop_indices = sorted(station_indices.keys())
            journey_start_idx = sorted_stop_indices[0]
            journey_end_idx = sorted_stop_indices[-1]
        elif stations:
            # Fallback to all stations if no stops are defined
            all_station_indices = self.match_stations_to_track(track_points, stations)
            if all_station_indices:
                all_sorted_indices = sorted(all_station_indices.keys())
                journey_start_idx = all_sorted_indices[0]
                journey_end_idx = all_sorted_indices[-1]
            else:
                # No stations matched to track
                journey_start_idx = 0
                journey_end_idx = len(track_points) - 1
        else:
            # No stations at all (track-only mode)
            journey_start_idx = 0
            journey_end_idx = len(track_points) - 1
        
        # Set terminus flags for actual stopping stations
        if station_indices and len(station_indices) >= 2:
            sorted_indices = sorted(station_indices.keys())
            for idx in station_indices:
                station_indices[idx].is_terminus = False
            
            station_indices[sorted_indices[0]].is_terminus = True
            station_indices[sorted_indices[-1]].is_terminus = True
        
        print(f"\n   Journey extent: index {journey_start_idx} to {journey_end_idx}")
        print(f"   Journey has {len(journey_stations)} stopping stations")
        if journey_stations:
            print(f"   First stop: {journey_stations[0].name}")
            print(f"   Last stop: {journey_stations[-1].name}")
        elif stations and service_pattern != "ALL":
            print(f"   Note: No stations designated as {service_pattern} stops")
            print(f"   Train will run non-stop between first and last station locations")
        
        if stations_skipped:
            print(f"   Skipping {len(stations_skipped)} stations: {', '.join(stations_skipped[:5])}" + 
                  ("..." if len(stations_skipped) > 5 else ""))
        
        # Calculate speed profile using improved physics
        is_express = service_pattern == "EXPRESS" or (service_pattern == "CUSTOM" and stations and len(journey_stations) < len(stations) * 0.5)
        speeds_kmh, times_s, accelerations_ms2 = self.physics.calculate_speed_profile(
            track_points[journey_start_idx:journey_end_idx+1], 
            {idx - journey_start_idx: station for idx, station in station_indices.items() if journey_start_idx <= idx <= journey_end_idx}, 
            is_express
        )
        
        # Calculate track geometry for the journey segment
        journey_track_points = track_points[journey_start_idx:journey_end_idx+1]
        distances, gradients, curve_radii = self.physics.calculate_track_geometry(journey_track_points)
        
        # Build track data for the journey segment
        track_data = []
        for i in range(len(journey_track_points)):
            point = journey_track_points[i]
            global_idx = i + journey_start_idx  # Index in the original track
            
            track_point = TrackPoint(
                lat=point[0],
                lon=point[1],
                elevation=point[2],
                distance_from_start=distances[i] if i < len(distances) else 0,
                speed=speeds_kmh[i] if i < len(speeds_kmh) else 0,
                speed_ms=speeds_kmh[i] / 3.6 if i < len(speeds_kmh) else 0,
                acceleration=accelerations_ms2[i] if i < len(accelerations_ms2) else 0,
                time_from_start=times_s[i] if i < len(times_s) else 0,
                at_station=station_indices.get(global_idx),
                segment_index=i,
                gradient=gradients[i] if i < len(gradients) else 0,
                curve_radius=curve_radii[i] if i < len(curve_radii) else float('inf'),
                speed_limit=min(
                    self.physics.calculate_gradient_speed_limit(gradients[i] if i < len(gradients) else 0),
                    self.physics.calculate_curve_speed_limit(curve_radii[i] if i < len(curve_radii) else float('inf'))
                )
            )
            
            track_data.append(track_point)
        
        # Calculate station-to-station segment times
        segment_times = []
        if len(station_indices) >= 2:
            sorted_stations = sorted(station_indices.items(), key=lambda x: x[0])
            
            for i in range(1, len(sorted_stations)):
                idx_from, station_from = sorted_stations[i-1]
                idx_to, station_to = sorted_stations[i]
                
                # Adjust indices to journey segment
                local_idx_from = idx_from - journey_start_idx
                local_idx_to = idx_to - journey_start_idx
                
                # Get times and distances
                time_at_departure = times_s[local_idx_from] if local_idx_from < len(times_s) else 0
                time_at_arrival = times_s[local_idx_to] if local_idx_to < len(times_s) else 0
                segment_running_time = time_at_arrival - time_at_departure
                
                dist_at_departure = distances[local_idx_from] if local_idx_from < len(distances) else 0
                dist_at_arrival = distances[local_idx_to] if local_idx_to < len(distances) else 0
                segment_distance = dist_at_arrival - dist_at_departure
                
                # Find max speed in segment
                segment_max_speed = 0
                for j in range(local_idx_from, min(local_idx_to + 1, len(speeds_kmh))):
                    segment_max_speed = max(segment_max_speed, speeds_kmh[j])
                
                # Calculate average speed
                avg_speed = (segment_distance / 1000) / (segment_running_time / 3600) if segment_running_time > 0 else 0
                
                # Count skipped stations in this segment
                skipped_in_segment = 0
                if stations:
                    for station in stations:
                        if station not in filtered_stations:
                            # Check if this station would be between idx_from and idx_to
                            for k in range(idx_from + 1, idx_to):
                                if k < len(track_points) and haversine_distance(station.lat, station.lon, 
                                                     track_points[k][0], track_points[k][1]) < STATION_MATCH_TOLERANCE:
                                    skipped_in_segment += 1
                                    break
                
                segment_times.append(StationSegmentTime(
                    from_station=station_from,
                    to_station=station_to,
                    segment_distance=segment_distance,
                    running_time=segment_running_time,
                    max_speed_achieved=segment_max_speed,
                    avg_speed=avg_speed,
                    start_time=time_at_departure,
                    arrival_time=time_at_arrival,
                    stations_skipped=skipped_in_segment
                ))
        
        # Calculate total times
        if journey_stations:
            # Running time is the total time for the journey segment
            running_time = times_s[-1] if times_s else 0
            
            # Calculate dwell times
            total_dwell = sum(station.get_dwell_time(is_express) for station in journey_stations)
            
            # Exclude terminal dwells from journey time
            first_station_dwell = journey_stations[0].get_dwell_time(is_express) if journey_stations else 0
            last_station_dwell = journey_stations[-1].get_dwell_time(is_express) if len(journey_stations) > 1 else 0
            intermediate_dwell = total_dwell - first_station_dwell - last_station_dwell
            
            journey_time = running_time + intermediate_dwell
        else:
            running_time = times_s[-1] if times_s else 0
            journey_time = running_time
            total_dwell = 0
            intermediate_dwell = 0
            first_station_dwell = 0
            last_station_dwell = 0
        
        # Calculate distance
        total_distance = distances[-1] if distances else 0
        
        print(f"\n Timing Breakdown:")
        print(f"   Running time: {running_time:.0f}s ({running_time/60:.1f} min)")
        print(f"   Intermediate dwells: {intermediate_dwell:.0f}s")
        print(f"   Terminal dwells (excluded): {first_station_dwell + last_station_dwell:.0f}s")
        print(f" >>> Journey time: {journey_time:.0f}s ({journey_time/60:.1f} min)")
        
        return {
            "service_pattern": service_pattern,
            "total_time_s": journey_time + first_station_dwell + last_station_dwell,
            "total_time_min": (journey_time + first_station_dwell + last_station_dwell) / 60,
            "journey_time_s": journey_time,
            "journey_time_min": journey_time / 60,
            "running_time_s": running_time,
            "running_time_min": running_time / 60,
            "dwell_time_s": total_dwell,
            "intermediate_dwell_s": intermediate_dwell,
            "first_station_dwell_s": first_station_dwell,
            "last_station_dwell_s": last_station_dwell,
            "distance_m": total_distance,
            "distance_km": total_distance / 1000,
            "average_speed_kmh": (total_distance / 1000) / (running_time / 3600) if running_time > 0 else 0,
            "commercial_speed_kmh": (total_distance / 1000) / (journey_time / 3600) if journey_time > 0 else 0,
            "num_stops": len(station_indices),
            "stations_skipped": len(stations_skipped),
            "track_data": track_data,
            "station_indices": station_indices,
            "journey_stations": journey_stations,
            "segment_times": segment_times,
            "journey_start_idx": journey_start_idx
        }

# ==================== VISUALIZATION AND OUTPUT FUNCTIONS ====================

def print_station_times_table(journey_data: Dict, units: UnitConverter):
    """Print formatted station-to-station times table"""
    if not journey_data or 'segment_times' not in journey_data:
        print("No segment timing data available.")
        return
    
    segment_times = journey_data['segment_times']
    station_indices = journey_data.get('station_indices', {})
    service_pattern = journey_data.get('service_pattern', 'ALL')
    
    if not segment_times and not station_indices:
        print("No station data available.")
        return
    
    print(f"\n{'='*130}")
    print(f"STATION-TO-STATION TRAVEL TIMES ({service_pattern} Service)")
    dwell = get_dwell_times()
    print(f"Service Type: {SERVICE_TYPE} | Dwell Times: Local={dwell['LOCAL']}s, Express={dwell['EXPRESS']}s, Limited={dwell['LIMITED']}s")
    print("* Terminal station dwells excluded from journey time")
    print(f"{'='*130}")
    print(f"{'Station':<35} {'Service':<12} {'Distance':<12} {'Travel':<10} {'Dwell':<10} {'Journey':<10} {'Max':<8} {'Avg':<8} {'Skipped':<8}")
    print(f"{'':35} {'':12} {'(cumulative)':12} {'Time':10} {'Time':10} {'Time':10} {'Speed':8} {'Speed':8} {'Stations':8}")
    print(f"{'-'*130}")
    
    sorted_stations = sorted(station_indices.items(), key=lambda x: x[0])
    if sorted_stations:
        first_idx, first_station = sorted_stations[0]
        dwell_time = first_station.get_dwell_time()
        print(f"{first_station.name:<35} {'ORIGIN':<12} {'0.0 ' + units.long_distance_unit:<12} {'---':<10} {f'{dwell_time}s*':<10} {'0:00':<10} {'---':<8} {'---':<8} {'---':<8}")
    
    journey_time_cumulative = 0
    cumulative_distance = 0
    total_skipped = 0
    
    for i, segment in enumerate(segment_times):
        is_last = (i == len(segment_times) - 1)
        
        segment_running_time = segment.running_time
        journey_time_cumulative += segment_running_time
        cumulative_distance += segment.segment_distance
        total_skipped += segment.stations_skipped
        
        dwell_time = segment.to_station.get_dwell_time()
        if is_last:
            dwell_display = f"{dwell_time}s*"
            station_type = "DESTINATION"
        else:
            journey_time_cumulative += dwell_time
            dwell_display = f"{dwell_time}s"
            station_type = get_station_service_level(segment.to_station.name, segment.to_station)
        
        dist_display = f"{units.long_distance(cumulative_distance/1000):.3f} {units.long_distance_unit}"
        time_display = f"{segment_running_time:.0f}s"
        
        journey_mins = int(journey_time_cumulative / 60)
        journey_secs = int(journey_time_cumulative % 60)
        journey_time_display = f"{journey_mins}:{journey_secs:02d}"
        
        max_speed_display = f"{units.speed(segment.max_speed_achieved):.1f}"
        avg_speed_display = f"{units.speed(segment.avg_speed):.1f}"
        skipped_display = str(segment.stations_skipped) if segment.stations_skipped > 0 else "---"
        
        print(f"{segment.to_station.name:<35} {station_type:<12} {dist_display:<12} "
              f"{time_display:<10} {dwell_display:<10} {journey_time_display:<10} "
              f"{max_speed_display:<8} {avg_speed_display:<8} {skipped_display:<8}")
    
    actual_journey_time_s = journey_data.get('journey_time_s', 0)
    actual_journey_time_min = actual_journey_time_s / 60
    
    print(f"\n{'='*130}")
    print(f"JOURNEY TIME (departure to arrival): {actual_journey_time_s:.0f}s ({actual_journey_time_min:.1f} minutes)")
    if total_skipped > 0:
        print(f"STATIONS SKIPPED: {total_skipped} ({journey_data.get('stations_skipped', 0)} total)")
    print(f"{'='*130}\n")

def create_journey_visualization(journey_data: Dict, direction: str, train_specs: TrainSpecs):
    """Create visualization for journey"""
    if not journey_data or 'track_data' not in journey_data:
        print(f"No track data for {direction} visualization")
        return None, None
    
    journey_stations = journey_data.get('journey_stations', [])
    track_data = journey_data['track_data']
    service_pattern = journey_data.get('service_pattern', 'ALL')
    
    route_name = direction
    if journey_data.get('station_indices'):
        sorted_indices = sorted(journey_data['station_indices'].items(), key=lambda x: x[0])
        if len(sorted_indices) >= 2:
            origin = sorted_indices[0][1].name.split('/')[0].strip()
            destination = sorted_indices[-1][1].name.split('/')[0].strip()
            route_name = f"{origin} â†’ {destination}"
    
    print(f"\nCreating {direction} visualization: {route_name} ({service_pattern} service)")
    
    x_offset = 50 if direction == "Forward" else 750
    y_offset = 50 if direction == "Forward" else 100
    
    # Speed profile figure
    fig_speed = plt.figure(num=f'Speed Profile - {direction}', figsize=(16, 10))
    
    gs = GridSpec(3, 1, figure=fig_speed, hspace=0.3)
    
    # Speed plot
    ax_speed = fig_speed.add_subplot(gs[0])
    distances = [units.long_distance(pt.distance_from_start / 1000) for pt in track_data]
    speeds = [units.speed(pt.speed) for pt in track_data]
    
    ax_speed.fill_between(distances, 0, speeds, alpha=0.3, color='green', label='Speed Profile')
    ax_speed.plot(distances, speeds, 'g-', linewidth=2, alpha=0.8)
    
    # Add speed limit line
    speed_limits = [units.speed(pt.speed_limit) for pt in track_data]
    ax_speed.plot(distances, speed_limits, 'r--', linewidth=1, alpha=0.5, label='Speed Limit')
    
    # Station markers
    if journey_data.get('station_indices') and journey_data.get('journey_start_idx') is not None:
        journey_start_idx = journey_data['journey_start_idx']
        for idx, station in journey_data['station_indices'].items():
            # Convert global index to journey-relative index
            relative_idx = idx - journey_start_idx
            if 0 <= relative_idx < len(distances):
                ax_speed.axvline(x=distances[relative_idx], color='blue', linestyle='--', alpha=0.5)
                station_name = station.name.split('/')[0].strip()[:15]
                ax_speed.text(distances[relative_idx], max(speeds) * 0.95 if speeds else 100, 
                            station_name, rotation=90, fontsize=8, ha='right', va='top')
    
    ax_speed.set_xlabel(f'Distance ({units.long_distance_unit})', fontsize=12)
    ax_speed.set_ylabel(f'Speed ({units.speed_unit})', fontsize=12)
    ax_speed.set_title(f'Speed Profile - {route_name} ({service_pattern})', fontsize=14, fontweight='bold')
    ax_speed.grid(True, alpha=0.3)
    ax_speed.legend(loc='upper right', fontsize=9)
    ax_speed.set_ylim(0, max(speeds) * 1.1 if speeds else 100)
    
    # Acceleration plot
    ax_accel = fig_speed.add_subplot(gs[1])
    accelerations = [units.acceleration(pt.acceleration) for pt in track_data]
    
    max_accel = units.acceleration(2.0)
    filtered_accel = [max(-max_accel, min(max_accel, a)) for a in accelerations]
    colors = ['green' if a > 0.1 else 'red' if a < -0.1 else 'yellow' for a in filtered_accel]
    
    for i in range(len(distances) - 1):
        ax_accel.plot(distances[i:i+2], filtered_accel[i:i+2], color=colors[i], linewidth=2, alpha=0.7)
    
    ax_accel.axhline(y=0, color='black', linewidth=0.5)
    ax_accel.set_xlabel(f'Distance ({units.long_distance_unit})', fontsize=12)
    ax_accel.set_ylabel(f'Acceleration ({units.acceleration_unit})', fontsize=12)
    ax_accel.set_title(f'Acceleration Profile', fontsize=14, fontweight='bold')
    ax_accel.grid(True, alpha=0.3)
    ax_accel.set_ylim(-max_accel * 1.2, max_accel * 1.2)
    
    # Gradient plot
    ax_gradient = fig_speed.add_subplot(gs[2])
    gradients = [pt.gradient for pt in track_data]
    
    gradient_colors = ['red' if g > 0 else 'green' if g < 0 else 'gray' for g in gradients]
    for i in range(len(distances) - 1):
        ax_gradient.plot(distances[i:i+2], gradients[i:i+2], color=gradient_colors[i], linewidth=2, alpha=0.7)
    
    ax_gradient.axhline(y=0, color='black', linewidth=0.5)
    ax_gradient.set_xlabel(f'Distance ({units.long_distance_unit})', fontsize=12)
    ax_gradient.set_ylabel('Gradient (%)', fontsize=12)
    ax_gradient.set_title('Track Gradient Profile', fontsize=14, fontweight='bold')
    ax_gradient.grid(True, alpha=0.3)
    ax_gradient.set_ylim(-5, 5)
    
    train_name = train_specs.train_type.replace('_', ' ')
    fig_speed.suptitle(f'Journey Analysis - {route_name} ({train_name}) - {service_pattern} Service', 
                       fontsize=16, fontweight='bold')
    plt.tight_layout()
    
    # Geographic map
    lats = [pt.lat for pt in track_data]
    lons = [pt.lon for pt in track_data]
    
    lat_range = max(lats) - min(lats) if lats else 1
    lon_range = max(lons) - min(lons) if lons else 1
    aspect_ratio = lat_range / lon_range if lon_range > 0 else 1
    
    if aspect_ratio > 1:
        fig_width = 14
        fig_height = min(fig_width * aspect_ratio * 1.2, 20)
    else:
        fig_width = 18
        fig_height = 10
    
    fig_map = plt.figure(num=f'Route Map - {direction}', figsize=(fig_width, fig_height))
    
    ax_map = fig_map.add_axes([0.05, 0.12, 0.88, 0.83])
    
    # Plot track with speed coloring
    points = np.array([lons, lats]).T.reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)
    
    norm = plt.Normalize(0, max(speeds) if speeds else 100)
    lc = LineCollection(segments, cmap='RdYlGn', norm=norm, linewidth=5, alpha=0.8)
    lc.set_array(np.array(speeds))
    
    line = ax_map.add_collection(lc)
    
    # Colorbar
    cbar_ax = fig_map.add_axes([0.94, 0.12, 0.02, 0.83])
    cbar = plt.colorbar(line, cax=cbar_ax)
    cbar.set_label(f'Speed ({units.speed_unit})', fontsize=12, fontweight='bold')
    
    # Plot stations
    # Calculate label offset based on map extent (very small for close positioning)
    lon_extent = max(lons) - min(lons) if lons else 0.01
    lat_extent = max(lats) - min(lats) if lats else 0.01
    
    for i, station in enumerate(journey_stations):
        is_origin = (i == 0)
        is_destination = (i == len(journey_stations) - 1)
        
        if is_origin or is_destination:
            marker = 's'
            color = 'red'
            size = 150
            zorder = 15
        elif station.is_express_stop:
            marker = '^'
            color = 'orange'
            size = 120
            zorder = 12
        else:
            marker = 'o'
            color = 'blue'
            size = 100
            zorder = 10
        
        ax_map.scatter(station.lon, station.lat, c=color, s=size, marker=marker, 
                      edgecolors='black', linewidth=1.5, zorder=zorder, alpha=0.9)
        
        label = station.name.split('/')[0].strip()
        if len(label) > 25:
            label = label[:22] + '...'
        
        # Very small offset for labels to be close to stations
        offset_distance_lon = lon_extent * 0.008  # Reduced for minimal gap
        offset_distance_lat = lat_extent * 0.008  # Reduced for minimal gap
        
        # Define the eight compass directions with appropriate text alignments
        # Format: (direction_name, x_offset, y_offset, horizontal_align, vertical_align)
        directions = [
            ('N', 0, offset_distance_lat, 'center', 'bottom'),           # North
            ('NE', offset_distance_lon, offset_distance_lat, 'left', 'bottom'),    # Northeast
            ('E', offset_distance_lon, 0, 'left', 'center'),            # East
            ('SE', offset_distance_lon, -offset_distance_lat, 'left', 'top'),      # Southeast
            ('S', 0, -offset_distance_lat, 'center', 'top'),            # South
            ('SW', -offset_distance_lon, -offset_distance_lat, 'right', 'top'),    # Southwest
            ('W', -offset_distance_lon, 0, 'right', 'center'),          # West
            ('NW', -offset_distance_lon, offset_distance_lat, 'right', 'bottom')   # Northwest
        ]
        
        # Count track points near each potential label position
        min_track_points = float('inf')
        best_direction = directions[0]  # Default to North
        
        for dir_name, dx, dy, ha, va in directions:
            label_lon = station.lon + dx
            label_lat = station.lat + dy
            
            # Count track points within a small radius of this label position
            nearby_count = 0
            check_radius_lon = lon_extent * 0.01  # Small radius for checking
            check_radius_lat = lat_extent * 0.01
            
            for pt in track_data:
                if (abs(pt.lon - label_lon) < check_radius_lon and 
                    abs(pt.lat - label_lat) < check_radius_lat):
                    nearby_count += 1
            
            # Add a small penalty for diagonal directions to prefer cardinal directions when equal
            if dir_name in ['NE', 'SE', 'SW', 'NW']:
                nearby_count += 0.5
            
            # Prefer this position if it has fewer nearby track points
            if nearby_count < min_track_points:
                min_track_points = nearby_count
                best_direction = (dir_name, dx, dy, ha, va)
        
        # Use the best direction found
        _, label_offset_x, label_offset_y, ha, va = best_direction
        
        ax_map.text(station.lon + label_offset_x, station.lat + label_offset_y, label,
                   fontsize=7, fontweight='bold', ha=ha, va=va,
                   bbox=dict(boxstyle='round,pad=0.15', facecolor='white', 
                           edgecolor='black', alpha=0.9, linewidth=0.5))
    
    # Set limits with extra padding for side labels
    lat_padding = lat_range * 0.05
    lon_padding = lon_range * 0.12  # Extra padding for side labels
    ax_map.set_xlim(min(lons) - lon_padding, max(lons) + lon_padding)
    ax_map.set_ylim(min(lats) - lat_padding, max(lats) + lat_padding)
    
    ax_map.set_xlabel('Longitude', fontsize=12, fontweight='bold')
    ax_map.set_ylabel('Latitude', fontsize=12, fontweight='bold')
    ax_map.set_title(f'{route_name} - {service_pattern} Service Map', fontsize=14, fontweight='bold', pad=10)
    ax_map.grid(True, alpha=0.25, linestyle='--', linewidth=0.5)
    ax_map.set_aspect('equal', adjustable='box')
    
    # Journey statistics
    total_dist = units.long_distance(journey_data.get('distance_km', 0))
    journey_time = journey_data.get('journey_time_min', 0)
    commercial_speed = units.speed(journey_data.get('commercial_speed_kmh', 0))
    num_stops = len(journey_stations)
    stations_skipped = journey_data.get('stations_skipped', 0)
    
    info_text = (f'Distance: {total_dist:.1f} {units.long_distance_unit}  |  '
                f'Time: {journey_time:.1f} min  |  '
                f'Speed: {commercial_speed:.1f} {units.speed_unit}  |  '
                f'Stops: {num_stops}')
    
    if stations_skipped > 0:
        info_text += f'  |  Skipped: {stations_skipped}'
    
    fig_map.text(0.5, 0.06, info_text, fontsize=9, fontweight='bold', 
                ha='center', va='center',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', 
                         edgecolor='black', alpha=0.9, linewidth=0.8))
    
    # Add legend
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], marker='s', color='w', markerfacecolor='red', 
               markersize=7, markeredgecolor='black', markeredgewidth=1, label='Terminus'),
        Line2D([0], [0], marker='^', color='w', markerfacecolor='orange', 
               markersize=7, markeredgecolor='black', markeredgewidth=1, label='Express Stop'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='blue', 
               markersize=6, markeredgecolor='black', markeredgewidth=1, label='Local Stop'),
    ]
    
    ax_map.legend(handles=legend_elements, loc='upper right', fontsize=8)
    
    plt.draw()
    plt.pause(0.1)
    
    return fig_speed, fig_map

def compare_service_patterns(track_points: List[Tuple[float, float, float]], 
                            stations: List[Station], 
                            calculator: JourneyCalculator,
                            units: UnitConverter) -> List[ServiceComparison]:
    """Compare different service patterns on the same track"""
    comparisons = []
    
    # Calculate all-stops baseline
    all_stops_data = calculator.calculate_journey(track_points, stations, "ALL")
    all_stops_time = all_stops_data['journey_time_min']
    all_stops_speed = all_stops_data['commercial_speed_kmh']
    
    comparisons.append(ServiceComparison(
        service_name="All Stops (Local)",
        stations_served=all_stops_data['num_stops'],
        stations_skipped=0,
        journey_time_min=all_stops_time,
        running_time_min=all_stops_data['running_time_min'],
        commercial_speed_kmh=all_stops_speed,
        time_savings_min=0,
        speed_improvement_percent=0
    ))
    
    # Calculate express service
    express_data = calculator.calculate_journey(track_points, stations, "EXPRESS")
    express_time = express_data['journey_time_min']
    express_speed = express_data['commercial_speed_kmh']
    
    # Only add meaningful express comparison if there are express stops
    if express_data['num_stops'] > 0:
        comparisons.append(ServiceComparison(
            service_name="Express",
            stations_served=express_data['num_stops'],
            stations_skipped=express_data['stations_skipped'],
            journey_time_min=express_time,
            running_time_min=express_data['running_time_min'],
            commercial_speed_kmh=express_speed,
            time_savings_min=all_stops_time - express_time,
            speed_improvement_percent=((express_speed - all_stops_speed) / all_stops_speed) * 100 if all_stops_speed > 0 else 0
        ))
    
    # Calculate theoretical non-stop (between first and last station)
    # Create a truncated track that only includes points between stations
    if stations and len(stations) >= 2:
        # Get station positions on track
        all_station_indices = calculator.match_stations_to_track(track_points, stations)
        if all_station_indices and len(all_station_indices) >= 2:
            sorted_indices = sorted(all_station_indices.keys())
            start_idx = sorted_indices[0]
            end_idx = sorted_indices[-1]
            
            # Create truncated track
            truncated_track = track_points[start_idx:end_idx+1]
            
            # Calculate non-stop on truncated track
            no_stops_data = calculator.calculate_journey(truncated_track, None, "NONE")
            no_stops_time = no_stops_data['journey_time_min']
            no_stops_speed = no_stops_data['commercial_speed_kmh']
            
            comparisons.append(ServiceComparison(
                service_name="Non-Stop (Theoretical)",
                stations_served=0,
                stations_skipped=len(stations),
                journey_time_min=no_stops_time,
                running_time_min=no_stops_data['running_time_min'],
                commercial_speed_kmh=no_stops_speed,
                time_savings_min=all_stops_time - no_stops_time,
                speed_improvement_percent=((no_stops_speed - all_stops_speed) / all_stops_speed) * 100 if all_stops_speed > 0 else 0
            ))
    
    # Print comparison table
    print(f"\n{'='*100}")
    print("SERVICE PATTERN COMPARISON")
    print(f"{'='*100}")
    print(f"{'Service':<20} {'Stops':<8} {'Skipped':<8} {'Time':<12} {'Speed':<12} {'Time Saved':<12} {'Speed +%':<10}")
    print(f"{'-'*100}")
    
    for comp in comparisons:
        time_str = f"{comp.journey_time_min:.1f} min"
        speed_str = f"{units.speed(comp.commercial_speed_kmh):.1f} {units.speed_unit}"
        saved_str = f"{comp.time_savings_min:.1f} min" if comp.time_savings_min > 0 else "---"
        improvement_str = f"+{comp.speed_improvement_percent:.1f}%" if comp.speed_improvement_percent > 0 else "---"
        
        print(f"{comp.service_name:<20} {comp.stations_served:<8} {comp.stations_skipped:<8} "
              f"{time_str:<12} {speed_str:<12} {saved_str:<12} {improvement_str:<10}")
    
    print(f"{'='*100}\n")
    
    return comparisons

# ==================== MAIN ANALYSIS FUNCTION ====================

def analyze_kmz_file(kmz_path: str, save_results: bool = True, 
                    create_visualization: bool = True,
                    interpolate: bool = True, spacing_m: float = 10.0, 
                    calculate_reverse: bool = True,
                    stations_kmz_path: Optional[str] = None,
                    station_source: str = "TRACK",
                    service_pattern: str = "ALL",
                    custom_stops: List[str] = None,
                    compare_services: bool = False) -> Dict:
    """
    Main analysis function for KMZ file with v7.0 enhancements
    
    Args:
        kmz_path: Path to track KMZ file
        save_results: Whether to save results to JSON
        create_visualization: Whether to create plots
        interpolate: Whether to interpolate track points
        spacing_m: Interpolation spacing in meters
        calculate_reverse: Whether to calculate reverse direction
        stations_kmz_path: Optional path to separate stations KMZ file
        station_source: Source for stations ("TRACK", "STATIONS_FILE", "BOTH", "NONE")
        service_pattern: Service pattern ("ALL", "EXPRESS", "CUSTOM")
        custom_stops: List of station names for CUSTOM pattern
        compare_services: Whether to compare different service patterns
    
    Returns:
        Dictionary with analysis results
    """
    results = {}
    temp_kml = None
    temp_stations_kml = None
    
    try:
        print(f"\n{'='*60}")
        print(f"TRAIN CONFIGURATION (v9.0)")
        print(f"{'='*60}")
        print(f" Train: {ACTIVE_TRAIN['name']}")
        print(f" Category: {ACTIVE_TRAIN.get('category', 'custom')}")
        print(f" Max Speed: {TRAIN_CONFIG['max_speed']} {units.speed_unit}")
        tilt = TRAIN_CONFIG.get('tilt_angle', 0)
        max_tilt = TRAIN_CONFIG.get('max_tilt_speed', 0)
        tilt_rate = TRAIN_CONFIG.get('max_tilt_rate', 0)
        if tilt > 0:
            print(f" Tilt: {tilt}° active up to {max_tilt} {units.speed_unit}, rate: {tilt_rate}°/s")
        print(f" Unit System: {UNIT_SYSTEM}")
        print(f" Service Type: {SERVICE_TYPE}")
        dwell = get_dwell_times()
        print(f" Dwell Times: Local={dwell['LOCAL']}s, Express={dwell['EXPRESS']}s, Limited={dwell['LIMITED']}s")
        print(f" Station Source: {station_source}")
        print(f" Service Pattern: {service_pattern}")
        print(f"{'='*60}")
        
        # Load track KMZ file
        print(f"\nLoading track KMZ file: {kmz_path}")
        temp_kml = KMZParser.extract_kmz(kmz_path)
        
        # Parse based on station source configuration
        stations = []
        track_segments = []
        
        if station_source == "TRACK":
            print("Parsing track and stations from track file...")
            stations, track_segments = KMZParser.parse_kml(temp_kml, interpolate=interpolate, spacing_m=spacing_m)
            
        elif station_source == "STATIONS_FILE":
            if not stations_kmz_path:
                raise ValueError("STATIONS_FILE selected but no stations_kmz_path provided")
            
            print("Parsing tracks from track file (ignoring stations)...")
            track_segments = KMZParser.parse_kml_tracks_only(temp_kml, interpolate=interpolate, spacing_m=spacing_m)
            
            print(f"\nLoading stations KMZ file: {stations_kmz_path}")
            temp_stations_kml = KMZParser.extract_kmz(stations_kmz_path)
            
            print("Parsing stations from stations file...")
            stations = KMZParser.parse_kml_stations_only(temp_stations_kml)
            
        elif station_source == "BOTH":
            print("Parsing track and stations from track file...")
            track_stations, track_segments = KMZParser.parse_kml(temp_kml, interpolate=interpolate, spacing_m=spacing_m)
            
            if stations_kmz_path:
                print(f"\nLoading stations KMZ file: {stations_kmz_path}")
                temp_stations_kml = KMZParser.extract_kmz(stations_kmz_path)
                
                print("Parsing additional stations from stations file...")
                file_stations = KMZParser.parse_kml_stations_only(temp_stations_kml)
                
                print(f"Merging stations (dedup radius: {STATION_DEDUP_RADIUS}m)...")
                print(f"  Track file stations: {len(track_stations)}")
                print(f"  Stations file stations: {len(file_stations)}")
                
                # Merge with priority to stations file (in case of updates)
                stations = KMZParser.merge_stations(file_stations, track_stations, STATION_DEDUP_RADIUS)
                print(f"  Merged total: {len(stations)} unique stations")
            else:
                stations = track_stations
            
        elif station_source == "NONE":
            print("Parsing tracks only (no stations)...")
            track_segments = KMZParser.parse_kml_tracks_only(temp_kml, interpolate=interpolate, spacing_m=spacing_m)
            stations = []
        
        else:
            raise ValueError(f"Invalid station_source: {station_source}")
        
        # Display summary
        print(f"\n{'='*60}")
        print(f"DATA SUMMARY:")
        print(f" Total stations: {len(stations)}")
        print(f" Total track segments: {len(track_segments)}")
        
        if stations:
            print(f"\n All stations found:")
            for idx, station in enumerate(stations):
                if SERVICE_TYPE == "APEX":
                    service_level = get_station_service_level(station.name, station)
                    terminus_flag = " [TERMINUS]" if station.is_terminus else ""
                    print(f"  {idx+1:2}. {station.name} ({service_level}){terminus_flag}")
                else:
                    flags = []
                    if station.is_terminus:
                        flags.append("TERMINUS")
                    if station.is_major:
                        flags.append("MAJOR")
                    if station.is_express_stop:
                        flags.append("EXPRESS")
                    flag_str = f" ({', '.join(flags)})" if flags else ""
                    print(f"  {idx+1:2}. {station.name}{flag_str}")
        else:
            print("\n No stations loaded (running track-only analysis)")
        
        train_specs = TrainSpecs.from_config(TRAIN_CONFIG, UNIT_SYSTEM, TRAIN_TYPE)
        calculator = JourneyCalculator(train_specs)
        
        for seg_idx, track_points in enumerate(track_segments):
            if len(track_points) < 2:
                continue
            
            print(f"\n{'='*60}")
            print(f"ANALYZING TRACK SEGMENT {seg_idx + 1}")
            print(f"{'='*60}")
            print(f" Track points: {len(track_points)}")
            
            # Compare service patterns if requested
            if compare_services and stations:
                print("\n=== COMPARING SERVICE PATTERNS ===")
                comparisons = compare_service_patterns(track_points, stations, calculator, units)
                results['service_comparisons'] = [
                    {
                        'service': comp.service_name,
                        'stops': comp.stations_served,
                        'skipped': comp.stations_skipped,
                        'time_min': comp.journey_time_min,
                        'speed_kmh': comp.commercial_speed_kmh,
                        'time_saved_min': comp.time_savings_min,
                        'speed_improvement_%': comp.speed_improvement_percent
                    }
                    for comp in comparisons
                ]
            
            # FORWARD JOURNEY
            print(f"\n=== FORWARD JOURNEY ({service_pattern} Service) ===")
            forward_data = calculator.calculate_journey(
                track_points, stations if stations else None, 
                service_pattern, custom_stops, is_reverse=False
            )
            
            print(f"\n FORWARD SUMMARY:")
            print(f"   Service Pattern: {service_pattern}")
            print(f"   Distance: {units.long_distance(forward_data['distance_km']):.2f} {units.long_distance_unit}")
            print(f"   Journey time: {forward_data['journey_time_min']:.1f} minutes")
            print(f"   Commercial speed: {units.speed(forward_data['commercial_speed_kmh']):.1f} {units.speed_unit}")
            print(f"   Stations served: {len(forward_data.get('journey_stations', []))}")
            print(f"   Stations skipped: {forward_data.get('stations_skipped', 0)}")
            
            if stations:
                print_station_times_table(forward_data, units)
            
            # REVERSE JOURNEY
            reverse_data = None
            if calculate_reverse:
                print(f"\n=== REVERSE JOURNEY ({service_pattern} Service) ===")
                # Reverse the track points
                reversed_track = list(reversed(track_points))
                reverse_data = calculator.calculate_journey(
                    reversed_track, stations if stations else None, 
                    service_pattern, custom_stops, is_reverse=True
                )
                
                print(f"\n REVERSE SUMMARY:")
                print(f"   Service Pattern: {service_pattern}")
                print(f"   Distance: {units.long_distance(reverse_data['distance_km']):.2f} {units.long_distance_unit}")
                print(f"   Journey time: {reverse_data['journey_time_min']:.1f} minutes")
                print(f"   Commercial speed: {units.speed(reverse_data['commercial_speed_kmh']):.1f} {units.speed_unit}")
                print(f"   Stations served: {len(reverse_data.get('journey_stations', []))}")
                print(f"   Stations skipped: {reverse_data.get('stations_skipped', 0)}")
                
                if stations:
                    print_station_times_table(reverse_data, units)
                
                # Compare directions
                print(f"\n{'='*80}")
                print("DIRECTIONAL COMPARISON")
                print(f"{'='*80}")
                print(f"{'Metric':<25} {'Forward':<20} {'Reverse':<20} {'Difference':<20}")
                print(f"{'-'*85}")
                
                metrics = [
                    ('Journey Time', 'journey_time_min', 'minutes'),
                    ('Running Time', 'running_time_min', 'minutes'),
                    ('Commercial Speed', 'commercial_speed_kmh', units.speed_unit),
                    ('Average Speed', 'average_speed_kmh', units.speed_unit),
                    ('Distance', 'distance_km', units.long_distance_unit),
                    ('Stations Served', 'num_stops', 'stops'),
                    ('Stations Skipped', 'stations_skipped', 'stations'),
                ]
                
                for name, key, unit in metrics:
                    fwd_val = forward_data.get(key, 0)
                    rev_val = reverse_data.get(key, 0)
                    
                    if 'speed' in key:
                        fwd_display = f"{units.speed(fwd_val):.1f} {unit}"
                        rev_display = f"{units.speed(rev_val):.1f} {unit}"
                        diff = abs(fwd_val - rev_val)
                        diff_display = f"{units.speed(diff):.1f} {unit}"
                    elif 'distance' in key:
                        fwd_display = f"{units.long_distance(fwd_val):.2f} {unit}"
                        rev_display = f"{units.long_distance(rev_val):.2f} {unit}"
                        diff = abs(fwd_val - rev_val)
                        diff_display = f"{units.long_distance(diff):.2f} {unit}"
                    else:
                        fwd_display = f"{fwd_val:.0f} {unit}" if key in ['num_stops', 'stations_skipped'] else f"{fwd_val:.1f} {unit}"
                        rev_display = f"{rev_val:.0f} {unit}" if key in ['num_stops', 'stations_skipped'] else f"{rev_val:.1f} {unit}"
                        diff_display = f"{abs(fwd_val - rev_val):.0f} {unit}" if key in ['num_stops', 'stations_skipped'] else f"{abs(fwd_val - rev_val):.1f} {unit}"
                    
                    print(f"{name:<25} {fwd_display:<20} {rev_display:<20} {diff_display:<20}")
                
                print(f"{'='*80}")
            
            # VISUALIZATION
            if create_visualization:
                print(f"\n{'='*60}")
                print("GENERATING VISUALIZATIONS...")
                print(f"{'='*60}")
                
                print("\nCreating FORWARD journey visualization...")
                fwd_speed_fig, fwd_map_fig = create_journey_visualization(
                    forward_data, "Forward", train_specs
                )
                
                if calculate_reverse and reverse_data:
                    print("\nCreating REVERSE journey visualization...")
                    rev_speed_fig, rev_map_fig = create_journey_visualization(
                        reverse_data, "Reverse", train_specs
                    )
            
            # Store results
            results['version'] = 'v9.0'
            results['train_type'] = TRAIN_TYPE
            results['train_name'] = ACTIVE_TRAIN['name']
            results['station_source'] = station_source
            results['service_pattern'] = service_pattern
            results['stations_count'] = len(stations)
            results['forward'] = {
                'distance': f"{units.long_distance(forward_data['distance_km']):.2f} {units.long_distance_unit}",
                'journey_time': f"{forward_data['journey_time_min']:.1f} minutes",
                'commercial_speed': f"{units.speed(forward_data['commercial_speed_kmh']):.1f} {units.speed_unit}",
                'stations_served': len(forward_data.get('journey_stations', [])),
                'stations_skipped': forward_data.get('stations_skipped', 0),
            }
            
            if calculate_reverse and reverse_data:
                results['reverse'] = {
                    'distance': f"{units.long_distance(reverse_data['distance_km']):.2f} {units.long_distance_unit}",
                    'journey_time': f"{reverse_data['journey_time_min']:.1f} minutes",
                    'commercial_speed': f"{units.speed(reverse_data['commercial_speed_kmh']):.1f} {units.speed_unit}",
                    'stations_served': len(reverse_data.get('journey_stations', [])),
                    'stations_skipped': reverse_data.get('stations_skipped', 0),
                }
        
        if save_results and results:
            output_file = kmz_path.replace('.kmz', f'_analysis_{TRAIN_TYPE.lower()}_{service_pattern.lower()}_v9.json')
            with open(output_file, 'w') as f:
                json.dump(results, f, indent=2)
            print(f"\nResults saved to: {output_file}")
        
        print(f"\n{'='*60}")
        print("ANALYSIS COMPLETE! (v9.0)")
        print(f"{'='*60}")
        
    except Exception as e:
        print(f"\nERROR: {e}")
        import traceback
        traceback.print_exc()
    finally:
        if temp_kml and os.path.exists(temp_kml):
            os.unlink(temp_kml)
        if temp_stations_kml and os.path.exists(temp_stations_kml):
            os.unlink(temp_stations_kml)
    
    return results

# ==================== MAIN ENTRY POINT ====================

if __name__ == "__main__":
    print("="*60)
    print("Enhanced Train Travel Time Calculator v9.0")
    print("Advanced Physics with Flexible Station Configuration")
    print("="*60)
    print(f"Configuration:")
    print(f" Track KMZ Path: {KMZ_PATH}")
    
    # Show station configuration
    if STATION_SOURCE == "STATIONS_FILE":
        print(f" Stations KMZ Path: {STATIONS_KMZ_PATH}")
        print(f" Station Source: Using separate stations file only")
    elif STATION_SOURCE == "BOTH":
        print(f" Stations KMZ Path: {STATIONS_KMZ_PATH}")
        print(f" Station Source: Merging stations from both files")
    elif STATION_SOURCE == "TRACK":
        print(f" Station Source: Using stations from track file")
    elif STATION_SOURCE == "NONE":
        print(f" Station Source: No stations (track-only analysis)")
    
    print(f" Service Pattern: {SERVICE_PATTERN}")
    if SERVICE_PATTERN == "CUSTOM":
        print(f" Custom Stops: {', '.join(CUSTOM_STOPS)}")
    
    print(f" Train Type: {TRAIN_TYPE}")
    print(f" Train: {ACTIVE_TRAIN['name']}")
    print(f" Unit System: {UNIT_SYSTEM}")
    print(f" Service Type: {SERVICE_TYPE}")
    dwell = get_dwell_times()
    print(f" Dwell Times: Local={dwell['LOCAL']}s, Express={dwell['EXPRESS']}s, Limited={dwell['LIMITED']}s")
    print(f" Interpolation: {INTERPOLATION_SPACING}m spacing")
    print(f" Station Match Tolerance: {STATION_MATCH_TOLERANCE}m")
    
    if STATION_SOURCE == "BOTH":
        print(f" Station Dedup Radius: {STATION_DEDUP_RADIUS}m")
    
    print(f" Compare Services: {COMPARE_SERVICE_PATTERNS}")
    print("="*60)
    
    # Optional: Preview stations before analysis
    # KMZParser.preview_station_sources(KMZ_PATH, STATIONS_KMZ_PATH if STATION_SOURCE != "TRACK" else None)
    
    # Determine which files to use
    stations_path = None
    if STATION_SOURCE in ["STATIONS_FILE", "BOTH"]:
        stations_path = STATIONS_KMZ_PATH
    
    results = analyze_kmz_file(
        KMZ_PATH, 
        save_results=SAVE_RESULTS, 
        create_visualization=CREATE_VISUALIZATIONS,
        interpolate=True, 
        spacing_m=INTERPOLATION_SPACING,
        calculate_reverse=CALCULATE_REVERSE,
        stations_kmz_path=stations_path,
        station_source=STATION_SOURCE,
        service_pattern=SERVICE_PATTERN,
        
        custom_stops=CUSTOM_STOPS if SERVICE_PATTERN == "CUSTOM" else None,
        compare_services=COMPARE_SERVICE_PATTERNS
    )
    
    print("\nAnalysis complete!")
    print("Check the plot windows for visualizations")
    plt.show()